In [1]:
# ==================================================================================================
# PARASURG / CISLR
# L20B-CORRECTED
#
# STRICT FULL-CISLR PROTOCOL FREEZE WITH PERMANENT L19 LOCKED-TEST RESERVATION
#
# PURPOSE
# -------
# Correct the original L20B protocol before any further L20 training.
#
# IMPORTANT:
#   * NO VIDEO DECODING
#   * NO MODEL LOADING
#   * NO TRAINING
#   * NO INFERENCE
#   * NO ACCURACY COMPUTATION
#   * NO TEST-BASED MODEL SELECTION
#
# STRICT DATA ISOLATION:
#
#   Original clean CISLR population
#           |
#           +---- L19 locked test (10)  ---> NEVER TOUCH DURING L20
#           |
#           +---- remaining development universe
#
#   Supervised class criterion:
#       support >= 4 in ORIGINAL clean population
#
#   The class-selection criterion is therefore independent of the locked-test removal.
#
#   After locked-test removal:
#       supervised population -> deterministic ~70/30 train/validation
#
#   Domain adaptation pool:
#       original clean population
#       MINUS L19 locked test
#       MINUS corrected L20 validation
#
#   NOTE:
#       L20 supervised TRAIN videos ARE allowed in unsupervised/domain adaptation.
#       L20 validation videos are NOT.
#       L19 locked test videos are NEVER allowed.
#
# ==================================================================================================

from pathlib import Path
import pandas as pd
import numpy as np
import hashlib
import json
import shutil
from datetime import datetime, timezone


# ==================================================================================================
# 0. CONFIGURATION
# ==================================================================================================

SEED = 42

SUPPORT_THRESHOLD = 4

TRAIN_RATIO = 0.70
VAL_RATIO = 0.30


ROOT = Path(
    "/home/dipshikha/Music/cao/CISLR_RESEARCH/"
    "CISLR_LANDMARK_MODEL"
)

L20A_DIR = (
    ROOT
    / "audit"
    / "l20a_full_cislr_dataset_audit"
)

OLD_L20B_DIR = (
    ROOT
    / "audit"
    / "l20b_full_cislr_protocol_freeze"
)

L18B_DIR = (
    ROOT
    / "audit"
    / "l18b_10class_dataset"
)

LOCKED_TEST_PATH = (
    L18B_DIR
    / "manifests"
    / "l18b_LOCKED_test_manifest.csv"
)


# --------------------------------------------------------------------------------------------------
# IMPORTANT:
#
# We DO NOT overwrite the old L20B.
#
# The old protocol is retained as an auditable historical experiment.
# --------------------------------------------------------------------------------------------------

OUT_DIR = (
    ROOT
    / "audit"
    / "l20b_corrected_strict_protocol_freeze"
)

MANIFEST_DIR = OUT_DIR / "manifests"
CONFIG_DIR = OUT_DIR / "config"
REPORT_DIR = OUT_DIR / "reports"
HASH_DIR = OUT_DIR / "hashes"

for directory in [
    OUT_DIR,
    MANIFEST_DIR,
    CONFIG_DIR,
    REPORT_DIR,
    HASH_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )


print("=" * 150)
print("PARASURG / CISLR — L20B-CORRECTED")
print("STRICT FULL-CISLR PROTOCOL FREEZE")
print("=" * 150)

print()
print("Project root:")
print(ROOT)

print()
print("L20A audit:")
print(L20A_DIR)

print()
print("L19 locked test:")
print(LOCKED_TEST_PATH)

print()
print("Corrected L20B output:")
print(OUT_DIR)

print()
print("Seed              :", SEED)
print("Support threshold :", SUPPORT_THRESHOLD)
print("Target train ratio:", TRAIN_RATIO)
print("Target val ratio  :", VAL_RATIO)


# ==================================================================================================
# 1. BASIC PATH VERIFICATION
# ==================================================================================================

print()
print("=" * 150)
print("1. BASIC PATH VERIFICATION")
print("=" * 150)

assert ROOT.exists(), (
    f"Project root not found:\n{ROOT}"
)

assert L20A_DIR.exists(), (
    f"L20A audit directory not found:\n{L20A_DIR}"
)

assert LOCKED_TEST_PATH.exists(), (
    f"L19 locked test manifest not found:\n{LOCKED_TEST_PATH}"
)

print("[PASS] Project root exists.")
print("[PASS] L20A audit exists.")
print("[PASS] L19 locked-test manifest exists.")


# ==================================================================================================
# 2. HELPERS
# ==================================================================================================

def sha256_file(path, chunk_size=1024 * 1024):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        while True:

            chunk = f.read(chunk_size)

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


def canonical_uid_series(series):

    return (
        series
        .astype(str)
        .str.strip()
    )


def detect_uid_column(df):

    candidates = [
        "uid",
        "UID",
        "video_uid",
        "video_id",
        "id",
    ]

    for col in candidates:

        if col in df.columns:
            return col

    raise RuntimeError(
        "Could not identify UID column. "
        f"Available columns: {list(df.columns)}"
    )


def detect_gloss_column(df):

    candidates = [
        "gloss",
        "Gloss",
        "label",
        "sign",
        "class",
        "word",
    ]

    for col in candidates:

        if col in df.columns:
            return col

    raise RuntimeError(
        "Could not identify gloss column. "
        f"Available columns: {list(df.columns)}"
    )


def stable_class_seed(gloss, global_seed=SEED):

    # DO NOT use Python hash(), because Python's hash is not stable
    # across independent interpreter sessions.

    seed_bytes = hashlib.sha256(
        f"{global_seed}|{gloss}".encode("utf-8")
    ).digest()

    return (
        int.from_bytes(
            seed_bytes[:8],
            "little"
        )
        % (2 ** 32)
    )


def save_json(data, path):

    with open(
        path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            data,
            f,
            indent=2,
            ensure_ascii=False
        )


# ==================================================================================================
# 3. LOCATE THE AUTHORITATIVE L20A CLEAN 7048-ROW MANIFEST
# ==================================================================================================

print()
print("=" * 150)
print("2. LOCATING AUTHORITATIVE L20A CLEAN MANIFEST")
print("=" * 150)

csv_candidates = list(
    L20A_DIR.rglob("*.csv")
)

print(
    "CSV candidates discovered:",
    len(csv_candidates)
)

valid_candidates = []

for path in csv_candidates:

    try:

        df = pd.read_csv(path)

    except Exception:

        continue

    if len(df) != 7048:
        continue

    try:

        uid_col = detect_uid_column(df)
        gloss_col = detect_gloss_column(df)

    except Exception:

        continue

    # Candidate must have 7048 unique UIDs.
    uids = canonical_uid_series(
        df[uid_col]
    )

    if uids.nunique() != 7048:
        continue

    filename = path.name.lower()

    score = 0

    for token in [
        "clean",
        "manifest",
        "valid",
        "resolved",
        "video",
    ]:

        if token in filename:
            score += 1

    valid_candidates.append(
        (
            score,
            path,
            df,
            uid_col,
            gloss_col
        )
    )


if not valid_candidates:

    raise RuntimeError(
        "Could not locate an authoritative 7048-row "
        "clean L20A manifest containing unique UID and gloss columns."
    )


valid_candidates.sort(
    key=lambda x: (
        x[0],
        str(x[1])
    ),
    reverse=True
)

(
    _,
    CLEAN_MANIFEST_PATH,
    clean_df,
    uid_col,
    gloss_col
) = valid_candidates[0]


print()
print("[FOUND] L20A clean manifest:")
print(CLEAN_MANIFEST_PATH)

print()
print("Rows       :", len(clean_df))
print("UID column :", uid_col)
print("Gloss col  :", gloss_col)


# ==================================================================================================
# 4. STANDARDIZE AUTHORITATIVE CLEAN POPULATION
# ==================================================================================================

print()
print("=" * 150)
print("3. STANDARDIZING ORIGINAL CLEAN POPULATION")
print("=" * 150)

clean_df = clean_df.copy()

clean_df["uid"] = canonical_uid_series(
    clean_df[uid_col]
)

clean_df["gloss"] = (
    clean_df[gloss_col]
    .astype(str)
    .str.strip()
)

if clean_df["uid"].duplicated().any():

    duplicates = clean_df.loc[
        clean_df["uid"].duplicated(
            keep=False
        ),
        "uid"
    ].tolist()

    raise RuntimeError(
        "Duplicate UID(s) found in authoritative clean population: "
        f"{duplicates[:20]}"
    )


if clean_df["gloss"].isna().any():

    raise RuntimeError(
        "Missing gloss labels detected."
    )


if (clean_df["gloss"] == "").any():

    raise RuntimeError(
        "Blank gloss labels detected."
    )


ORIGINAL_CLEAN_N = len(clean_df)
ORIGINAL_CLASS_N = clean_df["gloss"].nunique()


print(
    "Original clean videos :",
    ORIGINAL_CLEAN_N
)

print(
    "Original clean glosses:",
    ORIGINAL_CLASS_N
)


assert ORIGINAL_CLEAN_N == 7048, (
    f"Expected 7048 clean videos, got {ORIGINAL_CLEAN_N}."
)


# ==================================================================================================
# 5. LOAD PERMANENTLY LOCKED L19 TEST SET
# ==================================================================================================

print()
print("=" * 150)
print("4. LOADING PERMANENT L19 LOCKED TEST SET")
print("=" * 150)

locked_df = pd.read_csv(
    LOCKED_TEST_PATH
)

locked_uid_col = detect_uid_column(
    locked_df
)

locked_gloss_col = detect_gloss_column(
    locked_df
)

locked_df = locked_df.copy()

locked_df["uid"] = canonical_uid_series(
    locked_df[locked_uid_col]
)

locked_df["gloss"] = (
    locked_df[locked_gloss_col]
    .astype(str)
    .str.strip()
)


LOCKED_TEST_N = len(locked_df)

locked_uids = set(
    locked_df["uid"]
)

clean_uids = set(
    clean_df["uid"]
)


print(
    "Locked-test rows:",
    LOCKED_TEST_N
)

print(
    "Locked-test unique UIDs:",
    len(locked_uids)
)

print(
    "Locked-test unique glosses:",
    locked_df["gloss"].nunique()
)


assert LOCKED_TEST_N == 10, (
    f"Expected 10 locked test rows, got {LOCKED_TEST_N}."
)

assert len(locked_uids) == 10, (
    "Locked test does not contain exactly 10 unique UIDs."
)


missing_locked_from_clean = (
    locked_uids
    -
    clean_uids
)


if missing_locked_from_clean:

    raise RuntimeError(
        "Some L19 locked-test UIDs are missing from "
        "the authoritative 7048 clean population:\n"
        f"{sorted(missing_locked_from_clean)}"
    )


print(
    "[PASS] All 10 locked-test UIDs exist in the original clean population."
)


# ==================================================================================================
# 6. VERIFY LOCKED MANIFEST GLOSS AGAINST AUTHORITATIVE DATASET
# ==================================================================================================

print()
print("=" * 150)
print("5. VERIFYING LOCKED-TEST LABEL CONSISTENCY")
print("=" * 150)

clean_uid_to_gloss = (
    clean_df
    .set_index("uid")["gloss"]
    .to_dict()
)

label_mismatches = []

for row in locked_df.itertuples(
    index=False
):

    uid = row.uid
    locked_gloss = row.gloss

    authoritative_gloss = (
        clean_uid_to_gloss[uid]
    )

    if locked_gloss != authoritative_gloss:

        label_mismatches.append(
            {
                "uid": uid,
                "locked_gloss": locked_gloss,
                "authoritative_gloss": authoritative_gloss,
            }
        )


if label_mismatches:

    raise RuntimeError(
        "Locked-test gloss labels disagree with "
        "the authoritative clean population:\n"
        f"{label_mismatches}"
    )


print(
    "[PASS] Locked-test UID/gloss labels match authoritative clean population."
)


# ==================================================================================================
# 7. FREEZE CLASS SELECTION USING ORIGINAL SUPPORT >= 4
# ==================================================================================================

print()
print("=" * 150)
print("6. FREEZING CLASS SELECTION USING ORIGINAL SUPPORT >= 4")
print("=" * 150)

# IMPORTANT:
#
# Class eligibility is computed BEFORE locked-test removal.
#
# This preserves the originally frozen scientific criterion:
#
#     support >= 4 in the original clean 7048 population.
#
# We are NOT choosing classes based on post-test-removal performance.

original_support = (
    clean_df
    .groupby(
        "gloss"
    )
    .size()
    .sort_index()
)

selected_glosses = (
    original_support[
        original_support
        >= SUPPORT_THRESHOLD
    ]
    .index
    .tolist()
)

selected_glosses = sorted(
    selected_glosses
)

ORIGINAL_SELECTED_CLASS_N = len(
    selected_glosses
)

original_supervised_df = (
    clean_df[
        clean_df["gloss"].isin(
            selected_glosses
        )
    ]
    .copy()
    .reset_index(drop=True)
)

ORIGINAL_SUPERVISED_N = len(
    original_supervised_df
)


print(
    "Support threshold:",
    SUPPORT_THRESHOLD
)

print(
    "Selected classes:",
    ORIGINAL_SELECTED_CLASS_N
)

print(
    "Original selected-class videos:",
    ORIGINAL_SUPERVISED_N
)

print(
    "Minimum original support:",
    int(
        original_support.loc[
            selected_glosses
        ].min()
    )
)

print(
    "Maximum original support:",
    int(
        original_support.loc[
            selected_glosses
        ].max()
    )
)


assert ORIGINAL_SELECTED_CLASS_N == 215, (
    "Expected 215 classes from the frozen support>=4 criterion, "
    f"got {ORIGINAL_SELECTED_CLASS_N}."
)

assert ORIGINAL_SUPERVISED_N == 1024, (
    "Expected 1024 videos in original support>=4 population, "
    f"got {ORIGINAL_SUPERVISED_N}."
)


print(
    "[PASS] Original support>=4 criterion reproduces 215 classes / 1024 videos."
)


# ==================================================================================================
# 8. DETERMINE WHERE THE 10 LOCKED TEST VIDEOS BELONG
# ==================================================================================================

print()
print("=" * 150)
print("7. LOCKED-TEST MEMBERSHIP IN 215-CLASS POPULATION")
print("=" * 150)

selected_gloss_set = set(
    selected_glosses
)

locked_df["in_215_class_set"] = (
    locked_df["gloss"]
    .isin(
        selected_gloss_set
    )
)

locked_in_supervised_df = (
    locked_df[
        locked_df["in_215_class_set"]
    ]
    .copy()
)

LOCKED_IN_SUPERVISED_N = len(
    locked_in_supervised_df
)


print(
    "Locked test total:",
    len(locked_df)
)

print(
    "Locked test inside 215-class population:",
    LOCKED_IN_SUPERVISED_N
)

print(
    "Locked test outside 215-class population:",
    len(locked_df) - LOCKED_IN_SUPERVISED_N
)


print()
print("LOCKED TEST DETAILS")
print("-" * 150)

display_columns = [
    "uid",
    "gloss",
    "in_215_class_set",
]

print(
    locked_df[
        display_columns
    ]
    .sort_values(
        ["gloss", "uid"]
    )
    .to_string(
        index=False
    )
)


# ==================================================================================================
# 9. REMOVE ALL 10 LOCKED TEST VIDEOS FROM L20 UNIVERSE
# ==================================================================================================

print()
print("=" * 150)
print("8. PERMANENTLY RESERVING ALL 10 L19 TEST VIDEOS")
print("=" * 150)

development_df = (
    clean_df[
        ~clean_df["uid"].isin(
            locked_uids
        )
    ]
    .copy()
    .reset_index(drop=True)
)

DEVELOPMENT_N = len(
    development_df
)


print(
    "Original clean population:",
    len(clean_df)
)

print(
    "Permanently locked test:",
    len(locked_uids)
)

print(
    "Remaining L20 development universe:",
    DEVELOPMENT_N
)


assert DEVELOPMENT_N == (
    ORIGINAL_CLEAN_N - 10
)

assert (
    set(development_df["uid"])
    &
    locked_uids
) == set()


print(
    "[PASS] Zero locked-test UIDs remain in the L20 development universe."
)


# ==================================================================================================
# 10. BUILD CORRECTED SUPERVISED POPULATION
# ==================================================================================================

print()
print("=" * 150)
print("9. BUILDING CORRECTED 215-CLASS SUPERVISED POPULATION")
print("=" * 150)

supervised_df = (
    development_df[
        development_df["gloss"].isin(
            selected_glosses
        )
    ]
    .copy()
    .reset_index(drop=True)
)

SUPERVISED_N = len(
    supervised_df
)

SUPERVISED_CLASS_N = (
    supervised_df["gloss"].nunique()
)

post_reservation_support = (
    supervised_df
    .groupby(
        "gloss"
    )
    .size()
    .sort_index()
)


print(
    "Corrected supervised videos:",
    SUPERVISED_N
)

print(
    "Corrected supervised classes:",
    SUPERVISED_CLASS_N
)

print(
    "Post-reservation min support:",
    int(
        post_reservation_support.min()
    )
)

print(
    "Post-reservation max support:",
    int(
        post_reservation_support.max()
    )
)


# Every selected class MUST still have at least two examples
# so that both train and validation can contain that class.

too_small = (
    post_reservation_support[
        post_reservation_support < 2
    ]
)

if len(too_small) > 0:

    raise RuntimeError(
        "After locked-test reservation, some selected classes "
        "have fewer than 2 examples and cannot be represented "
        "in both train and validation:\n"
        f"{too_small.to_dict()}"
    )


assert SUPERVISED_CLASS_N == 215, (
    f"Expected all 215 classes to survive, got {SUPERVISED_CLASS_N}."
)


expected_supervised_n = (
    ORIGINAL_SUPERVISED_N
    -
    LOCKED_IN_SUPERVISED_N
)

assert SUPERVISED_N == expected_supervised_n


print()
print(
    "[PASS] Corrected supervised population = "
    f"{SUPERVISED_N} videos / {SUPERVISED_CLASS_N} classes."
)


# ==================================================================================================
# 11. BUILD FROZEN CLASS MAPPING
# ==================================================================================================

print()
print("=" * 150)
print("10. BUILDING FROZEN CLASS MAPPING")
print("=" * 150)

class_mapping_df = pd.DataFrame(
    {
        "class_id": np.arange(
            len(selected_glosses),
            dtype=int
        ),
        "gloss": selected_glosses,
    }
)

gloss_to_class_id = dict(
    zip(
        class_mapping_df["gloss"],
        class_mapping_df["class_id"]
    )
)

supervised_df["class_id"] = (
    supervised_df["gloss"]
    .map(
        gloss_to_class_id
    )
    .astype(int)
)


assert (
    supervised_df["class_id"].isna().sum()
    == 0
)

assert (
    supervised_df["class_id"].nunique()
    == 215
)


print(
    "[PASS] Frozen class mapping created: IDs 0..214."
)


# ==================================================================================================
# 12. COMPUTE GLOBAL 70/30 TARGET
# ==================================================================================================

print()
print("=" * 150)
print("11. COMPUTING GLOBAL TRAIN / VALIDATION TARGET")
print("=" * 150)

TARGET_TRAIN_N = int(
    round(
        TRAIN_RATIO
        *
        SUPERVISED_N
    )
)

TARGET_VAL_N = (
    SUPERVISED_N
    -
    TARGET_TRAIN_N
)


print(
    "Supervised population:",
    SUPERVISED_N
)

print(
    "Target train:",
    TARGET_TRAIN_N
)

print(
    "Target validation:",
    TARGET_VAL_N
)

print(
    "Target train %:",
    100.0
    *
    TARGET_TRAIN_N
    /
    SUPERVISED_N
)

print(
    "Target validation %:",
    100.0
    *
    TARGET_VAL_N
    /
    SUPERVISED_N
)


# ==================================================================================================
# 13. DETERMINE PER-CLASS TRAIN COUNTS
# ==================================================================================================

print()
print("=" * 150)
print("12. DETERMINING PER-CLASS TRAIN COUNTS")
print("=" * 150)

allocation_rows = []

for gloss in selected_glosses:

    n = int(
        post_reservation_support.loc[
            gloss
        ]
    )

    exact_train = (
        TRAIN_RATIO
        *
        n
    )

    base_train = int(
        np.floor(
            exact_train
        )
    )

    # Guarantee:
    #     >=1 training sample
    #     >=1 validation sample

    base_train = max(
        1,
        min(
            base_train,
            n - 1
        )
    )

    remainder = (
        exact_train
        -
        np.floor(
            exact_train
        )
    )

    allocation_rows.append(
        {
            "gloss": gloss,
            "class_id": int(
                gloss_to_class_id[gloss]
            ),
            "n_total": n,
            "exact_train": float(
                exact_train
            ),
            "base_train": int(
                base_train
            ),
            "remainder": float(
                remainder
            ),
        }
    )


allocation_df = pd.DataFrame(
    allocation_rows
)

base_total = int(
    allocation_df["base_train"].sum()
)

delta = (
    TARGET_TRAIN_N
    -
    base_total
)


print(
    "Initial bounded floor allocation:",
    base_total
)

print(
    "Required global target:",
    TARGET_TRAIN_N
)

print(
    "Adjustment required:",
    delta
)


# ==================================================================================================
# 14. LARGEST-REMAINDER GLOBAL ADJUSTMENT
# ==================================================================================================

print()
print("=" * 150)
print("13. GLOBAL LARGEST-REMAINDER ADJUSTMENT")
print("=" * 150)

allocation_df["train_n"] = (
    allocation_df["base_train"]
    .astype(int)
)


if delta > 0:

    # Add train samples to classes with the largest fractional remainder
    # while preserving at least one validation example.

    candidates = (
        allocation_df[
            allocation_df["train_n"]
            <
            (
                allocation_df["n_total"]
                -
                1
            )
        ]
        .sort_values(
            by=[
                "remainder",
                "gloss",
            ],
            ascending=[
                False,
                True,
            ]
        )
        .index
        .tolist()
    )

    if len(candidates) < delta:

        raise RuntimeError(
            "Not enough eligible classes to increase train allocation "
            f"by {delta}."
        )

    for idx in candidates[:delta]:

        allocation_df.loc[
            idx,
            "train_n"
        ] += 1


elif delta < 0:

    # Remove train samples from classes with the smallest fractional remainder
    # while preserving at least one training example.

    remove_n = abs(
        delta
    )

    candidates = (
        allocation_df[
            allocation_df["train_n"]
            >
            1
        ]
        .sort_values(
            by=[
                "remainder",
                "gloss",
            ],
            ascending=[
                True,
                True,
            ]
        )
        .index
        .tolist()
    )

    if len(candidates) < remove_n:

        raise RuntimeError(
            "Not enough eligible classes to reduce train allocation "
            f"by {remove_n}."
        )

    for idx in candidates[:remove_n]:

        allocation_df.loc[
            idx,
            "train_n"
        ] -= 1


allocation_df["val_n"] = (
    allocation_df["n_total"]
    -
    allocation_df["train_n"]
)


assert (
    allocation_df["train_n"].sum()
    ==
    TARGET_TRAIN_N
)

assert (
    allocation_df["val_n"].sum()
    ==
    TARGET_VAL_N
)

assert (
    allocation_df["train_n"]
    >= 1
).all()

assert (
    allocation_df["val_n"]
    >= 1
).all()


print(
    "[PASS] Final global train allocation:",
    int(
        allocation_df["train_n"].sum()
    )
)

print(
    "[PASS] Final global validation allocation:",
    int(
        allocation_df["val_n"].sum()
    )
)

print(
    "[PASS] Every class has >=1 train and >=1 validation video."
)


# ==================================================================================================
# 15. DETERMINISTIC PER-CLASS SPLIT
# ==================================================================================================

print()
print("=" * 150)
print("14. CREATING DETERMINISTIC PER-CLASS SPLIT")
print("=" * 150)

train_parts = []
val_parts = []


for row in allocation_df.itertuples(
    index=False
):

    gloss = row.gloss

    class_df = (
        supervised_df[
            supervised_df["gloss"]
            ==
            gloss
        ]
        .copy()
        .sort_values(
            "uid"
        )
        .reset_index(drop=True)
    )

    n = len(
        class_df
    )

    assert n == row.n_total

    class_seed = stable_class_seed(
        gloss,
        SEED
    )

    rng = np.random.default_rng(
        class_seed
    )

    permutation = rng.permutation(
        n
    )

    train_indices = permutation[
        :row.train_n
    ]

    val_indices = permutation[
        row.train_n:
    ]

    class_train = (
        class_df
        .iloc[
            train_indices
        ]
        .copy()
    )

    class_val = (
        class_df
        .iloc[
            val_indices
        ]
        .copy()
    )

    class_train["split"] = "train"

    class_val["split"] = "validation"

    class_train["split_seed"] = int(
        class_seed
    )

    class_val["split_seed"] = int(
        class_seed
    )

    train_parts.append(
        class_train
    )

    val_parts.append(
        class_val
    )


train_df = (
    pd.concat(
        train_parts,
        ignore_index=True
    )
    .sort_values(
        [
            "class_id",
            "uid"
        ]
    )
    .reset_index(drop=True)
)

val_df = (
    pd.concat(
        val_parts,
        ignore_index=True
    )
    .sort_values(
        [
            "class_id",
            "uid"
        ]
    )
    .reset_index(drop=True)
)


print(
    "Train rows:",
    len(train_df)
)

print(
    "Validation rows:",
    len(val_df)
)

print(
    "Train classes:",
    train_df["gloss"].nunique()
)

print(
    "Validation classes:",
    val_df["gloss"].nunique()
)


# ==================================================================================================
# 16. SUPERVISED SPLIT INTEGRITY
# ==================================================================================================

print()
print("=" * 150)
print("15. SUPERVISED SPLIT INTEGRITY")
print("=" * 150)

train_uids = set(
    train_df["uid"]
)

val_uids = set(
    val_df["uid"]
)

supervised_uids = set(
    supervised_df["uid"]
)


train_val_overlap = (
    train_uids
    &
    val_uids
)

train_locked_overlap = (
    train_uids
    &
    locked_uids
)

val_locked_overlap = (
    val_uids
    &
    locked_uids
)

split_union = (
    train_uids
    |
    val_uids
)


checks = {
    "train_target_exact":
        len(train_df)
        ==
        TARGET_TRAIN_N,

    "validation_target_exact":
        len(val_df)
        ==
        TARGET_VAL_N,

    "train_validation_overlap_zero":
        len(train_val_overlap)
        ==
        0,

    "train_locked_test_overlap_zero":
        len(train_locked_overlap)
        ==
        0,

    "validation_locked_test_overlap_zero":
        len(val_locked_overlap)
        ==
        0,

    "split_union_equals_supervised_population":
        split_union
        ==
        supervised_uids,

    "train_all_215_classes":
        train_df["gloss"].nunique()
        ==
        215,

    "validation_all_215_classes":
        val_df["gloss"].nunique()
        ==
        215,

    "train_min_one_per_class":
        train_df.groupby(
            "gloss"
        ).size().min()
        >=
        1,

    "validation_min_one_per_class":
        val_df.groupby(
            "gloss"
        ).size().min()
        >=
        1,
}


failed_checks = []

for name, passed in checks.items():

    print(
        f"{name:<55}: "
        f"{'PASS' if passed else 'FAIL'}"
    )

    if not passed:

        failed_checks.append(
            name
        )


if failed_checks:

    raise RuntimeError(
        "Corrected supervised split integrity failure: "
        f"{failed_checks}"
    )


print()
print(
    "[PASS] Corrected supervised split is clean."
)


# ==================================================================================================
# 17. CREATE CORRECTED DOMAIN-ADAPTATION POPULATION
# ==================================================================================================

print()
print("=" * 150)
print("16. BUILDING STRICT L20C DOMAIN-ADAPTATION POPULATION")
print("=" * 150)

# Domain adaptation is allowed to see:
#
#   * supervised TRAIN videos
#   * singleton / rare-class videos
#   * other non-validation development videos
#
# Domain adaptation is NOT allowed to see:
#
#   * L19 locked test
#   * corrected L20 validation
#
# Therefore:
#
#     domain pool =
#         original clean population
#         - locked test
#         - corrected validation

domain_df = (
    clean_df[
        ~clean_df["uid"].isin(
            locked_uids
            |
            val_uids
        )
    ]
    .copy()
    .sort_values(
        "uid"
    )
    .reset_index(drop=True)
)

domain_uids = set(
    domain_df["uid"]
)

DOMAIN_N = len(
    domain_df
)


domain_locked_overlap = (
    domain_uids
    &
    locked_uids
)

domain_val_overlap = (
    domain_uids
    &
    val_uids
)

domain_train_overlap = (
    domain_uids
    &
    train_uids
)


print(
    "Domain-adaptation videos:",
    DOMAIN_N
)

print(
    "Domain ∩ locked test:",
    len(domain_locked_overlap)
)

print(
    "Domain ∩ validation:",
    len(domain_val_overlap)
)

print(
    "Domain ∩ supervised train:",
    len(domain_train_overlap)
)


expected_domain_n = (
    ORIGINAL_CLEAN_N
    -
    len(locked_uids)
    -
    len(val_uids)
)


assert DOMAIN_N == expected_domain_n

assert len(
    domain_locked_overlap
) == 0

assert len(
    domain_val_overlap
) == 0

assert (
    train_uids
    <=
    domain_uids
)


print()
print(
    "[PASS] L19 locked test excluded from domain adaptation."
)

print(
    "[PASS] L20 validation excluded from domain adaptation."
)

print(
    "[PASS] L20 supervised train is contained in the domain-adaptation pool."
)


# ==================================================================================================
# 18. COMPLETE POPULATION ACCOUNTING
# ==================================================================================================

print()
print("=" * 150)
print("17. COMPLETE POPULATION ACCOUNTING")
print("=" * 150)

# For domain-adaptation accounting:
#
# domain pool + validation + locked test = original clean population

accounting_union = (
    domain_uids
    |
    val_uids
    |
    locked_uids
)

pairwise_overlap_ok = (
    len(
        domain_uids
        &
        val_uids
    )
    ==
    0
    and
    len(
        domain_uids
        &
        locked_uids
    )
    ==
    0
    and
    len(
        val_uids
        &
        locked_uids
    )
    ==
    0
)


print(
    "Original clean population :",
    ORIGINAL_CLEAN_N
)

print(
    "Domain adaptation         :",
    DOMAIN_N
)

print(
    "L20 validation            :",
    len(val_uids)
)

print(
    "L19 locked test           :",
    len(locked_uids)
)

print(
    "Union size                :",
    len(accounting_union)
)


assert accounting_union == clean_uids

assert pairwise_overlap_ok


print()
print(
    "[PASS] domain + validation + locked test exactly reconstructs the 7048 clean videos."
)


# ==================================================================================================
# 19. ADD PROTOCOL METADATA TO MANIFESTS
# ==================================================================================================

print()
print("=" * 150)
print("18. ADDING PROTOCOL METADATA")
print("=" * 150)

train_df = train_df.copy()
val_df = val_df.copy()
domain_df = domain_df.copy()
locked_export_df = locked_df.copy()


train_df["protocol_role"] = (
    "L20_SUPERVISED_TRAIN"
)

val_df["protocol_role"] = (
    "L20_VALIDATION_ONLY"
)

domain_df["protocol_role"] = (
    "L20C_DOMAIN_ADAPTATION_ELIGIBLE"
)

locked_export_df["protocol_role"] = (
    "L19_PERMANENT_LOCKED_TEST"
)


# Explicitly mark whether domain videos belong to supervised train.

domain_df[
    "is_l20_supervised_train"
] = (
    domain_df["uid"]
    .isin(
        train_uids
    )
)


domain_df[
    "is_l20_validation"
] = False

domain_df[
    "is_l19_locked_test"
] = False


train_df[
    "is_l19_locked_test"
] = False

val_df[
    "is_l19_locked_test"
] = False


print(
    "[PASS] Protocol-role metadata attached."
)


# ==================================================================================================
# 20. SAVE MANIFESTS
# ==================================================================================================

print()
print("=" * 150)
print("19. SAVING CORRECTED MANIFESTS")
print("=" * 150)

ORIGINAL_SELECTED_PATH = (
    MANIFEST_DIR
    / "l20b_original_support4_215class_1024_manifest.csv"
)

SUPERVISED_MASTER_PATH = (
    MANIFEST_DIR
    / "l20b_corrected_supervised_master_manifest.csv"
)

TRAIN_PATH = (
    MANIFEST_DIR
    / "l20b_corrected_train_manifest.csv"
)

VAL_PATH = (
    MANIFEST_DIR
    / "l20b_corrected_validation_manifest.csv"
)

DOMAIN_PATH = (
    MANIFEST_DIR
    / "l20b_corrected_domain_pretraining_manifest.csv"
)

LOCKED_COPY_PATH = (
    MANIFEST_DIR
    / "l20b_permanent_l19_locked_test_manifest.csv"
)

CLASS_MAPPING_PATH = (
    MANIFEST_DIR
    / "l20b_corrected_class_mapping.csv"
)

ALLOCATION_PATH = (
    REPORT_DIR
    / "l20b_corrected_per_class_split.csv"
)


original_supervised_export = (
    original_supervised_df
    .copy()
)

original_supervised_export[
    "class_id"
] = (
    original_supervised_export[
        "gloss"
    ]
    .map(
        gloss_to_class_id
    )
    .astype(int)
)


supervised_master_export = (
    supervised_df
    .copy()
    .sort_values(
        [
            "class_id",
            "uid"
        ]
    )
)


original_supervised_export.to_csv(
    ORIGINAL_SELECTED_PATH,
    index=False
)

supervised_master_export.to_csv(
    SUPERVISED_MASTER_PATH,
    index=False
)

train_df.to_csv(
    TRAIN_PATH,
    index=False
)

val_df.to_csv(
    VAL_PATH,
    index=False
)

domain_df.to_csv(
    DOMAIN_PATH,
    index=False
)

locked_export_df.to_csv(
    LOCKED_COPY_PATH,
    index=False
)

class_mapping_df.to_csv(
    CLASS_MAPPING_PATH,
    index=False
)

allocation_df.to_csv(
    ALLOCATION_PATH,
    index=False
)


for path in [
    ORIGINAL_SELECTED_PATH,
    SUPERVISED_MASTER_PATH,
    TRAIN_PATH,
    VAL_PATH,
    DOMAIN_PATH,
    LOCKED_COPY_PATH,
    CLASS_MAPPING_PATH,
    ALLOCATION_PATH,
]:

    print(
        "[SAVED]",
        path
    )


# ==================================================================================================
# 21. SAVE CONFIGURATION
# ==================================================================================================

print()
print("=" * 150)
print("20. SAVING PROTOCOL CONFIGURATION")
print("=" * 150)

CONFIG_PATH = (
    CONFIG_DIR
    / "l20b_corrected_protocol_config.json"
)

config = {

    "stage":
        "L20B_CORRECTED",

    "status":
        "COMPLETE_AND_FROZEN",

    "purpose":
        "Strict full-CISLR protocol freeze with permanent L19 locked-test reservation",

    "seed":
        SEED,

    "support_threshold":
        SUPPORT_THRESHOLD,

    "class_selection_rule":
        "support >= 4 in original clean 7048-video population before locked-test removal",

    "train_ratio_target":
        TRAIN_RATIO,

    "validation_ratio_target":
        VAL_RATIO,

    "split_algorithm":
        (
            "Per-class bounded floor allocation followed by "
            "global largest-remainder adjustment and deterministic "
            "SHA256-derived per-class RNG permutation."
        ),

    "locked_test_policy":
        (
            "All 10 L19 locked-test UIDs are permanently excluded "
            "from L20 domain adaptation, supervised training, "
            "validation, model selection, and hyperparameter tuning."
        ),

    "domain_adaptation_policy":
        (
            "Domain adaptation may use L20 supervised-training videos "
            "without gloss labels, but may not use L20 validation "
            "or L19 locked-test videos."
        ),

    "original_clean_videos":
        int(
            ORIGINAL_CLEAN_N
        ),

    "original_clean_glosses":
        int(
            ORIGINAL_CLASS_N
        ),

    "original_support4_classes":
        int(
            ORIGINAL_SELECTED_CLASS_N
        ),

    "original_support4_videos":
        int(
            ORIGINAL_SUPERVISED_N
        ),

    "locked_test_videos":
        int(
            len(locked_uids)
        ),

    "locked_test_inside_support4_population":
        int(
            LOCKED_IN_SUPERVISED_N
        ),

    "development_universe_videos":
        int(
            DEVELOPMENT_N
        ),

    "corrected_supervised_videos":
        int(
            SUPERVISED_N
        ),

    "corrected_supervised_classes":
        int(
            SUPERVISED_CLASS_N
        ),

    "train_videos":
        int(
            len(train_df)
        ),

    "validation_videos":
        int(
            len(val_df)
        ),

    "domain_adaptation_videos":
        int(
            DOMAIN_N
        ),

    "locked_test_source":
        str(
            LOCKED_TEST_PATH
        ),

    "authoritative_clean_manifest":
        str(
            CLEAN_MANIFEST_PATH
        ),

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


save_json(
    config,
    CONFIG_PATH
)

print(
    "[SAVED]",
    CONFIG_PATH
)


# ==================================================================================================
# 22. SAVE SUMMARY REPORT
# ==================================================================================================

print()
print("=" * 150)
print("21. SAVING SUMMARY REPORT")
print("=" * 150)

SUMMARY_PATH = (
    REPORT_DIR
    / "l20b_corrected_summary.json"
)

summary = {

    "original_clean_population":
        int(
            ORIGINAL_CLEAN_N
        ),

    "original_unique_glosses":
        int(
            ORIGINAL_CLASS_N
        ),

    "support_threshold":
        int(
            SUPPORT_THRESHOLD
        ),

    "original_selected_classes":
        int(
            ORIGINAL_SELECTED_CLASS_N
        ),

    "original_selected_videos":
        int(
            ORIGINAL_SUPERVISED_N
        ),

    "locked_test_total":
        int(
            len(locked_uids)
        ),

    "locked_test_in_selected_classes":
        int(
            LOCKED_IN_SUPERVISED_N
        ),

    "corrected_supervised_population":
        int(
            SUPERVISED_N
        ),

    "corrected_supervised_classes":
        int(
            SUPERVISED_CLASS_N
        ),

    "train_videos":
        int(
            len(train_df)
        ),

    "validation_videos":
        int(
            len(val_df)
        ),

    "train_percentage":
        float(
            100.0
            *
            len(train_df)
            /
            SUPERVISED_N
        ),

    "validation_percentage":
        float(
            100.0
            *
            len(val_df)
            /
            SUPERVISED_N
        ),

    "domain_adaptation_videos":
        int(
            DOMAIN_N
        ),

    "train_validation_uid_overlap":
        int(
            len(
                train_uids
                &
                val_uids
            )
        ),

    "train_locked_test_uid_overlap":
        int(
            len(
                train_uids
                &
                locked_uids
            )
        ),

    "validation_locked_test_uid_overlap":
        int(
            len(
                val_uids
                &
                locked_uids
            )
        ),

    "domain_validation_uid_overlap":
        int(
            len(
                domain_uids
                &
                val_uids
            )
        ),

    "domain_locked_test_uid_overlap":
        int(
            len(
                domain_uids
                &
                locked_uids
            )
        ),

    "domain_supervised_train_overlap":
        int(
            len(
                domain_uids
                &
                train_uids
            )
        ),

    "minimum_train_examples_per_class":
        int(
            train_df
            .groupby(
                "gloss"
            )
            .size()
            .min()
        ),

    "maximum_train_examples_per_class":
        int(
            train_df
            .groupby(
                "gloss"
            )
            .size()
            .max()
        ),

    "minimum_validation_examples_per_class":
        int(
            val_df
            .groupby(
                "gloss"
            )
            .size()
            .min()
        ),

    "maximum_validation_examples_per_class":
        int(
            val_df
            .groupby(
                "gloss"
            )
            .size()
            .max()
        ),

    "training_performed":
        False,

    "inference_performed":
        False,

    "video_decoding_performed":
        False,

    "accuracy_used_for_selection":
        False,
}


save_json(
    summary,
    SUMMARY_PATH
)

print(
    "[SAVED]",
    SUMMARY_PATH
)


# ==================================================================================================
# 23. HASH ALL FROZEN ARTIFACTS
# ==================================================================================================

print()
print("=" * 150)
print("22. COMPUTING SHA256 HASHES")
print("=" * 150)

files_to_hash = {

    "original_support4_manifest":
        ORIGINAL_SELECTED_PATH,

    "corrected_supervised_master":
        SUPERVISED_MASTER_PATH,

    "corrected_train_manifest":
        TRAIN_PATH,

    "corrected_validation_manifest":
        VAL_PATH,

    "corrected_domain_manifest":
        DOMAIN_PATH,

    "permanent_locked_test_copy":
        LOCKED_COPY_PATH,

    "class_mapping":
        CLASS_MAPPING_PATH,

    "per_class_split":
        ALLOCATION_PATH,

    "protocol_config":
        CONFIG_PATH,

    "summary":
        SUMMARY_PATH,
}


hash_manifest = {}

for name, path in files_to_hash.items():

    digest = sha256_file(
        path
    )

    hash_manifest[name] = {
        "path": str(path),
        "sha256": digest,
    }

    print(
        f"{name:<35}: {digest}"
    )


HASH_MANIFEST_PATH = (
    HASH_DIR
    / "l20b_corrected_sha256_manifest.json"
)

save_json(
    hash_manifest,
    HASH_MANIFEST_PATH
)

print()
print(
    "[SAVED]",
    HASH_MANIFEST_PATH
)


# ==================================================================================================
# 24. MASTER FREEZE
# ==================================================================================================

print()
print("=" * 150)
print("23. WRITING MASTER FREEZE")
print("=" * 150)

MASTER_FREEZE_PATH = (
    OUT_DIR
    / "L20B_CORRECTED_MASTER_FREEZE.json"
)

master_freeze = {

    "stage":
        "L20B_CORRECTED",

    "status":
        "COMPLETE_AND_FROZEN",

    "scientific_role":
        "Dataset/protocol freeze only",

    "supersedes":
        str(
            OLD_L20B_DIR
        ),

    "reason_for_correction":
        (
            "Original L20B/L20C protocol did not permanently reserve "
            "all 10 L19 locked-test UIDs. Subsequent audit found that "
            "6 locked-test UIDs were exposed to the original L20C "
            "domain-adaptation run. This corrected protocol reserves "
            "all 10 before constructing L20 train, validation, and "
            "domain-adaptation populations."
        ),

    "class_selection_rule":
        "Original clean support >= 4",

    "selected_classes":
        int(
            ORIGINAL_SELECTED_CLASS_N
        ),

    "locked_test_reserved":
        int(
            len(locked_uids)
        ),

    "corrected_supervised_population":
        int(
            SUPERVISED_N
        ),

    "corrected_train_population":
        int(
            len(train_df)
        ),

    "corrected_validation_population":
        int(
            len(val_df)
        ),

    "corrected_domain_population":
        int(
            DOMAIN_N
        ),

    "training_performed":
        False,

    "inference_performed":
        False,

    "validation_accuracy_observed":
        False,

    "locked_test_accuracy_observed":
        False,

    "locked_test_used_for_training":
        False,

    "locked_test_used_for_model_selection":
        False,

    "hash_manifest":
        str(
            HASH_MANIFEST_PATH
        ),

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


save_json(
    master_freeze,
    MASTER_FREEZE_PATH
)

MASTER_FREEZE_SHA = sha256_file(
    MASTER_FREEZE_PATH
)


print(
    "[SAVED]",
    MASTER_FREEZE_PATH
)

print(
    "Master freeze SHA256:",
    MASTER_FREEZE_SHA
)


# ==================================================================================================
# 25. RELOAD SAVED MANIFESTS AND VERIFY DISK STATE
# ==================================================================================================

print()
print("=" * 150)
print("24. RELOADING SAVED ARTIFACTS FOR FINAL DISK VERIFICATION")
print("=" * 150)

disk_train = pd.read_csv(
    TRAIN_PATH
)

disk_val = pd.read_csv(
    VAL_PATH
)

disk_domain = pd.read_csv(
    DOMAIN_PATH
)

disk_locked = pd.read_csv(
    LOCKED_COPY_PATH
)

disk_classes = pd.read_csv(
    CLASS_MAPPING_PATH
)


disk_train_uids = set(
    canonical_uid_series(
        disk_train["uid"]
    )
)

disk_val_uids = set(
    canonical_uid_series(
        disk_val["uid"]
    )
)

disk_domain_uids = set(
    canonical_uid_series(
        disk_domain["uid"]
    )
)

disk_locked_uids = set(
    canonical_uid_series(
        disk_locked["uid"]
    )
)


disk_checks = {

    "train_rows_correct":
        len(disk_train)
        ==
        TARGET_TRAIN_N,

    "validation_rows_correct":
        len(disk_val)
        ==
        TARGET_VAL_N,

    "domain_rows_correct":
        len(disk_domain)
        ==
        DOMAIN_N,

    "locked_rows_10":
        len(disk_locked)
        ==
        10,

    "class_mapping_215":
        len(disk_classes)
        ==
        215,

    "train_validation_overlap_zero":
        len(
            disk_train_uids
            &
            disk_val_uids
        )
        ==
        0,

    "train_locked_overlap_zero":
        len(
            disk_train_uids
            &
            disk_locked_uids
        )
        ==
        0,

    "validation_locked_overlap_zero":
        len(
            disk_val_uids
            &
            disk_locked_uids
        )
        ==
        0,

    "domain_validation_overlap_zero":
        len(
            disk_domain_uids
            &
            disk_val_uids
        )
        ==
        0,

    "domain_locked_overlap_zero":
        len(
            disk_domain_uids
            &
            disk_locked_uids
        )
        ==
        0,

    "all_train_inside_domain":
        disk_train_uids
        <=
        disk_domain_uids,
}


final_failures = []

for name, passed in disk_checks.items():

    print(
        f"{name:<55}: "
        f"{'PASS' if passed else 'FAIL'}"
    )

    if not passed:

        final_failures.append(
            name
        )


if final_failures:

    raise RuntimeError(
        "L20B-CORRECTED final disk verification failed: "
        f"{final_failures}"
    )


# ==================================================================================================
# 26. FINAL PROTOCOL SUMMARY
# ==================================================================================================

print()
print("=" * 150)
print("25. FINAL CORRECTED PROTOCOL")
print("=" * 150)

print()
print(
    f"Original clean CISLR population       : {ORIGINAL_CLEAN_N}"
)

print(
    f"Original unique glosses               : {ORIGINAL_CLASS_N}"
)

print()
print(
    f"Frozen class criterion                : original support >= {SUPPORT_THRESHOLD}"
)

print(
    f"Selected classes                      : {ORIGINAL_SELECTED_CLASS_N}"
)

print(
    f"Original selected-class population    : {ORIGINAL_SUPERVISED_N}"
)

print()
print(
    f"Permanent L19 locked test             : {len(locked_uids)}"
)

print(
    f"Locked test inside selected classes   : {LOCKED_IN_SUPERVISED_N}"
)

print()
print(
    f"Corrected supervised population       : {SUPERVISED_N}"
)

print(
    f"Corrected supervised classes          : {SUPERVISED_CLASS_N}"
)

print()
print(
    f"L20 supervised TRAIN                  : {len(train_df)}"
)

print(
    f"L20 VALIDATION ONLY                   : {len(val_df)}"
)

print(
    f"L20C DOMAIN ADAPTATION ELIGIBLE       : {DOMAIN_N}"
)

print()
print(
    f"Actual train ratio                    : "
    f"{100.0 * len(train_df) / SUPERVISED_N:.4f}%"
)

print(
    f"Actual validation ratio               : "
    f"{100.0 * len(val_df) / SUPERVISED_N:.4f}%"
)

print()
print(
    "Train ∩ Validation                   :",
    len(
        train_uids
        &
        val_uids
    )
)

print(
    "Train ∩ Locked test                  :",
    len(
        train_uids
        &
        locked_uids
    )
)

print(
    "Validation ∩ Locked test             :",
    len(
        val_uids
        &
        locked_uids
    )
)

print(
    "Domain ∩ Validation                  :",
    len(
        domain_uids
        &
        val_uids
    )
)

print(
    "Domain ∩ Locked test                 :",
    len(
        domain_uids
        &
        locked_uids
    )
)

print()
print(
    "Training performed                   : NO"
)

print(
    "Inference performed                  : NO"
)

print(
    "Video decoding performed             : NO"
)

print(
    "Validation accuracy observed         : NO"
)

print(
    "Locked-test accuracy observed        : NO"
)

print()
print(
    "Master freeze:"
)

print(
    MASTER_FREEZE_PATH
)

print()
print(
    "Master freeze SHA256:"
)

print(
    MASTER_FREEZE_SHA
)


# ==================================================================================================
# 27. EXPECTED-COUNT INFORMATION
# ==================================================================================================

print()
print("=" * 150)
print("26. EXPECTED-COUNT CHECK")
print("=" * 150)

# Based on the previously observed dataset structure,
# we expect all 10 locked videos to belong to the 215 selected classes.
#
# IF that is verified above:
#
#   supervised = 1024 - 10 = 1014
#   train      = round(0.70 * 1014) = 710
#   validation = 304
#   domain     = 7048 - 10 - 304 = 6734
#
# We do NOT silently force those values.
# The actual dataset was measured above.

if LOCKED_IN_SUPERVISED_N == 10:

    expected_values = {
        "supervised": 1014,
        "train": 710,
        "validation": 304,
        "domain": 6734,
    }

    measured_values = {
        "supervised": SUPERVISED_N,
        "train": len(train_df),
        "validation": len(val_df),
        "domain": DOMAIN_N,
    }

    print(
        "All 10 locked-test videos belong to the selected 215-class population."
    )

    print()

    for key in expected_values:

        expected = expected_values[key]
        measured = measured_values[key]

        status = (
            "PASS"
            if expected == measured
            else "FAIL"
        )

        print(
            f"{key:<20}: "
            f"expected={expected:<6} "
            f"measured={measured:<6} "
            f"{status}"
        )

        if expected != measured:

            raise RuntimeError(
                f"Expected-count mismatch for {key}: "
                f"expected={expected}, measured={measured}"
            )

else:

    print(
        "[INFO] Not all 10 locked-test videos belong to the selected "
        "215-class population."
    )

    print(
        "[INFO] Therefore the generic measured counts above are authoritative."
    )


# ==================================================================================================
# 28. FINAL STATUS
# ==================================================================================================

print()
print("=" * 150)
print("L20B-CORRECTED — COMPLETE AND FROZEN")
print("=" * 150)

print()
print(
    "[PASS] Original 7048-video clean population verified."
)

print(
    "[PASS] All 10 L19 locked-test UIDs permanently reserved."
)

print(
    "[PASS] Frozen support>=4 class criterion reproduced."
)

print(
    "[PASS] Corrected supervised population constructed."
)

print(
    "[PASS] Deterministic global ~70/30 supervised split constructed."
)

print(
    "[PASS] Every selected class represented in train and validation."
)

print(
    "[PASS] Train/validation UID overlap = 0."
)

print(
    "[PASS] Locked-test overlap with train = 0."
)

print(
    "[PASS] Locked-test overlap with validation = 0."
)

print(
    "[PASS] Locked-test overlap with domain adaptation = 0."
)

print(
    "[PASS] Validation overlap with domain adaptation = 0."
)

print(
    "[PASS] No training performed."
)

print(
    "[PASS] No inference performed."
)

print(
    "[PASS] No validation/test accuracy observed."
)

print()
print(
    "NEXT STAGE:"
)

print(
    "L20C-CORRECTED — CISLR domain adaptation using ONLY the "
    "new corrected domain manifest."
)

print()
print(
    "IMPORTANT:"
)

print(
    "DO NOT resume from the old 6741-video L20C checkpoint."
)

print(
    "The corrected L20C must start from the original pretrained "
    "VideoMAE base model."
)

print("=" * 150)

PARASURG / CISLR — L20B-CORRECTED
STRICT FULL-CISLR PROTOCOL FREEZE

Project root:
/home/dipshikha/Music/cao/CISLR_RESEARCH/CISLR_LANDMARK_MODEL

L20A audit:
/home/dipshikha/Music/cao/CISLR_RESEARCH/CISLR_LANDMARK_MODEL/audit/l20a_full_cislr_dataset_audit

L19 locked test:
/home/dipshikha/Music/cao/CISLR_RESEARCH/CISLR_LANDMARK_MODEL/audit/l18b_10class_dataset/manifests/l18b_LOCKED_test_manifest.csv

Corrected L20B output:
/home/dipshikha/Music/cao/CISLR_RESEARCH/CISLR_LANDMARK_MODEL/audit/l20b_corrected_strict_protocol_freeze

Seed              : 42
Support threshold : 4
Target train ratio: 0.7
Target val ratio  : 0.3

1. BASIC PATH VERIFICATION
[PASS] Project root exists.
[PASS] L20A audit exists.
[PASS] L19 locked-test manifest exists.

2. LOCATING AUTHORITATIVE L20A CLEAN MANIFEST
CSV candidates discovered: 18

[FOUND] L20A clean manifest:
/home/dipshikha/Music/cao/CISLR_RESEARCH/CISLR_LANDMARK_MODEL/audit/l20a_full_cislr_dataset_audit/manifests/l20a_clean_full_cislr_manifest.csv



In [9]:
# ==================================================================================================
# PARASURG / CISLR
# L20C — CORRECTED STRICT DOMAIN ADAPTATION
# ROBUST NO-CACHE VERSION
#
# FINAL PROTOCOL:
#
#   Domain adaptation videos : 6734
#   Validation videos        : 304  (EXCLUDED)
#   L19 locked test videos   : 10   (EXCLUDED)
#
#   Epochs                   : 15
#   Physical batch size      : 1
#   Gradient accumulation    : 8
#   Effective batch size     : 8
#
#   GPU                      : CUDA
#   AMP                      : FP16
#   CPU decoding             : parallel DataLoader workers
#   Frame cache              : NONE
#
#   Model initialization     : ORIGINAL pretrained VideoMAE
#   Old contaminated L20C    : NEVER USED
#
# ROBUST VIDEO DECODING:
#
#   MP4
#    |
#    +--> sequentially decode all readable frames
#    |
#    +--> uniformly select 16 frames
#    |
#    +--> short videos automatically repeat frames
#
# This avoids unreliable OpenCV random seeking such as:
#
#   Failed to decode frame 84 from gK-FT0qEvnw_2.mp4
#
# SCIENTIFIC NOTE:
#
# This is unlabeled temporal-consistency domain adaptation of a VideoMAE backbone.
# It is NOT VideoMAE masked-autoencoder pretraining.
#
# ==================================================================================================


# ==================================================================================================
# 0. ENVIRONMENT
# ==================================================================================================

import os

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

os.environ.setdefault(
    "PYTORCH_CUDA_ALLOC_CONF",
    "expandable_segments:True"
)


# ==================================================================================================
# 1. IMPORTS
# ==================================================================================================

import gc
import cv2
import json
import time
import random
import hashlib
import shutil
import warnings

from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

import torch
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader

from transformers import (
    VideoMAEModel,
    VideoMAEImageProcessor
)


warnings.filterwarnings("default")


try:
    cv2.setNumThreads(0)
except Exception:
    pass


# ==================================================================================================
# 2. CONFIGURATION
# ==================================================================================================

SEED = 42


MODEL_NAME = (
    "MCG-NJU/videomae-base-finetuned-kinetics"
)


NUM_FRAMES = 16


EPOCHS = 15


# --------------------------------------------------------------------------------------------------
# RTX 4050 Laptop 5.64 GB
# --------------------------------------------------------------------------------------------------

BATCH_SIZE = 1

GRAD_ACCUM_STEPS = 8

EFFECTIVE_BATCH_SIZE = (
    BATCH_SIZE
    *
    GRAD_ACCUM_STEPS
)


# --------------------------------------------------------------------------------------------------
# OPTIMIZATION
# --------------------------------------------------------------------------------------------------

LEARNING_RATE = 1e-5

WEIGHT_DECAY = 1e-4

MAX_GRAD_NORM = 1.0


# --------------------------------------------------------------------------------------------------
# VIDEOMAE
# --------------------------------------------------------------------------------------------------

NUM_TRAINABLE_BLOCKS = 4


# --------------------------------------------------------------------------------------------------
# PARALLEL CPU DECODING
# --------------------------------------------------------------------------------------------------

NUM_WORKERS = 4

PREFETCH_FACTOR = 2

PIN_MEMORY = True

PERSISTENT_WORKERS = True


# --------------------------------------------------------------------------------------------------
# CUDA
# --------------------------------------------------------------------------------------------------

USE_AMP = True

USE_TF32 = True


# --------------------------------------------------------------------------------------------------
# LOGGING
# --------------------------------------------------------------------------------------------------

LOG_EVERY = 100


# ==================================================================================================
# 3. PATHS
# ==================================================================================================

ROOT = Path(
    "/home/dipshikha/Music/cao/CISLR_RESEARCH/"
    "CISLR_LANDMARK_MODEL"
)


VIDEO_ROOT = Path(
    "/home/dipshikha/Music/cao/CISLR_v1.5-a_videos"
)


# --------------------------------------------------------------------------------------------------
# CORRECTED L20B
# --------------------------------------------------------------------------------------------------

L20B_DIR = (
    ROOT
    / "audit"
    / "l20b_corrected_strict_protocol_freeze"
)


DOMAIN_MANIFEST = (
    L20B_DIR
    / "manifests"
    / "l20b_corrected_domain_pretraining_manifest.csv"
)


VALIDATION_MANIFEST = (
    L20B_DIR
    / "manifests"
    / "l20b_corrected_validation_manifest.csv"
)


LOCKED_TEST_MANIFEST = (
    L20B_DIR
    / "manifests"
    / "l20b_permanent_l19_locked_test_manifest.csv"
)


L20B_FREEZE = (
    L20B_DIR
    / "L20B_CORRECTED_MASTER_FREEZE.json"
)


# --------------------------------------------------------------------------------------------------
# CORRECTED L20C OUTPUT
# --------------------------------------------------------------------------------------------------

OUT_DIR = (
    ROOT
    / "audit"
    / "l20c_corrected_nocache_domain_adaptation"
)


MODEL_DIR = OUT_DIR / "models"

MANIFEST_DIR = OUT_DIR / "manifests"

REPORT_DIR = OUT_DIR / "reports"

CONFIG_DIR = OUT_DIR / "config"

HASH_DIR = OUT_DIR / "hashes"


for directory in [

    OUT_DIR,
    MODEL_DIR,
    MANIFEST_DIR,
    REPORT_DIR,
    CONFIG_DIR,
    HASH_DIR

]:

    directory.mkdir(
        parents=True,
        exist_ok=True
    )


# --------------------------------------------------------------------------------------------------
# OUTPUT FILES
# --------------------------------------------------------------------------------------------------

RESUME_CHECKPOINT = (
    MODEL_DIR
    / "l20c_corrected_resume.pt"
)


BEST_CHECKPOINT = (
    MODEL_DIR
    / "l20c_corrected_best.pt"
)


FINAL_BACKBONE = (
    MODEL_DIR
    / "l20c_corrected_final_backbone.pt"
)


ELIGIBLE_MANIFEST_PATH = (
    MANIFEST_DIR
    / "l20c_corrected_domain_6734.csv"
)


HISTORY_PATH = (
    REPORT_DIR
    / "l20c_corrected_training_history.csv"
)


CONFIG_PATH = (
    CONFIG_DIR
    / "l20c_corrected_config.json"
)


SUMMARY_PATH = (
    REPORT_DIR
    / "l20c_corrected_summary.json"
)


HASH_PATH = (
    HASH_DIR
    / "l20c_corrected_hashes.json"
)


MASTER_FREEZE_PATH = (
    OUT_DIR
    / "L20C_CORRECTED_MASTER_FREEZE.json"
)


# ==================================================================================================
# 4. HELPER FUNCTIONS
# ==================================================================================================

def gb(x):

    return x / (1024 ** 3)


def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        while True:

            block = f.read(
                1024 * 1024
            )

            if not block:
                break

            h.update(block)

    return h.hexdigest()


def save_json(data, path):

    with open(
        path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            data,
            f,
            indent=2,
            ensure_ascii=False
        )


def uid_set(df):

    return set(
        df["uid"]
        .astype(str)
        .str.strip()
    )


# ==================================================================================================
# 5. RANDOM SEEDS
# ==================================================================================================

random.seed(SEED)

np.random.seed(SEED)

torch.manual_seed(SEED)


if torch.cuda.is_available():

    torch.cuda.manual_seed_all(SEED)


# ==================================================================================================
# 6. CUDA CONFIGURATION
# ==================================================================================================

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


if DEVICE.type != "cuda":

    raise RuntimeError(
        "CUDA is unavailable. "
        "L20C should not be run on CPU."
    )


torch.backends.cudnn.benchmark = True


try:

    torch.backends.cuda.matmul.allow_tf32 = (
        USE_TF32
    )

except Exception:
    pass


try:

    torch.backends.cudnn.allow_tf32 = (
        USE_TF32
    )

except Exception:
    pass


try:

    torch.set_float32_matmul_precision(
        "high"
    )

except Exception:
    pass


# ==================================================================================================
# 7. STARTUP REPORT
# ==================================================================================================

print("=" * 120)

print(
    "PARASURG / CISLR — L20C CORRECTED"
)

print(
    "ROBUST NO-CACHE 15-EPOCH DOMAIN ADAPTATION"
)

print("=" * 120)


print()

print(
    "PyTorch              :",
    torch.__version__
)

print(
    "OpenCV               :",
    cv2.__version__
)

print(
    "CUDA available       :",
    torch.cuda.is_available()
)

print(
    "Device               :",
    DEVICE
)

print(
    "GPU                  :",
    torch.cuda.get_device_name(0)
)


gpu_properties = (
    torch.cuda.get_device_properties(0)
)


print(
    "GPU memory           :",
    f"{gb(gpu_properties.total_memory):.2f} GB"
)


disk_total, disk_used, disk_free = (
    shutil.disk_usage("/home")
)


print()

print(
    "Disk free            :",
    f"{gb(disk_free):.2f} GB"
)

print(
    "CPU count            :",
    os.cpu_count()
)

print(
    "DataLoader workers   :",
    NUM_WORKERS
)

print(
    "Batch size           :",
    BATCH_SIZE
)

print(
    "Gradient accumulation:",
    GRAD_ACCUM_STEPS
)

print(
    "Effective batch      :",
    EFFECTIVE_BATCH_SIZE
)

print(
    "Epochs               :",
    EPOCHS
)

print(
    "Frame cache          : NONE"
)


# ==================================================================================================
# 8. VERIFY CORRECTED L20B
# ==================================================================================================

print()

print("=" * 120)

print(
    "1. VERIFYING CORRECTED L20B"
)

print("=" * 120)


required_files = [

    DOMAIN_MANIFEST,
    VALIDATION_MANIFEST,
    LOCKED_TEST_MANIFEST,
    L20B_FREEZE

]


for path in required_files:

    if not path.exists():

        raise FileNotFoundError(
            f"Missing required file:\n{path}"
        )

    print(
        "[FOUND]",
        path
    )


with open(
    L20B_FREEZE,
    "r",
    encoding="utf-8"
) as f:

    l20b_freeze = json.load(f)


if (
    l20b_freeze.get("status")
    !=
    "COMPLETE_AND_FROZEN"
):

    raise RuntimeError(
        "Corrected L20B is not frozen."
    )


print()

print(
    "[PASS] Corrected L20B verified."
)


# ==================================================================================================
# 9. LOAD FROZEN MANIFESTS
# ==================================================================================================

print()

print("=" * 120)

print(
    "2. LOADING FROZEN PROTOCOL"
)

print("=" * 120)


domain_df = pd.read_csv(
    DOMAIN_MANIFEST
)


validation_df = pd.read_csv(
    VALIDATION_MANIFEST
)


locked_df = pd.read_csv(
    LOCKED_TEST_MANIFEST
)


for df in [

    domain_df,
    validation_df,
    locked_df

]:

    if "uid" not in df.columns:

        raise RuntimeError(
            "Manifest missing UID column."
        )

    df["uid"] = (
        df["uid"]
        .astype(str)
        .str.strip()
    )


domain_uids = uid_set(
    domain_df
)

validation_uids = uid_set(
    validation_df
)

locked_uids = uid_set(
    locked_df
)


print(
    "Domain      :",
    len(domain_df)
)

print(
    "Validation  :",
    len(validation_df)
)

print(
    "Locked test :",
    len(locked_df)
)


# ==================================================================================================
# 10. STRICT ISOLATION CHECK
# ==================================================================================================

print()

print("=" * 120)

print(
    "3. STRICT ISOLATION CHECK"
)

print("=" * 120)


checks = {

    "domain_rows_6734":
        len(domain_df) == 6734,

    "domain_unique_6734":
        len(domain_uids) == 6734,

    "validation_rows_304":
        len(validation_df) == 304,

    "validation_unique_304":
        len(validation_uids) == 304,

    "locked_rows_10":
        len(locked_df) == 10,

    "locked_unique_10":
        len(locked_uids) == 10,

    "domain_validation_overlap_zero":
        len(
            domain_uids
            &
            validation_uids
        ) == 0,

    "domain_locked_overlap_zero":
        len(
            domain_uids
            &
            locked_uids
        ) == 0,

    "validation_locked_overlap_zero":
        len(
            validation_uids
            &
            locked_uids
        ) == 0

}


for name, passed in checks.items():

    print(
        f"{name:<50}: "
        f"{'PASS' if passed else 'FAIL'}"
    )


if not all(
    checks.values()
):

    failed = [

        name
        for name, passed in checks.items()
        if not passed

    ]

    raise RuntimeError(
        f"Protocol integrity failure: {failed}"
    )


print()

print(
    "[PASS] 6734 / 304 / 10 protocol preserved."
)


# ==================================================================================================
# 11. INDEX VIDEO FILES
# ==================================================================================================

print()

print("=" * 120)

print(
    "4. INDEXING ORIGINAL CISLR VIDEOS"
)

print("=" * 120)


video_files = list(
    VIDEO_ROOT.rglob("*.mp4")
)


print(
    "MP4 files found:",
    len(video_files)
)


video_by_name = {}

video_by_stem = {}


for path in video_files:

    video_by_name.setdefault(
        path.name,
        path
    )

    video_by_stem.setdefault(
        path.stem,
        path
    )


# ==================================================================================================
# 12. VIDEO PATH RESOLVER
# ==================================================================================================

def resolve_video(row):

    uid = str(
        row["uid"]
    ).strip()


    for column in [

        "video_path",
        "path",
        "filepath",
        "file_path",
        "resolved_path"

    ]:

        if column not in row.index:
            continue


        value = row[column]


        if pd.isna(value):
            continue


        value = str(value).strip()


        if not value:
            continue


        p = Path(value)


        if p.exists():
            return p


        candidate = (
            VIDEO_ROOT
            /
            value
        )


        if candidate.exists():
            return candidate


    exact_filename = (
        uid
        +
        ".mp4"
    )


    if exact_filename in video_by_name:

        return video_by_name[
            exact_filename
        ]


    if uid in video_by_stem:

        return video_by_stem[
            uid
        ]


    return None


resolved_paths = []

missing_uids = []


for _, row in domain_df.iterrows():

    path = resolve_video(row)


    if path is None:

        resolved_paths.append(
            None
        )

        missing_uids.append(
            row["uid"]
        )

    else:

        resolved_paths.append(
            str(path)
        )


domain_df = domain_df.copy()


domain_df[
    "resolved_video_path"
] = resolved_paths


print(
    "Resolved:",
    domain_df[
        "resolved_video_path"
    ].notna().sum()
)

print(
    "Missing :",
    len(missing_uids)
)


if missing_uids:

    print(
        "Missing UIDs:",
        missing_uids[:20]
    )

    raise RuntimeError(
        "Not all 6734 domain videos resolved."
    )


domain_df.to_csv(
    ELIGIBLE_MANIFEST_PATH,
    index=False
)


print()

print(
    "[SAVED]",
    ELIGIBLE_MANIFEST_PATH
)


# ==================================================================================================
# 13. ROBUST VIDEO DECODER
#
# IMPORTANT:
#
# We intentionally DO NOT use:
#
#     cap.set(CAP_PROP_POS_FRAMES, frame_index)
#
# for every selected frame.
#
# Some compressed MP4s report a nominal frame count but cannot reliably seek
# to an arbitrary compressed frame.
#
# Instead:
#
#     sequential decode -> actual readable frames -> uniform sample
#
# ==================================================================================================

def decode_uniform_frames(
    video_path,
    num_frames=16
):

    video_path = str(
        video_path
    )


    cap = cv2.VideoCapture(
        video_path
    )


    if not cap.isOpened():

        raise RuntimeError(
            f"Could not open video: {video_path}"
        )


    decoded_frames = []


    try:

        while True:

            ok, frame = cap.read()


            if not ok:

                break


            if (
                frame is None
                or
                frame.size == 0
            ):

                continue


            try:

                frame = cv2.cvtColor(
                    frame,
                    cv2.COLOR_BGR2RGB
                )

            except Exception:

                continue


            decoded_frames.append(
                frame
            )


    finally:

        cap.release()


    actual_total = len(
        decoded_frames
    )


    if actual_total == 0:

        raise RuntimeError(
            f"Video contains zero decodable frames: "
            f"{video_path}"
        )


    indices = np.linspace(
        0,
        actual_total - 1,
        num_frames
    ).astype(
        np.int64
    )


    sampled = [

        decoded_frames[
            int(i)
        ]

        for i in indices

    ]


    if len(sampled) != num_frames:

        raise RuntimeError(
            f"Decoder produced {len(sampled)} "
            f"instead of {num_frames}: "
            f"{video_path}"
        )


    return sampled


print()

print(
    "[PASS] Robust sequential decoder installed."
)


# ==================================================================================================
# 14. TEST PREVIOUSLY FAILED VIDEO
# ==================================================================================================

print()

print("=" * 120)

print(
    "5. TESTING PREVIOUSLY FAILED VIDEO"
)

print("=" * 120)


KNOWN_PROBLEM_VIDEO = (
    VIDEO_ROOT
    / "gK-FT0qEvnw_2.mp4"
)


if KNOWN_PROBLEM_VIDEO.exists():

    test_frames = decode_uniform_frames(
        KNOWN_PROBLEM_VIDEO,
        NUM_FRAMES
    )


    print(
        "Video :",
        KNOWN_PROBLEM_VIDEO.name
    )

    print(
        "Frames:",
        len(test_frames)
    )

    print(
        "Shape :",
        test_frames[0].shape
    )


    if len(test_frames) != NUM_FRAMES:

        raise RuntimeError(
            "Known problem video still fails."
        )


    print()

    print(
        "[PASS] Previously failing video "
        "now decodes successfully."
    )


    del test_frames


else:

    print(
        "[INFO] Previously failing video "
        "not found at expected direct path."
    )

    print(
        "[INFO] Training path resolver "
        "will still use robust decoder."
    )


# ==================================================================================================
# 15. LOAD VIDEOMAE PROCESSOR
# ==================================================================================================

print()

print("=" * 120)

print(
    "6. LOADING VIDEOMAE PROCESSOR"
)

print("=" * 120)


processor = (
    VideoMAEImageProcessor
    .from_pretrained(
        MODEL_NAME
    )
)


print(
    "[PASS]",
    MODEL_NAME
)


# ==================================================================================================
# 16. DATASET
# ==================================================================================================

class CISLRDomainDataset(
    Dataset
):

    def __init__(
        self,
        dataframe,
        processor
    ):

        self.df = (
            dataframe
            .reset_index(drop=True)
            .copy()
        )

        self.processor = processor


    def __len__(self):

        return len(
            self.df
        )


    @staticmethod
    def temporal_shift(
        frames
    ):

        shift = random.choice(
            [-1, 0, 1]
        )


        if shift == 0:

            return list(
                frames
            )


        n = len(frames)


        shifted = []


        for i in range(n):

            source_index = min(
                max(
                    i + shift,
                    0
                ),
                n - 1
            )


            shifted.append(
                frames[
                    source_index
                ]
            )


        return shifted


    def __getitem__(
        self,
        index
    ):

        row = self.df.iloc[
            index
        ]


        uid = str(
            row["uid"]
        )


        video_path = row[
            "resolved_video_path"
        ]


        # ------------------------------------------------------------------------------------------
        # Decode video ONCE.
        # ------------------------------------------------------------------------------------------

        frames_a = decode_uniform_frames(
            video_path,
            NUM_FRAMES
        )


        # ------------------------------------------------------------------------------------------
        # Create second temporal view WITHOUT decoding MP4 again.
        # ------------------------------------------------------------------------------------------

        frames_b = self.temporal_shift(
            frames_a
        )


        # ------------------------------------------------------------------------------------------
        # VIDEOMAE PREPROCESSING
        # ------------------------------------------------------------------------------------------

        pixel_a = self.processor(

            frames_a,

            return_tensors="pt"

        )[
            "pixel_values"
        ][0]


        pixel_b = self.processor(

            frames_b,

            return_tensors="pt"

        )[
            "pixel_values"
        ][0]


        return {

            "uid":
                uid,

            "view_a":
                pixel_a,

            "view_b":
                pixel_b

        }


# ==================================================================================================
# 17. DATASET
# ==================================================================================================

dataset = CISLRDomainDataset(
    domain_df,
    processor
)


assert len(dataset) == 6734


# ==================================================================================================
# 18. WORKER SEEDING
# ==================================================================================================

def seed_worker(
    worker_id
):

    worker_seed = (
        SEED
        +
        worker_id
    )


    random.seed(
        worker_seed
    )


    np.random.seed(
        worker_seed
    )


# ==================================================================================================
# 19. DATALOADER
# ==================================================================================================

loader_generator = torch.Generator()


loader_generator.manual_seed(
    SEED
)


loader_kwargs = {

    "dataset":
        dataset,

    "batch_size":
        BATCH_SIZE,

    "shuffle":
        True,

    "num_workers":
        NUM_WORKERS,

    "pin_memory":
        PIN_MEMORY,

    "drop_last":
        False,

    "worker_init_fn":
        seed_worker,

    "generator":
        loader_generator

}


if NUM_WORKERS > 0:

    loader_kwargs[
        "persistent_workers"
    ] = PERSISTENT_WORKERS


    loader_kwargs[
        "prefetch_factor"
    ] = PREFETCH_FACTOR


train_loader = DataLoader(
    **loader_kwargs
)


print()

print("=" * 120)

print(
    "7. DATALOADER READY"
)

print("=" * 120)


print(
    "Dataset videos      :",
    len(dataset)
)

print(
    "Batches per epoch   :",
    len(train_loader)
)

print(
    "CPU workers         :",
    NUM_WORKERS
)

print(
    "Physical batch      :",
    BATCH_SIZE
)

print(
    "Gradient accumulation:",
    GRAD_ACCUM_STEPS
)

print(
    "Effective batch     :",
    EFFECTIVE_BATCH_SIZE
)


# ==================================================================================================
# 20. DATALOADER SANITY CHECK
#
# This forces the multiprocessing workers to start BEFORE loading VideoMAE onto CUDA.
# ==================================================================================================

print()

print("=" * 120)

print(
    "8. DATALOADER SANITY CHECK"
)

print("=" * 120)


sanity_start = time.time()


sanity_iterator = iter(
    train_loader
)


sanity_batch = next(
    sanity_iterator
)


print(
    "UID:",
    sanity_batch["uid"][0]
)


print(
    "View A:",
    tuple(
        sanity_batch[
            "view_a"
        ].shape
    )
)


print(
    "View B:",
    tuple(
        sanity_batch[
            "view_b"
        ].shape
    )
)


print(
    "Load time:",
    f"{time.time() - sanity_start:.2f} sec"
)


expected_shape = (
    BATCH_SIZE,
    NUM_FRAMES,
    3,
    224,
    224
)


if (
    tuple(
        sanity_batch[
            "view_a"
        ].shape
    )
    !=
    expected_shape
):

    raise RuntimeError(
        "Unexpected VideoMAE input shape.\n"
        f"Expected: {expected_shape}\n"
        f"Received: "
        f"{tuple(sanity_batch['view_a'].shape)}"
    )


print()

print(
    "[PASS] DataLoader + robust decoder "
    "working."
)


del sanity_batch
del sanity_iterator

gc.collect()


# ==================================================================================================
# 21. LOAD ORIGINAL PRETRAINED VIDEOMAE
# ==================================================================================================

print()

print("=" * 120)

print(
    "9. LOADING ORIGINAL PRETRAINED VIDEOMAE"
)

print("=" * 120)


print(
    "Model:",
    MODEL_NAME
)


print(
    "[LOCK] Old contaminated L20C "
    "will NOT be loaded."
)


model = (
    VideoMAEModel
    .from_pretrained(
        MODEL_NAME
    )
)


model = model.to(
    DEVICE
)


print()

print(
    "[PASS] Original pretrained "
    "VideoMAE loaded."
)


# ==================================================================================================
# 22. FREEZE MODEL EXCEPT FINAL FOUR BLOCKS
# ==================================================================================================

print()

print("=" * 120)

print(
    "10. CONFIGURING TRAINABLE PARAMETERS"
)

print("=" * 120)


for parameter in model.parameters():

    parameter.requires_grad = False


encoder_layers = (
    model.encoder.layer
)


num_encoder_blocks = len(
    encoder_layers
)


if (
    NUM_TRAINABLE_BLOCKS
    >
    num_encoder_blocks
):

    raise RuntimeError(
        "NUM_TRAINABLE_BLOCKS exceeds "
        "encoder depth."
    )


for block in encoder_layers[
    -NUM_TRAINABLE_BLOCKS:
]:

    for parameter in block.parameters():

        parameter.requires_grad = True


final_layernorm = getattr(
    model,
    "layernorm",
    None
)


if final_layernorm is not None:

    for parameter in (
        final_layernorm.parameters()
    ):

        parameter.requires_grad = True


total_parameters = sum(

    p.numel()

    for p in model.parameters()

)


trainable_parameter_count = sum(

    p.numel()

    for p in model.parameters()

    if p.requires_grad

)


trainable_parameters = [

    p

    for p in model.parameters()

    if p.requires_grad

]


print(
    "Encoder blocks       :",
    num_encoder_blocks
)

print(
    "Trainable final blocks:",
    NUM_TRAINABLE_BLOCKS
)

print(
    "Total parameters     :",
    f"{total_parameters:,}"
)

print(
    "Trainable parameters :",
    f"{trainable_parameter_count:,}"
)


# ==================================================================================================
# 23. OPTIMIZER
# ==================================================================================================

optimizer = torch.optim.AdamW(

    trainable_parameters,

    lr=LEARNING_RATE,

    weight_decay=WEIGHT_DECAY

)


# ==================================================================================================
# 24. AMP
# ==================================================================================================

AMP_ENABLED = (
    USE_AMP
    and
    DEVICE.type == "cuda"
)


try:

    scaler = torch.amp.GradScaler(

        "cuda",

        enabled=AMP_ENABLED

    )

except Exception:

    scaler = torch.cuda.amp.GradScaler(

        enabled=AMP_ENABLED

    )


print(
    "AMP enabled          :",
    AMP_ENABLED
)


# ==================================================================================================
# 25. REPRESENTATION FUNCTION
# ==================================================================================================

def extract_representation(
    pixel_values
):

    outputs = model(
        pixel_values=pixel_values
    )


    representation = (
        outputs
        .last_hidden_state
        .mean(dim=1)
    )


    representation = F.normalize(

        representation,

        p=2,

        dim=-1

    )


    return representation


# ==================================================================================================
# 26. TEMPORAL CONSISTENCY LOSS
# ==================================================================================================

def consistency_loss(
    z1,
    z2
):

    cosine_similarity = (
        z1
        *
        z2
    ).sum(
        dim=-1
    )


    return (
        1.0
        -
        cosine_similarity
    ).mean()


# ==================================================================================================
# 27. SAVE CONFIG
# ==================================================================================================

config = {

    "stage":
        "L20C_CORRECTED_NOCACHE_ROBUST",

    "status":
        "TRAINING",

    "protocol":
        "L20C_CORRECTED_6734_NOCACHE_ROBUST",

    "seed":
        SEED,

    "base_model":
        MODEL_NAME,

    "initialization":
        "ORIGINAL_PRETRAINED_VIDEOMAE",

    "old_contaminated_l20c_used":
        False,

    "domain_population":
        6734,

    "validation_population_excluded":
        304,

    "locked_test_population_excluded":
        10,

    "epochs":
        EPOCHS,

    "num_frames":
        NUM_FRAMES,

    "decoder":
        "sequential_decode_then_uniform_sample",

    "frame_cache":
        False,

    "batch_size":
        BATCH_SIZE,

    "gradient_accumulation":
        GRAD_ACCUM_STEPS,

    "effective_batch_size":
        EFFECTIVE_BATCH_SIZE,

    "learning_rate":
        LEARNING_RATE,

    "weight_decay":
        WEIGHT_DECAY,

    "trainable_final_blocks":
        NUM_TRAINABLE_BLOCKS,

    "dataloader_workers":
        NUM_WORKERS,

    "prefetch_factor":
        PREFETCH_FACTOR,

    "persistent_workers":
        PERSISTENT_WORKERS,

    "amp":
        AMP_ENABLED,

    "objective":
        "temporal_consistency",

    "masked_reconstruction_pretraining":
        False,

    "validation_used":
        False,

    "locked_test_used":
        False,

    "gloss_labels_used":
        False,

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat()

}


save_json(
    config,
    CONFIG_PATH
)


# ==================================================================================================
# 28. RESUME SUPPORT
# ==================================================================================================

start_epoch = 1

best_loss = float("inf")

best_epoch = 0

history = []


if HISTORY_PATH.exists():

    try:

        history = (
            pd.read_csv(
                HISTORY_PATH
            )
            .to_dict(
                orient="records"
            )
        )

    except Exception:

        history = []


if RESUME_CHECKPOINT.exists():

    print()

    print("=" * 120)

    print(
        "RESUME CHECKPOINT FOUND"
    )

    print("=" * 120)


    checkpoint = torch.load(

        RESUME_CHECKPOINT,

        map_location=DEVICE,

        weights_only=False

    )


    if (
        checkpoint.get("protocol")
        !=
        "L20C_CORRECTED_6734_NOCACHE_ROBUST"
    ):

        raise RuntimeError(
            "Existing resume checkpoint belongs "
            "to another L20C protocol."
        )


    current_manifest_hash = sha256_file(
        ELIGIBLE_MANIFEST_PATH
    )


    if (
        checkpoint.get(
            "domain_manifest_sha256"
        )
        !=
        current_manifest_hash
    ):

        raise RuntimeError(
            "Resume checkpoint manifest "
            "does not match current domain manifest."
        )


    model.load_state_dict(

        checkpoint[
            "model_state_dict"
        ]

    )


    optimizer.load_state_dict(

        checkpoint[
            "optimizer_state_dict"
        ]

    )


    if (
        AMP_ENABLED
        and
        checkpoint.get(
            "scaler_state_dict"
        )
        is not None
    ):

        scaler.load_state_dict(

            checkpoint[
                "scaler_state_dict"
            ]

        )


    completed_epoch = int(
        checkpoint["epoch"]
    )


    start_epoch = (
        completed_epoch
        +
        1
    )


    best_loss = float(
        checkpoint.get(
            "best_loss",
            float("inf")
        )
    )


    best_epoch = int(
        checkpoint.get(
            "best_epoch",
            0
        )
    )


    print(
        "Completed epoch:",
        completed_epoch
    )

    print(
        "Resume epoch   :",
        start_epoch
    )

    print(
        "Best epoch     :",
        best_epoch
    )

    print(
        "Best loss      :",
        best_loss
    )


else:

    print()

    print(
        "[INFO] No corrected resume "
        "checkpoint found."
    )

    print(
        "[INFO] Starting from original "
        "pretrained VideoMAE."
    )


# ==================================================================================================
# 29. TRAINING
# ==================================================================================================

print()

print("=" * 120)

print(
    "11. STARTING 15-EPOCH L20C TRAINING"
)

print("=" * 120)


print(
    "Start epoch     :",
    start_epoch
)

print(
    "Final epoch     :",
    EPOCHS
)

print(
    "Videos/epoch    :",
    len(dataset)
)

print(
    "Batches/epoch   :",
    len(train_loader)
)


total_training_start = time.time()


if start_epoch <= EPOCHS:

    for epoch in range(
        start_epoch,
        EPOCHS + 1
    ):

        print()

        print("=" * 120)

        print(
            f"EPOCH {epoch:02d}/{EPOCHS}"
        )

        print("=" * 120)


        epoch_start = time.time()


        model.train()


        optimizer.zero_grad(
            set_to_none=True
        )


        running_loss = 0.0

        processed_batches = 0

        processed_samples = 0


        for step, batch in enumerate(
            train_loader,
            start=1
        ):

            # --------------------------------------------------------------------------------------
            # CPU -> CUDA
            # --------------------------------------------------------------------------------------

            view_a = batch[
                "view_a"
            ].to(

                DEVICE,

                non_blocking=True

            )


            view_b = batch[
                "view_b"
            ].to(

                DEVICE,

                non_blocking=True

            )


            # --------------------------------------------------------------------------------------
            # FORWARD
            # --------------------------------------------------------------------------------------

            if AMP_ENABLED:

                with torch.autocast(

                    device_type="cuda",

                    dtype=torch.float16

                ):

                    z1 = extract_representation(
                        view_a
                    )


                    z2 = extract_representation(
                        view_b
                    )


                    raw_loss = consistency_loss(
                        z1,
                        z2
                    )


                    scaled_loss = (
                        raw_loss
                        /
                        GRAD_ACCUM_STEPS
                    )


                scaler.scale(
                    scaled_loss
                ).backward()


            else:

                z1 = extract_representation(
                    view_a
                )


                z2 = extract_representation(
                    view_b
                )


                raw_loss = consistency_loss(
                    z1,
                    z2
                )


                scaled_loss = (
                    raw_loss
                    /
                    GRAD_ACCUM_STEPS
                )


                scaled_loss.backward()


            running_loss += float(
                raw_loss.detach().cpu()
            )


            processed_batches += 1


            processed_samples += int(
                view_a.shape[0]
            )


            # --------------------------------------------------------------------------------------
            # OPTIMIZER STEP
            # --------------------------------------------------------------------------------------

            should_step = (

                step % GRAD_ACCUM_STEPS == 0

                or

                step == len(train_loader)

            )


            if should_step:

                if AMP_ENABLED:

                    scaler.unscale_(
                        optimizer
                    )


                torch.nn.utils.clip_grad_norm_(

                    trainable_parameters,

                    MAX_GRAD_NORM

                )


                if AMP_ENABLED:

                    scaler.step(
                        optimizer
                    )

                    scaler.update()


                else:

                    optimizer.step()


                optimizer.zero_grad(
                    set_to_none=True
                )


            # --------------------------------------------------------------------------------------
            # PROGRESS
            # --------------------------------------------------------------------------------------

            if (

                step == 1

                or

                step % LOG_EVERY == 0

                or

                step == len(train_loader)

            ):

                elapsed = (
                    time.time()
                    -
                    epoch_start
                )


                average_loss = (
                    running_loss
                    /
                    max(
                        processed_batches,
                        1
                    )
                )


                throughput = (
                    processed_samples
                    /
                    max(
                        elapsed,
                        1e-9
                    )
                )


                remaining_videos = (
                    len(dataset)
                    -
                    processed_samples
                )


                epoch_eta_seconds = (
                    remaining_videos
                    /
                    max(
                        throughput,
                        1e-9
                    )
                )


                allocated = (
                    torch.cuda.memory_allocated()
                    /
                    1024**3
                )


                reserved = (
                    torch.cuda.memory_reserved()
                    /
                    1024**3
                )


                print(

                    f"[E{epoch:02d}] "

                    f"{step:04d}/"
                    f"{len(train_loader):04d} | "

                    f"loss={average_loss:.6f} | "

                    f"videos="
                    f"{processed_samples}/"
                    f"{len(dataset)} | "

                    f"{throughput:.2f} vid/s | "

                    f"epoch ETA="
                    f"{epoch_eta_seconds/60:.1f} min | "

                    f"GPU="
                    f"{allocated:.2f}/"
                    f"{reserved:.2f} GB"

                )


            del (
                view_a,
                view_b,
                z1,
                z2,
                raw_loss,
                scaled_loss
            )


        # ==========================================================================================
        # END EPOCH
        # ==========================================================================================

        epoch_seconds = (
            time.time()
            -
            epoch_start
        )


        epoch_loss = (
            running_loss
            /
            max(
                processed_batches,
                1
            )
        )


        is_best = (
            epoch_loss
            <
            best_loss
        )


        if is_best:

            best_loss = float(
                epoch_loss
            )

            best_epoch = int(
                epoch
            )


        # ==========================================================================================
        # SAVE RESUME CHECKPOINT
        # ==========================================================================================

        resume_payload = {

            "protocol":
                "L20C_CORRECTED_6734_NOCACHE_ROBUST",

            "epoch":
                int(epoch),

            "model_state_dict":
                model.state_dict(),

            "optimizer_state_dict":
                optimizer.state_dict(),

            "scaler_state_dict":
                (
                    scaler.state_dict()
                    if AMP_ENABLED
                    else None
                ),

            "best_epoch":
                int(best_epoch),

            "best_loss":
                float(best_loss),

            "domain_manifest_sha256":
                sha256_file(
                    ELIGIBLE_MANIFEST_PATH
                ),

            "base_model":
                MODEL_NAME,

            "domain_population":
                6734,

            "validation_population":
                304,

            "locked_test_population":
                10

        }


        torch.save(
            resume_payload,
            RESUME_CHECKPOINT
        )


        # ==========================================================================================
        # BEST CHECKPOINT
        # ==========================================================================================

        if is_best:

            torch.save(

                {

                    "protocol":
                        "L20C_CORRECTED_6734_NOCACHE_ROBUST",

                    "epoch":
                        int(epoch),

                    "training_loss":
                        float(epoch_loss),

                    "model_state_dict":
                        model.state_dict(),

                    "base_model":
                        MODEL_NAME,

                    "domain_population":
                        6734,

                    "validation_used":
                        False,

                    "locked_test_used":
                        False,

                    "gloss_labels_used":
                        False

                },

                BEST_CHECKPOINT

            )


        # ==========================================================================================
        # HISTORY
        # ==========================================================================================

        epoch_record = {

            "epoch":
                int(epoch),

            "loss":
                float(epoch_loss),

            "seconds":
                float(epoch_seconds),

            "minutes":
                float(
                    epoch_seconds / 60
                ),

            "videos_per_second":
                float(
                    processed_samples
                    /
                    max(
                        epoch_seconds,
                        1e-9
                    )
                ),

            "best_epoch":
                int(best_epoch),

            "best_loss":
                float(best_loss),

            "is_best":
                bool(is_best)

        }


        history = [

            item

            for item in history

            if int(
                item.get(
                    "epoch",
                    -1
                )
            ) != epoch

        ]


        history.append(
            epoch_record
        )


        history = sorted(

            history,

            key=lambda x: int(
                x["epoch"]
            )

        )


        pd.DataFrame(
            history
        ).to_csv(

            HISTORY_PATH,

            index=False

        )


        # ==========================================================================================
        # ETA
        # ==========================================================================================

        completed_times = [

            float(item["seconds"])

            for item in history

            if "seconds" in item

        ]


        average_epoch_seconds = (

            sum(completed_times)

            /
            len(completed_times)

        )


        remaining_epochs = (
            EPOCHS
            -
            epoch
        )


        estimated_remaining_hours = (

            remaining_epochs
            *
            average_epoch_seconds
            /
            3600

        )


        print()

        print(
            f"[EPOCH {epoch:02d} COMPLETE]"
        )


        print(
            f"Loss               : "
            f"{epoch_loss:.8f}"
        )


        print(
            f"Epoch time         : "
            f"{epoch_seconds/60:.2f} min"
        )


        print(
            f"Throughput         : "
            f"{processed_samples/max(epoch_seconds,1e-9):.2f} "
            f"videos/sec"
        )


        print(
            f"Best epoch         : "
            f"{best_epoch}"
        )


        print(
            f"Best loss          : "
            f"{best_loss:.8f}"
        )


        print(
            f"Remaining epochs   : "
            f"{remaining_epochs}"
        )


        print(
            f"Estimated remaining: "
            f"{estimated_remaining_hours:.2f} hours"
        )


        print()

        print(
            "[SAVED RESUME]",
            RESUME_CHECKPOINT
        )


        if is_best:

            print(
                "[SAVED BEST]",
                BEST_CHECKPOINT
            )


        gc.collect()

        torch.cuda.empty_cache()


else:

    print(
        "[INFO] All 15 epochs already completed."
    )


# ==================================================================================================
# 30. LOAD BEST CHECKPOINT
# ==================================================================================================

print()

print("=" * 120)

print(
    "12. LOADING BEST CHECKPOINT"
)

print("=" * 120)


if not BEST_CHECKPOINT.exists():

    raise RuntimeError(
        "Best checkpoint does not exist."
    )


best_checkpoint = torch.load(

    BEST_CHECKPOINT,

    map_location=DEVICE,

    weights_only=False

)


if (
    best_checkpoint.get("protocol")
    !=
    "L20C_CORRECTED_6734_NOCACHE_ROBUST"
):

    raise RuntimeError(
        "Best checkpoint protocol mismatch."
    )


best_epoch = int(
    best_checkpoint["epoch"]
)


best_loss = float(
    best_checkpoint[
        "training_loss"
    ]
)


model.load_state_dict(

    best_checkpoint[
        "model_state_dict"
    ]

)


print(
    "Best epoch:",
    best_epoch
)

print(
    "Best loss :",
    best_loss
)


# ==================================================================================================
# 31. EXPORT FINAL BACKBONE
# ==================================================================================================

print()

print("=" * 120)

print(
    "13. EXPORTING FINAL ADAPTED BACKBONE"
)

print("=" * 120)


torch.save(

    {

        "protocol":
            "L20C_CORRECTED_6734_NOCACHE_ROBUST",

        "stage":
            "L20C_CORRECTED",

        "base_model":
            MODEL_NAME,

        "initialization":
            "ORIGINAL_PRETRAINED_VIDEOMAE",

        "old_contaminated_l20c_used":
            False,

        "best_epoch":
            best_epoch,

        "best_training_loss":
            best_loss,

        "domain_population":
            6734,

        "validation_population_excluded":
            304,

        "locked_test_population_excluded":
            10,

        "validation_used":
            False,

        "locked_test_used":
            False,

        "gloss_labels_used":
            False,

        "model_state_dict":
            model.state_dict()

    },

    FINAL_BACKBONE

)


print(
    "[SAVED]",
    FINAL_BACKBONE
)


# ==================================================================================================
# 32. SUMMARY
# ==================================================================================================

summary = {

    "stage":
        "L20C_CORRECTED_NOCACHE_ROBUST",

    "status":
        "COMPLETE_AND_FROZEN",

    "protocol":
        "L20C_CORRECTED_6734_NOCACHE_ROBUST",

    "epochs":
        EPOCHS,

    "best_epoch":
        best_epoch,

    "best_training_loss":
        best_loss,

    "base_model":
        MODEL_NAME,

    "domain_population":
        6734,

    "validation_population_excluded":
        304,

    "locked_test_population_excluded":
        10,

    "domain_validation_overlap":
        0,

    "domain_locked_overlap":
        0,

    "validation_used":
        False,

    "locked_test_used":
        False,

    "gloss_labels_used":
        False,

    "old_contaminated_l20c_used":
        False,

    "frame_cache_used":
        False,

    "decoder":
        "sequential_decode_then_uniform_sample",

    "batch_size":
        BATCH_SIZE,

    "gradient_accumulation":
        GRAD_ACCUM_STEPS,

    "effective_batch_size":
        EFFECTIVE_BATCH_SIZE,

    "dataloader_workers":
        NUM_WORKERS,

    "amp":
        AMP_ENABLED,

    "best_checkpoint":
        str(BEST_CHECKPOINT),

    "final_backbone":
        str(FINAL_BACKBONE),

    "completed_utc":
        datetime.now(
            timezone.utc
        ).isoformat()

}


save_json(
    summary,
    SUMMARY_PATH
)


# ==================================================================================================
# 33. HASH ARTIFACTS
# ==================================================================================================

print()

print("=" * 120)

print(
    "14. HASHING FINAL ARTIFACTS"
)

print("=" * 120)


hash_targets = {

    "eligible_manifest":
        ELIGIBLE_MANIFEST_PATH,

    "history":
        HISTORY_PATH,

    "config":
        CONFIG_PATH,

    "best_checkpoint":
        BEST_CHECKPOINT,

    "final_backbone":
        FINAL_BACKBONE,

    "summary":
        SUMMARY_PATH

}


hashes = {}


for name, path in hash_targets.items():

    if not path.exists():

        raise RuntimeError(
            f"Missing artifact: {path}"
        )


    digest = sha256_file(
        path
    )


    hashes[name] = {

        "path":
            str(path),

        "sha256":
            digest

    }


    print(
        f"{name:<25}: "
        f"{digest}"
    )


save_json(
    hashes,
    HASH_PATH
)


# ==================================================================================================
# 34. FINAL INTEGRITY CHECK
# ==================================================================================================

print()

print("=" * 120)

print(
    "15. FINAL SCIENTIFIC INTEGRITY CHECK"
)

print("=" * 120)


final_checks = {

    "domain_6734":
        len(domain_uids) == 6734,

    "validation_304":
        len(validation_uids) == 304,

    "locked_test_10":
        len(locked_uids) == 10,

    "domain_validation_overlap_zero":
        len(
            domain_uids
            &
            validation_uids
        ) == 0,

    "domain_locked_overlap_zero":
        len(
            domain_uids
            &
            locked_uids
        ) == 0,

    "best_checkpoint_exists":
        BEST_CHECKPOINT.exists(),

    "final_backbone_exists":
        FINAL_BACKBONE.exists(),

    "old_l20c_not_used":
        True,

    "validation_not_used":
        True,

    "locked_test_not_used":
        True

}


for name, passed in final_checks.items():

    print(
        f"{name:<50}: "
        f"{'PASS' if passed else 'FAIL'}"
    )


if not all(
    final_checks.values()
):

    raise RuntimeError(
        "Final L20C integrity check failed."
    )


# ==================================================================================================
# 35. MASTER FREEZE
# ==================================================================================================

master_freeze = {

    "stage":
        "L20C_CORRECTED_NOCACHE_ROBUST",

    "status":
        "COMPLETE_AND_FROZEN",

    "protocol":
        "L20C_CORRECTED_6734_NOCACHE_ROBUST",

    "source_l20b":
        str(L20B_FREEZE),

    "source_l20b_sha256":
        sha256_file(
            L20B_FREEZE
        ),

    "base_model":
        MODEL_NAME,

    "initialization":
        "ORIGINAL_PRETRAINED_VIDEOMAE",

    "old_contaminated_l20c_used":
        False,

    "domain_population":
        6734,

    "validation_population_excluded":
        304,

    "locked_test_population_excluded":
        10,

    "domain_validation_overlap":
        0,

    "domain_locked_overlap":
        0,

    "epochs":
        EPOCHS,

    "best_epoch":
        best_epoch,

    "best_training_loss":
        best_loss,

    "decoder":
        "sequential_decode_then_uniform_sample",

    "frame_cache_used":
        False,

    "validation_used":
        False,

    "locked_test_used":
        False,

    "gloss_labels_used":
        False,

    "best_checkpoint":
        str(BEST_CHECKPOINT),

    "final_backbone":
        str(FINAL_BACKBONE),

    "hash_manifest":
        str(HASH_PATH),

    "completed_utc":
        datetime.now(
            timezone.utc
        ).isoformat()

}


save_json(
    master_freeze,
    MASTER_FREEZE_PATH
)


MASTER_SHA = sha256_file(
    MASTER_FREEZE_PATH
)


# ==================================================================================================
# 36. FINAL REPORT
# ==================================================================================================

disk_total, disk_used, disk_free = (
    shutil.disk_usage("/home")
)


print()

print("=" * 120)

print(
    "L20C CORRECTED — COMPLETE AND FROZEN"
)

print("=" * 120)


print()

print(
    "Domain videos                  : 6734"
)

print(
    "Validation excluded            : 304"
)

print(
    "Locked test excluded           : 10"
)


print()

print(
    "Epochs                         :",
    EPOCHS
)

print(
    "Best epoch                     :",
    best_epoch
)

print(
    "Best training loss             :",
    f"{best_loss:.9f}"
)


print()

print(
    "Original VideoMAE initialization: YES"
)

print(
    "Old contaminated L20C used      : NO"
)

print(
    "Frame cache                     : NO"
)

print(
    "Robust sequential decoder       : YES"
)

print(
    "Validation used                 : NO"
)

print(
    "Locked test used                : NO"
)

print(
    "Gloss labels used               : NO"
)


print()

print(
    "Disk free                       :",
    f"{gb(disk_free):.2f} GB"
)


print()

print(
    "Final backbone:"
)

print(
    FINAL_BACKBONE
)


print()

print(
    "Master freeze:"
)

print(
    MASTER_FREEZE_PATH
)


print()

print(
    "Master SHA256:"
)

print(
    MASTER_SHA
)


print()

print(
    "[PASS] Corrected L20C completed."
)

print(
    "[PASS] 6734-domain protocol preserved."
)

print(
    "[PASS] Validation remained isolated."
)

print(
    "[PASS] L19 locked test remained isolated."
)

print(
    "[PASS] Ready for corrected L20D."
)

print("=" * 120)

PARASURG / CISLR — L20C CORRECTED
ROBUST NO-CACHE 15-EPOCH DOMAIN ADAPTATION

PyTorch              : 2.14.0+cu130
OpenCV               : 5.0.0
CUDA available       : True
Device               : cuda
GPU                  : NVIDIA GeForce RTX 4050 Laptop GPU
GPU memory           : 5.64 GB

Disk free            : 18.56 GB
CPU count            : 16
DataLoader workers   : 4
Batch size           : 1
Gradient accumulation: 8
Effective batch      : 8
Epochs               : 15
Frame cache          : NONE

1. VERIFYING CORRECTED L20B
[FOUND] /home/dipshikha/Music/cao/CISLR_RESEARCH/CISLR_LANDMARK_MODEL/audit/l20b_corrected_strict_protocol_freeze/manifests/l20b_corrected_domain_pretraining_manifest.csv
[FOUND] /home/dipshikha/Music/cao/CISLR_RESEARCH/CISLR_LANDMARK_MODEL/audit/l20b_corrected_strict_protocol_freeze/manifests/l20b_corrected_validation_manifest.csv
[FOUND] /home/dipshikha/Music/cao/CISLR_RESEARCH/CISLR_LANDMARK_MODEL/audit/l20b_corrected_strict_protocol_freeze/manifests/l20b_permane

In [5]:
# ==================================================================================================
# CISLR — L20D-B0 CORRECTED BASELINE
# ORIGINAL PRETRAINED VIDEOMAE -> 215-CLASS SUPERVISED CLASSIFICATION
# ==================================================================================================
#
# PURPOSE
# -------
# Establish the clean supervised baseline BEFORE trusting L20C adaptation.
#
# INITIALIZATION
# --------------
# MCG-NJU/videomae-base-finetuned-kinetics
#
# THIS CELL DOES NOT LOAD L20C.
#
# STRICT PROTOCOL
# ---------------
# Train      : corrected L20B 710 videos
# Validation : corrected L20B 304 videos
# Classes    : 215
# Locked test: 10 videos -- NEVER decoded/evaluated here
#
# TRAINING
# --------
# Stage 1 : classifier-head warm-up
# Stage 2 : fine-tune final 4 VideoMAE encoder blocks + classifier
#
# MODEL SELECTION
# ---------------
# Primary   : validation Top-1
# Secondary : validation Top-5
# Tertiary  : validation loss
#
# IMPORTANT
# ---------
# - Balanced WeightedRandomSampler for training.
# - No horizontal flip.
# - No temporal reversal.
# - Unique unaugmented train accuracy reported at end.
# - No L20C checkpoint is loaded anywhere.
# - No locked-test video is decoded.
# ==================================================================================================

import os
import gc
import cv2
import json
import time
import random
import hashlib
import shutil
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import (
    Dataset,
    DataLoader,
    WeightedRandomSampler,
)

from transformers import (
    VideoMAEModel,
    VideoMAEImageProcessor,
)


# ==================================================================================================
# 0. CONFIGURATION
# ==================================================================================================

SEED = 42

PROJECT_ROOT = Path(
    "/home/dipshikha/Music/cao/CISLR_RESEARCH"
)

LANDMARK_ROOT = (
    PROJECT_ROOT
    / "CISLR_LANDMARK_MODEL"
)

VIDEO_ROOT = Path(
    "/home/dipshikha/Music/cao/CISLR_v1.5-a_videos"
)

L20B_ROOT = (
    LANDMARK_ROOT
    / "audit"
    / "l20b_corrected_strict_protocol_freeze"
)

OUTPUT_ROOT = (
    LANDMARK_ROOT
    / "audit"
    / "l20d_b0_original_videomae_215class"
)

MANIFEST_DIR = OUTPUT_ROOT / "manifests"
MODEL_DIR = OUTPUT_ROOT / "models"
REPORT_DIR = OUTPUT_ROOT / "reports"

for d in [
    OUTPUT_ROOT,
    MANIFEST_DIR,
    MODEL_DIR,
    REPORT_DIR,
]:
    d.mkdir(
        parents=True,
        exist_ok=True,
    )


# --------------------------------------------------------------------------------------------------
# AUTHORITATIVE CORRECTED L20B FILES
# --------------------------------------------------------------------------------------------------

TRAIN_MANIFEST = (
    L20B_ROOT
    / "manifests"
    / "l20b_corrected_train_manifest.csv"
)

VAL_MANIFEST = (
    L20B_ROOT
    / "manifests"
    / "l20b_corrected_validation_manifest.csv"
)

LOCKED_TEST_MANIFEST = (
    L20B_ROOT
    / "manifests"
    / "l20b_permanent_l19_locked_test_manifest.csv"
)

CLASS_MAPPING_FILE = (
    L20B_ROOT
    / "manifests"
    / "l20b_corrected_class_mapping.csv"
)

L20B_FREEZE = (
    L20B_ROOT
    / "L20B_CORRECTED_MASTER_FREEZE.json"
)


# --------------------------------------------------------------------------------------------------
# MODEL / TRAINING
# --------------------------------------------------------------------------------------------------

MODEL_NAME = "MCG-NJU/videomae-base-finetuned-kinetics"

NUM_CLASSES = 215
NUM_FRAMES = 16

HEAD_EPOCHS = 8
FINETUNE_EPOCHS = 35

HEAD_LR = 3e-4
FINETUNE_LR = 2e-5

WEIGHT_DECAY = 1e-4

BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 8

NUM_WORKERS = 4

TRAINABLE_FINAL_BLOCKS = 4

EARLY_STOPPING_PATIENCE = 10

GRAD_CLIP = 1.0

LOG_EVERY = 100

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


# ==================================================================================================
# 1. REPRODUCIBILITY
# ==================================================================================================

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():

    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

torch.backends.cudnn.benchmark = True

cv2.setNumThreads(0)

os.environ["TOKENIZERS_PARALLELISM"] = "false"


# ==================================================================================================
# 2. HELPERS
# ==================================================================================================

def section(title):

    print("\n" + "=" * 120)
    print(title)
    print("=" * 120)


def normalize_uid(x):

    x = str(x).strip()

    if x.endswith(".mp4"):
        x = x[:-4]

    return x


def find_column(
    df,
    candidates,
):

    mapping = {
        str(c).lower(): c
        for c in df.columns
    }

    for candidate in candidates:

        key = candidate.lower()

        if key in mapping:
            return mapping[key]

    return None


def sha256_file(
    path,
    chunk_size=1024 * 1024,
):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        while True:

            chunk = f.read(
                chunk_size
            )

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


def disk_free_gb(path):

    return (
        shutil.disk_usage(path).free
        / 1024**3
    )


# ==================================================================================================
# 3. START
# ==================================================================================================

section(
    "CISLR — L20D-B0\n"
    "ORIGINAL VIDEOMAE 215-CLASS SUPERVISED BASELINE"
)

print(
    f"PyTorch                 : "
    f"{torch.__version__}"
)

print(
    f"OpenCV                  : "
    f"{cv2.__version__}"
)

print(
    f"CUDA available          : "
    f"{torch.cuda.is_available()}"
)

print(
    f"Device                  : "
    f"{DEVICE}"
)

if torch.cuda.is_available():

    print(
        f"GPU                     : "
        f"{torch.cuda.get_device_name(0)}"
    )

    print(
        f"GPU memory              : "
        f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB"
    )

print(
    f"Disk free               : "
    f"{disk_free_gb(PROJECT_ROOT):.2f} GB"
)

print(
    f"Classes                 : "
    f"{NUM_CLASSES}"
)

print(
    f"Frames/video            : "
    f"{NUM_FRAMES}"
)

print(
    f"Physical batch          : "
    f"{BATCH_SIZE}"
)

print(
    f"Gradient accumulation   : "
    f"{GRAD_ACCUM_STEPS}"
)

print(
    f"Effective batch         : "
    f"{BATCH_SIZE * GRAD_ACCUM_STEPS}"
)

print(
    f"Head epochs             : "
    f"{HEAD_EPOCHS}"
)

print(
    f"Fine-tune epochs        : "
    f"{FINETUNE_EPOCHS}"
)

print(
    "Initialization          : ORIGINAL pretrained VideoMAE"
)

print(
    "L20C loaded             : NO"
)

print(
    "Locked test evaluation  : DISABLED"
)


# ==================================================================================================
# 4. VERIFY INPUT FILES
# ==================================================================================================

section(
    "4. VERIFYING CORRECTED L20B ARTIFACTS"
)

required_files = [
    TRAIN_MANIFEST,
    VAL_MANIFEST,
    LOCKED_TEST_MANIFEST,
    CLASS_MAPPING_FILE,
    L20B_FREEZE,
]

for path in required_files:

    if not path.exists():

        raise FileNotFoundError(
            f"\nMissing required file:\n{path}"
        )

    print(
        f"[FOUND] {path}"
    )

print(
    "\n[PASS] Corrected L20B artifacts found."
)


# ==================================================================================================
# 5. LOAD MANIFESTS
# ==================================================================================================

section(
    "5. LOADING CORRECTED L20B PROTOCOL"
)

train_df = pd.read_csv(
    TRAIN_MANIFEST
)

val_df = pd.read_csv(
    VAL_MANIFEST
)

# Read only for UID-isolation checking.
# No locked-test VIDEO is opened.
locked_df = pd.read_csv(
    LOCKED_TEST_MANIFEST
)

class_map_df = pd.read_csv(
    CLASS_MAPPING_FILE
)


uid_candidates = [
    "uid",
    "video_uid",
    "video_id",
    "id",
    "name",
]

gloss_candidates = [
    "gloss",
    "label",
    "word",
    "class_name",
]


train_uid_col = find_column(
    train_df,
    uid_candidates,
)

val_uid_col = find_column(
    val_df,
    uid_candidates,
)

locked_uid_col = find_column(
    locked_df,
    uid_candidates,
)

train_gloss_col = find_column(
    train_df,
    gloss_candidates,
)

val_gloss_col = find_column(
    val_df,
    gloss_candidates,
)


if train_uid_col is None:
    raise RuntimeError(
        f"Train UID column not found. "
        f"Columns={train_df.columns.tolist()}"
    )

if val_uid_col is None:
    raise RuntimeError(
        f"Validation UID column not found. "
        f"Columns={val_df.columns.tolist()}"
    )

if locked_uid_col is None:
    raise RuntimeError(
        f"Locked UID column not found. "
        f"Columns={locked_df.columns.tolist()}"
    )

if train_gloss_col is None:
    raise RuntimeError(
        f"Train gloss column not found. "
        f"Columns={train_df.columns.tolist()}"
    )

if val_gloss_col is None:
    raise RuntimeError(
        f"Validation gloss column not found. "
        f"Columns={val_df.columns.tolist()}"
    )


train_df["uid"] = (
    train_df[train_uid_col]
    .astype(str)
    .map(normalize_uid)
)

val_df["uid"] = (
    val_df[val_uid_col]
    .astype(str)
    .map(normalize_uid)
)

locked_df["uid"] = (
    locked_df[locked_uid_col]
    .astype(str)
    .map(normalize_uid)
)


train_df["gloss"] = (
    train_df[train_gloss_col]
    .astype(str)
    .str.strip()
)

val_df["gloss"] = (
    val_df[val_gloss_col]
    .astype(str)
    .str.strip()
)


# ==================================================================================================
# 6. AUTHORITATIVE CLASS MAPPING
# ==================================================================================================

section(
    "6. VERIFYING 215-CLASS MAPPING"
)

print(
    "Mapping columns:",
    class_map_df.columns.tolist(),
)


mapping_gloss_col = find_column(
    class_map_df,
    [
        "gloss",
        "label",
        "word",
        "class_name",
    ],
)

mapping_id_col = find_column(
    class_map_df,
    [
        "class_id",
        "label_id",
        "class_index",
        "index",
        "target",
    ],
)


if (
    mapping_gloss_col is None
    or
    mapping_id_col is None
):

    raise RuntimeError(
        "Could not identify class mapping columns."
    )


mapping = class_map_df[
    [
        mapping_gloss_col,
        mapping_id_col,
    ]
].copy()

mapping.columns = [
    "gloss",
    "class_id",
]

mapping["gloss"] = (
    mapping["gloss"]
    .astype(str)
    .str.strip()
)

mapping["class_id"] = (
    mapping["class_id"]
    .astype(int)
)

mapping = (
    mapping
    .drop_duplicates("gloss")
    .sort_values("class_id")
    .reset_index(drop=True)
)


if len(mapping) != NUM_CLASSES:

    raise RuntimeError(
        f"Expected 215 classes, "
        f"found {len(mapping)}."
    )


expected_ids = set(
    range(NUM_CLASSES)
)

actual_ids = set(
    mapping["class_id"]
)

if expected_ids != actual_ids:

    raise RuntimeError(
        "Class IDs are not exactly 0..214."
    )


gloss_to_id = dict(
    zip(
        mapping["gloss"],
        mapping["class_id"],
    )
)

id_to_gloss = {
    idx: gloss
    for gloss, idx
    in gloss_to_id.items()
}


train_df["label"] = (
    train_df["gloss"]
    .map(gloss_to_id)
)

val_df["label"] = (
    val_df["gloss"]
    .map(gloss_to_id)
)


if train_df["label"].isna().any():

    raise RuntimeError(
        "Unmapped training gloss detected."
    )

if val_df["label"].isna().any():

    raise RuntimeError(
        "Unmapped validation gloss detected."
    )


train_df["label"] = (
    train_df["label"]
    .astype(int)
)

val_df["label"] = (
    val_df["label"]
    .astype(int)
)


print(
    f"Classes mapped: "
    f"{len(gloss_to_id)}"
)

print(
    "ID range      : 0..214"
)

print(
    "\n[PASS] Mapping ready."
)


# ==================================================================================================
# 7. STRICT SPLIT CHECK
# ==================================================================================================

section(
    "7. STRICT PROTOCOL INTEGRITY CHECK"
)

train_uids = set(
    train_df["uid"]
)

val_uids = set(
    val_df["uid"]
)

locked_uids = set(
    locked_df["uid"]
)


checks = {
    "train_rows_710":
        len(train_df) == 710,

    "train_unique_710":
        len(train_uids) == 710,

    "validation_rows_304":
        len(val_df) == 304,

    "validation_unique_304":
        len(val_uids) == 304,

    "locked_rows_10":
        len(locked_df) == 10,

    "locked_unique_10":
        len(locked_uids) == 10,

    "train_validation_overlap_zero":
        len(train_uids & val_uids) == 0,

    "train_locked_overlap_zero":
        len(train_uids & locked_uids) == 0,

    "validation_locked_overlap_zero":
        len(val_uids & locked_uids) == 0,

    "train_classes_215":
        train_df["label"].nunique()
        == NUM_CLASSES,

    "validation_classes_215":
        val_df["label"].nunique()
        == NUM_CLASSES,
}


for name, passed in checks.items():

    print(
        f"{name:50s}: "
        f"{'PASS' if passed else 'FAIL'}"
    )


if not all(checks.values()):

    raise RuntimeError(
        "Protocol integrity failure."
    )


print(
    "\n[PASS] 710 / 304 / 10 split preserved."
)


# ==================================================================================================
# 8. CLASS DISTRIBUTION
# ==================================================================================================

section(
    "8. CLASS DISTRIBUTION"
)

train_counts = (
    train_df
    .groupby("label")
    .size()
)

val_counts = (
    val_df
    .groupby("label")
    .size()
)


print("TRAIN")
print(
    f"Videos     : {len(train_df)}"
)
print(
    f"Classes    : {len(train_counts)}"
)
print(
    f"Min/class  : {train_counts.min()}"
)
print(
    f"Max/class  : {train_counts.max()}"
)
print(
    f"Mean/class : {train_counts.mean():.3f}"
)


print("\nVALIDATION")
print(
    f"Videos     : {len(val_df)}"
)
print(
    f"Classes    : {len(val_counts)}"
)
print(
    f"Min/class  : {val_counts.min()}"
)
print(
    f"Max/class  : {val_counts.max()}"
)
print(
    f"Mean/class : {val_counts.mean():.3f}"
)


# ==================================================================================================
# 9. VIDEO INDEX
# ==================================================================================================

section(
    "9. INDEXING CISLR VIDEOS"
)

all_video_paths = list(
    VIDEO_ROOT.rglob("*.mp4")
)

video_index = {}

for path in all_video_paths:

    uid = normalize_uid(
        path.stem
    )

    if uid not in video_index:

        video_index[uid] = path


print(
    f"MP4 files found: "
    f"{len(all_video_paths)}"
)


def resolve_paths(df):

    df = df.copy()

    paths = []
    missing = []

    for uid in df["uid"]:

        p = video_index.get(uid)

        if p is None:

            paths.append(None)
            missing.append(uid)

        else:

            paths.append(
                str(p)
            )

    df["video_path"] = paths

    return df, missing


train_df, train_missing = resolve_paths(
    train_df
)

val_df, val_missing = resolve_paths(
    val_df
)


print(
    f"Train resolved: "
    f"{len(train_df) - len(train_missing)}/"
    f"{len(train_df)}"
)

print(
    f"Val resolved  : "
    f"{len(val_df) - len(val_missing)}/"
    f"{len(val_df)}"
)


if train_missing or val_missing:

    raise RuntimeError(
        f"Missing videos.\n"
        f"Train: {train_missing[:10]}\n"
        f"Val: {val_missing[:10]}"
    )


# Save exact baseline manifests.

B0_TRAIN_MANIFEST = (
    MANIFEST_DIR
    / "l20d_b0_train_710.csv"
)

B0_VAL_MANIFEST = (
    MANIFEST_DIR
    / "l20d_b0_validation_304.csv"
)

train_df.to_csv(
    B0_TRAIN_MANIFEST,
    index=False,
)

val_df.to_csv(
    B0_VAL_MANIFEST,
    index=False,
)


# ==================================================================================================
# 10. ROBUST VIDEO DECODER
# ==================================================================================================

section(
    "10. ROBUST VIDEO DECODER"
)


def decode_uniform_frames(
    video_path,
    num_frames=NUM_FRAMES,
):

    cap = cv2.VideoCapture(
        str(video_path)
    )

    if not cap.isOpened():

        raise RuntimeError(
            f"Cannot open:\n{video_path}"
        )


    frames = []

    try:

        while True:

            ok, frame = cap.read()

            if not ok:
                break

            if frame is None:
                continue

            frame = cv2.cvtColor(
                frame,
                cv2.COLOR_BGR2RGB,
            )

            frames.append(
                frame
            )

    finally:

        cap.release()


    if len(frames) == 0:

        raise RuntimeError(
            f"No frames decoded:\n{video_path}"
        )


    indices = np.linspace(
        0,
        len(frames) - 1,
        num_frames,
    ).round().astype(
        np.int64
    )


    return [
        frames[int(i)]
        for i in indices
    ]


print(
    "[PASS] Decoder ready."
)


# ==================================================================================================
# 11. PROCESSOR
# ==================================================================================================

section(
    "11. LOADING VIDEOMAE PROCESSOR"
)

processor = (
    VideoMAEImageProcessor
    .from_pretrained(
        MODEL_NAME
    )
)

print(
    f"[PASS] {MODEL_NAME}"
)


# ==================================================================================================
# 12. AUGMENTATION
# ==================================================================================================

def augment_train_frames(
    frames
):

    # Conservative photometric augmentation.
    #
    # NO horizontal flip.
    # NO temporal reversal.

    brightness = random.uniform(
        0.90,
        1.10,
    )

    contrast = random.uniform(
        0.90,
        1.10,
    )


    result = []

    for frame in frames:

        x = frame.astype(
            np.float32
        )

        mean = x.mean(
            axis=(0, 1),
            keepdims=True,
        )

        x = (
            (x - mean)
            * contrast
            + mean
        )

        x *= brightness

        x = np.clip(
            x,
            0,
            255,
        ).astype(
            np.uint8
        )

        result.append(x)

    return result


# ==================================================================================================
# 13. DATASET
# ==================================================================================================

class CISLRDataset(Dataset):

    def __init__(
        self,
        df,
        processor,
        train=False,
    ):

        self.df = (
            df
            .reset_index(drop=True)
            .copy()
        )

        self.processor = processor
        self.train = train


    def __len__(self):

        return len(self.df)


    def __getitem__(
        self,
        idx,
    ):

        row = self.df.iloc[idx]

        frames = decode_uniform_frames(
            row["video_path"]
        )


        if self.train:

            frames = (
                augment_train_frames(
                    frames
                )
            )


        encoded = self.processor(
            frames,
            return_tensors="pt",
        )


        pixel_values = (
            encoded[
                "pixel_values"
            ]
            .squeeze(0)
        )


        return {
            "pixel_values":
                pixel_values,

            "label":
                torch.tensor(
                    int(row["label"]),
                    dtype=torch.long,
                ),

            "uid":
                row["uid"],
        }


train_dataset = CISLRDataset(
    train_df,
    processor,
    train=True,
)

val_dataset = CISLRDataset(
    val_df,
    processor,
    train=False,
)


# ==================================================================================================
# 14. BALANCED SAMPLER
# ==================================================================================================

section(
    "14. BALANCED TRAIN SAMPLER"
)

class_counts = Counter(
    train_df["label"].tolist()
)

weights = torch.DoubleTensor(
    [
        1.0
        / class_counts[int(label)]

        for label
        in train_df["label"]
    ]
)

sampler_generator = (
    torch.Generator()
)

sampler_generator.manual_seed(
    SEED
)


train_sampler = WeightedRandomSampler(
    weights=weights,
    num_samples=len(train_df),
    replacement=True,
    generator=sampler_generator,
)


print(
    f"Samples/epoch : "
    f"{len(train_df)}"
)

print(
    f"Classes       : "
    f"{len(class_counts)}"
)

print(
    "[PASS] Balanced sampler ready."
)


# ==================================================================================================
# 15. DATALOADERS
# ==================================================================================================

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=train_sampler,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=(
        NUM_WORKERS > 0
    ),
    prefetch_factor=(
        2
        if NUM_WORKERS > 0
        else None
    ),
)


val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=(
        NUM_WORKERS > 0
    ),
    prefetch_factor=(
        2
        if NUM_WORKERS > 0
        else None
    ),
)


section(
    "15. DATALOADER SANITY CHECK"
)

batch = next(
    iter(train_loader)
)

print(
    "Pixel values:",
    tuple(
        batch["pixel_values"].shape
    ),
)

print(
    "Label       :",
    batch["label"].tolist(),
)

print(
    "UID         :",
    batch["uid"][0],
)

del batch

gc.collect()


# ==================================================================================================
# 16. LOAD ORIGINAL PRETRAINED VIDEOMAE
# ==================================================================================================

section(
    "16. LOADING ORIGINAL PRETRAINED VIDEOMAE"
)

backbone = (
    VideoMAEModel
    .from_pretrained(
        MODEL_NAME
    )
)

print(
    "[PASS] Original pretrained VideoMAE loaded."
)

print(
    "[PASS] L20C checkpoint was NOT loaded."
)


# ==================================================================================================
# 17. REPRESENTATION DIAGNOSTIC — ORIGINAL MODEL
# ==================================================================================================

section(
    "17. ORIGINAL VIDEOMAE REPRESENTATION DIAGNOSTIC"
)

backbone = backbone.to(
    DEVICE
)

backbone.eval()

diagnostic_features = []

DIAGNOSTIC_SAMPLES = min(
    64,
    len(val_dataset),
)


with torch.no_grad():

    for idx in range(
        DIAGNOSTIC_SAMPLES
    ):

        sample = val_dataset[idx]

        x = (
            sample["pixel_values"]
            .unsqueeze(0)
            .to(
                DEVICE,
                non_blocking=True,
            )
        )


        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(
                DEVICE.type
                == "cuda"
            ),
        ):

            outputs = backbone(
                pixel_values=x
            )

            feature = (
                outputs
                .last_hidden_state
                .mean(dim=1)
            )


        diagnostic_features.append(
            feature
            .float()
            .cpu()
            .squeeze(0)
        )


diagnostic_features = (
    torch.stack(
        diagnostic_features
    )
)

normalized = F.normalize(
    diagnostic_features,
    dim=1,
)

cosine_matrix = (
    normalized
    @ normalized.T
)

n = cosine_matrix.shape[0]

off_diag_mask = ~torch.eye(
    n,
    dtype=torch.bool,
)

off_diag = cosine_matrix[
    off_diag_mask
]


mean_cosine = float(
    off_diag.mean()
)

std_cosine = float(
    off_diag.std()
)

feature_dimension_std = float(
    diagnostic_features
    .std(dim=0)
    .mean()
)

feature_norm = float(
    diagnostic_features
    .norm(dim=1)
    .mean()
)


print(
    f"Samples tested           : "
    f"{DIAGNOSTIC_SAMPLES}"
)

print(
    f"Mean pairwise cosine     : "
    f"{mean_cosine:.6f}"
)

print(
    f"Std pairwise cosine      : "
    f"{std_cosine:.6f}"
)

print(
    f"Mean feature-dimension SD: "
    f"{feature_dimension_std:.6f}"
)

print(
    f"Mean feature norm        : "
    f"{feature_norm:.6f}"
)


# --------------------------------------------------------------------------------------------------
# IMPORTANT:
# We flag extremely high cosine independently.
# --------------------------------------------------------------------------------------------------

if mean_cosine >= 0.999:

    diagnostic_status = (
        "SEVERE_SIMILARITY_WARNING"
    )

elif mean_cosine >= 0.99:

    diagnostic_status = (
        "HIGH_SIMILARITY_WARNING"
    )

else:

    diagnostic_status = (
        "NO_EXTREME_SIMILARITY"
    )


print(
    f"Diagnostic status        : "
    f"{diagnostic_status}"
)


# Comparison with already observed L20C result.

L20C_OBSERVED_COSINE = 0.999967

print(
    "\nREFERENCE FROM PREVIOUS L20C-INITIALIZED RUN"
)

print(
    f"L20C mean cosine         : "
    f"{L20C_OBSERVED_COSINE:.6f}"
)

print(
    f"Original mean cosine     : "
    f"{mean_cosine:.6f}"
)

print(
    f"Difference               : "
    f"{L20C_OBSERVED_COSINE - mean_cosine:.6f}"
)


diagnostic_report = {
    "model":
        MODEL_NAME,

    "initialization":
        "original_pretrained",

    "samples":
        DIAGNOSTIC_SAMPLES,

    "mean_pairwise_cosine":
        mean_cosine,

    "std_pairwise_cosine":
        std_cosine,

    "mean_feature_dimension_std":
        feature_dimension_std,

    "mean_feature_norm":
        feature_norm,

    "status":
        diagnostic_status,

    "previous_l20c_mean_cosine":
        L20C_OBSERVED_COSINE,
}


DIAGNOSTIC_REPORT = (
    REPORT_DIR
    / "l20d_b0_original_representation_diagnostic.json"
)

with open(
    DIAGNOSTIC_REPORT,
    "w",
) as f:

    json.dump(
        diagnostic_report,
        f,
        indent=2,
    )


del diagnostic_features
del normalized
del cosine_matrix
del off_diag

gc.collect()

if torch.cuda.is_available():

    torch.cuda.empty_cache()


# ==================================================================================================
# 18. CLASSIFIER
# ==================================================================================================
#
# IMPORTANT CHANGE FROM PREVIOUS L20D:
#
# Normalize pooled features before classification.
#
# Previous L20C features had norm ~1441.
# Feeding such large vectors directly into a random Linear head can produce
# extremely large logits and huge initial CE losses.
#
# F.normalize() stabilizes the classifier-head optimization while preserving
# angular information in the representation.
#
# ==================================================================================================

class VideoMAEClassifier(nn.Module):

    def __init__(
        self,
        backbone,
        num_classes,
    ):

        super().__init__()

        self.backbone = backbone

        hidden = (
            backbone.config.hidden_size
        )

        self.dropout = nn.Dropout(
            0.20
        )

        self.classifier = nn.Linear(
            hidden,
            num_classes,
        )


    def forward(
        self,
        pixel_values,
    ):

        outputs = self.backbone(
            pixel_values=pixel_values
        )

        features = (
            outputs
            .last_hidden_state
            .mean(dim=1)
        )

        normalized_features = (
            F.normalize(
                features,
                p=2,
                dim=1,
            )
        )

        logits = self.classifier(
            self.dropout(
                normalized_features
            )
        )

        return (
            logits,
            features,
        )


model = VideoMAEClassifier(
    backbone,
    NUM_CLASSES,
).to(
    DEVICE
)


# ==================================================================================================
# 19. MODEL SANITY TEST
# ==================================================================================================

section(
    "19. CLASSIFIER SANITY TEST"
)

sample = val_dataset[0]

x = (
    sample["pixel_values"]
    .unsqueeze(0)
    .to(DEVICE)
)


model.eval()

with torch.no_grad():

    logits, features = model(x)


print(
    f"Logits shape       : "
    f"{tuple(logits.shape)}"
)

print(
    f"Raw feature norm   : "
    f"{features.norm(dim=1).item():.4f}"
)

print(
    f"Logit min/max      : "
    f"{logits.min().item():.4f} / "
    f"{logits.max().item():.4f}"
)

initial_loss = F.cross_entropy(
    logits,
    torch.tensor(
        [int(sample["label"])],
        device=DEVICE,
    ),
)

print(
    f"Single-sample CE   : "
    f"{initial_loss.item():.4f}"
)

print(
    f"Random CE reference: "
    f"{np.log(NUM_CLASSES):.4f}"
)


del x
del logits
del features

gc.collect()

if torch.cuda.is_available():

    torch.cuda.empty_cache()


# ==================================================================================================
# 20. METRIC HELPERS
# ==================================================================================================

criterion = nn.CrossEntropyLoss()


def topk_correct(
    logits,
    labels,
    k,
):

    indices = logits.topk(
        min(
            k,
            logits.shape[1],
        ),
        dim=1,
    ).indices

    correct = (
        indices
        == labels.unsqueeze(1)
    ).any(
        dim=1
    )

    return int(
        correct.sum().item()
    )


# ==================================================================================================
# 21. PARAMETER FREEZING
# ==================================================================================================

def freeze_for_head(
    model
):

    for p in model.backbone.parameters():

        p.requires_grad = False

    for p in model.classifier.parameters():

        p.requires_grad = True


def unfreeze_final_blocks(
    model,
    n_blocks=4,
):

    for p in model.backbone.parameters():

        p.requires_grad = False


    layers = (
        model
        .backbone
        .encoder
        .layer
    )


    print(
        f"VideoMAE encoder blocks: "
        f"{len(layers)}"
    )


    for layer in layers[
        -n_blocks:
    ]:

        for p in layer.parameters():

            p.requires_grad = True


    if hasattr(
        model.backbone,
        "layernorm",
    ):

        for p in (
            model
            .backbone
            .layernorm
            .parameters()
        ):

            p.requires_grad = True


    for p in model.classifier.parameters():

        p.requires_grad = True


def parameter_counts(
    model
):

    total = sum(
        p.numel()
        for p in model.parameters()
    )

    trainable = sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )

    return total, trainable


# ==================================================================================================
# 22. TRAINING FUNCTION
# ==================================================================================================

def train_one_epoch(
    model,
    loader,
    optimizer,
    scaler,
    stage,
    epoch,
):

    model.train()

    optimizer.zero_grad(
        set_to_none=True
    )

    total_loss = 0.0
    total = 0

    correct1 = 0
    correct5 = 0

    start = time.time()

    num_batches = len(loader)


    for step, batch in enumerate(
        loader,
        start=1,
    ):

        x = batch[
            "pixel_values"
        ].to(
            DEVICE,
            non_blocking=True,
        )

        y = batch[
            "label"
        ].to(
            DEVICE,
            non_blocking=True,
        )


        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(
                DEVICE.type
                == "cuda"
            ),
        ):

            logits, _ = model(x)

            raw_loss = criterion(
                logits,
                y,
            )

            loss = (
                raw_loss
                / GRAD_ACCUM_STEPS
            )


        scaler.scale(
            loss
        ).backward()


        should_update = (
            step
            % GRAD_ACCUM_STEPS
            == 0
            or
            step
            == num_batches
        )


        if should_update:

            scaler.unscale_(
                optimizer
            )

            torch.nn.utils.clip_grad_norm_(
                [
                    p
                    for p
                    in model.parameters()
                    if p.requires_grad
                ],
                GRAD_CLIP,
            )

            scaler.step(
                optimizer
            )

            scaler.update()

            optimizer.zero_grad(
                set_to_none=True
            )


        bs = y.size(0)

        total += bs

        total_loss += (
            raw_loss.item()
            * bs
        )

        correct1 += topk_correct(
            logits,
            y,
            1,
        )

        correct5 += topk_correct(
            logits,
            y,
            5,
        )


        if (
            step == 1
            or
            step % LOG_EVERY == 0
            or
            step == num_batches
        ):

            elapsed = (
                time.time()
                - start
            )

            speed = (
                total
                / max(
                    elapsed,
                    1e-9,
                )
            )

            train_loss = (
                total_loss
                / total
            )

            t1 = (
                100.0
                * correct1
                / total
            )

            t5 = (
                100.0
                * correct5
                / total
            )


            print(
                f"[{stage}] "
                f"E{epoch:02d} "
                f"{step:04d}/{num_batches} | "
                f"loss={train_loss:.4f} | "
                f"T1={t1:.2f}% | "
                f"T5={t5:.2f}% | "
                f"{speed:.2f} vid/s"
            )


    return {
        "loss":
            total_loss / total,

        "top1":
            100.0
            * correct1 / total,

        "top5":
            100.0
            * correct5 / total,

        "seconds":
            time.time() - start,

        "samples":
            total,
    }


# ==================================================================================================
# 23. VALIDATION FUNCTION
# ==================================================================================================

@torch.no_grad()
def evaluate(
    model,
    loader,
    collect_predictions=False,
):

    model.eval()

    total_loss = 0.0
    total = 0

    correct1 = 0
    correct5 = 0

    predictions = []

    start = time.time()


    for batch in loader:

        x = batch[
            "pixel_values"
        ].to(
            DEVICE,
            non_blocking=True,
        )

        y = batch[
            "label"
        ].to(
            DEVICE,
            non_blocking=True,
        )


        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(
                DEVICE.type
                == "cuda"
            ),
        ):

            logits, _ = model(x)

            loss = criterion(
                logits,
                y,
            )


        bs = y.size(0)

        total += bs

        total_loss += (
            loss.item()
            * bs
        )

        correct1 += topk_correct(
            logits,
            y,
            1,
        )

        correct5 += topk_correct(
            logits,
            y,
            5,
        )


        if collect_predictions:

            probs = torch.softmax(
                logits.float(),
                dim=1,
            )

            top_probs, top_ids = (
                probs.topk(
                    5,
                    dim=1,
                )
            )


            for i in range(bs):

                true_id = int(
                    y[i].item()
                )

                pred_id = int(
                    top_ids[
                        i,
                        0,
                    ].item()
                )


                row = {
                    "uid":
                        batch[
                            "uid"
                        ][i],

                    "true_id":
                        true_id,

                    "true_gloss":
                        id_to_gloss[
                            true_id
                        ],

                    "pred_id":
                        pred_id,

                    "pred_gloss":
                        id_to_gloss[
                            pred_id
                        ],

                    "confidence":
                        float(
                            top_probs[
                                i,
                                0,
                            ].item()
                        ),

                    "correct_top1":
                        int(
                            true_id
                            == pred_id
                        ),
                }


                for rank in range(5):

                    cid = int(
                        top_ids[
                            i,
                            rank,
                        ].item()
                    )

                    row[
                        f"top{rank+1}_id"
                    ] = cid

                    row[
                        f"top{rank+1}_gloss"
                    ] = id_to_gloss[
                        cid
                    ]

                    row[
                        f"top{rank+1}_prob"
                    ] = float(
                        top_probs[
                            i,
                            rank,
                        ].item()
                    )


                predictions.append(
                    row
                )


    return {
        "loss":
            total_loss / total,

        "top1":
            100.0
            * correct1 / total,

        "top5":
            100.0
            * correct5 / total,

        "correct1":
            correct1,

        "correct5":
            correct5,

        "total":
            total,

        "seconds":
            time.time() - start,

        "predictions":
            predictions,
    }


# ==================================================================================================
# 24. OUTPUT FILES
# ==================================================================================================

BEST_MODEL = (
    MODEL_DIR
    / "l20d_b0_best_original_videomae.pt"
)

RESUME_MODEL = (
    MODEL_DIR
    / "l20d_b0_resume.pt"
)

HISTORY_FILE = (
    REPORT_DIR
    / "l20d_b0_training_history.csv"
)

BEST_VAL_PREDICTIONS = (
    REPORT_DIR
    / "l20d_b0_best_validation_predictions.csv"
)

FINAL_VAL_PREDICTIONS = (
    REPORT_DIR
    / "l20d_b0_final_validation_predictions.csv"
)

FINAL_MODEL = (
    MODEL_DIR
    / "l20d_b0_final_original_videomae_215class.pt"
)

SUMMARY_FILE = (
    OUTPUT_ROOT
    / "l20d_b0_summary.json"
)

MASTER_FREEZE = (
    OUTPUT_ROOT
    / "L20D_B0_MASTER_FREEZE.json"
)


# ==================================================================================================
# 25. TRAINING STATE
# ==================================================================================================

history = []

global_epoch = 0

best_top1 = -1.0
best_top5 = -1.0
best_loss = float("inf")

best_epoch = None
best_stage = None

epochs_without_improvement = 0


def better_model(
    top1,
    top5,
    loss,
):

    if top1 > best_top1:
        return True

    if top1 == best_top1:

        if top5 > best_top5:
            return True

        if (
            top5 == best_top5
            and
            loss < best_loss
        ):
            return True

    return False


def save_checkpoint(
    path,
    optimizer,
    scaler,
    stage,
    stage_epoch,
):

    torch.save(
        {
            "model_state_dict":
                model.state_dict(),

            "optimizer_state_dict":
                optimizer.state_dict(),

            "scaler_state_dict":
                scaler.state_dict(),

            "stage":
                stage,

            "stage_epoch":
                stage_epoch,

            "global_epoch":
                global_epoch,

            "best_top1":
                best_top1,

            "best_top5":
                best_top5,

            "best_loss":
                best_loss,

            "num_classes":
                NUM_CLASSES,

            "gloss_to_id":
                gloss_to_id,

            "id_to_gloss":
                id_to_gloss,

            "initialization":
                MODEL_NAME,

            "l20c_loaded":
                False,

            "locked_test_used":
                False,
        },
        path,
    )


# ==================================================================================================
# 26. TRAIN STAGE
# ==================================================================================================

def train_stage(
    stage,
    epochs,
    lr,
):

    global global_epoch
    global best_top1
    global best_top5
    global best_loss
    global best_epoch
    global best_stage
    global epochs_without_improvement
    global history


    if stage == "HEAD":

        freeze_for_head(
            model
        )

    elif stage == "FINETUNE4":

        unfreeze_final_blocks(
            model,
            TRAINABLE_FINAL_BLOCKS,
        )

    else:

        raise ValueError(
            stage
        )


    total_params, trainable_params = (
        parameter_counts(
            model
        )
    )


    section(
        f"STAGE {stage}"
    )

    print(
        f"LR                   : "
        f"{lr}"
    )

    print(
        f"Epochs               : "
        f"{epochs}"
    )

    print(
        f"Total parameters     : "
        f"{total_params:,}"
    )

    print(
        f"Trainable parameters : "
        f"{trainable_params:,}"
    )


    optimizer = torch.optim.AdamW(
        [
            p
            for p
            in model.parameters()
            if p.requires_grad
        ],
        lr=lr,
        weight_decay=WEIGHT_DECAY,
    )


    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=(
            DEVICE.type
            == "cuda"
        ),
    )


    for stage_epoch in range(
        1,
        epochs + 1,
    ):

        global_epoch += 1


        section(
            f"{stage} — "
            f"EPOCH {stage_epoch}/{epochs} "
            f"(GLOBAL {global_epoch})"
        )


        train_result = train_one_epoch(
            model,
            train_loader,
            optimizer,
            scaler,
            stage,
            global_epoch,
        )


        val_result = evaluate(
            model,
            val_loader,
            collect_predictions=True,
        )


        print("\nTRAIN")

        print(
            f"Loss : "
            f"{train_result['loss']:.6f}"
        )

        print(
            f"Top1 : "
            f"{train_result['top1']:.3f}%"
        )

        print(
            f"Top5 : "
            f"{train_result['top5']:.3f}%"
        )


        print("\nVALIDATION")

        print(
            f"Loss : "
            f"{val_result['loss']:.6f}"
        )

        print(
            f"Top1 : "
            f"{val_result['top1']:.3f}% "
            f"({val_result['correct1']}/"
            f"{val_result['total']})"
        )

        print(
            f"Top5 : "
            f"{val_result['top5']:.3f}% "
            f"({val_result['correct5']}/"
            f"{val_result['total']})"
        )


        history.append(
            {
                "global_epoch":
                    global_epoch,

                "stage":
                    stage,

                "stage_epoch":
                    stage_epoch,

                "train_loss":
                    train_result[
                        "loss"
                    ],

                "train_top1":
                    train_result[
                        "top1"
                    ],

                "train_top5":
                    train_result[
                        "top5"
                    ],

                "val_loss":
                    val_result[
                        "loss"
                    ],

                "val_top1":
                    val_result[
                        "top1"
                    ],

                "val_top5":
                    val_result[
                        "top5"
                    ],
            }
        )


        pd.DataFrame(
            history
        ).to_csv(
            HISTORY_FILE,
            index=False,
        )


        improved = better_model(
            val_result["top1"],
            val_result["top5"],
            val_result["loss"],
        )


        if improved:

            best_top1 = (
                val_result[
                    "top1"
                ]
            )

            best_top5 = (
                val_result[
                    "top5"
                ]
            )

            best_loss = (
                val_result[
                    "loss"
                ]
            )

            best_epoch = (
                global_epoch
            )

            best_stage = (
                stage
            )

            epochs_without_improvement = 0


            save_checkpoint(
                BEST_MODEL,
                optimizer,
                scaler,
                stage,
                stage_epoch,
            )


            pd.DataFrame(
                val_result[
                    "predictions"
                ]
            ).to_csv(
                BEST_VAL_PREDICTIONS,
                index=False,
            )


            print(
                "\n[SAVED BEST]"
            )

            print(
                f"Epoch    : "
                f"{best_epoch}"
            )

            print(
                f"Stage    : "
                f"{best_stage}"
            )

            print(
                f"Val Top1 : "
                f"{best_top1:.3f}%"
            )

            print(
                f"Val Top5 : "
                f"{best_top5:.3f}%"
            )


        else:

            epochs_without_improvement += 1

            print(
                f"\n[NO IMPROVEMENT] "
                f"{epochs_without_improvement}/"
                f"{EARLY_STOPPING_PATIENCE}"
            )


        save_checkpoint(
            RESUME_MODEL,
            optimizer,
            scaler,
            stage,
            stage_epoch,
        )


        if (
            stage == "FINETUNE4"
            and
            epochs_without_improvement
            >= EARLY_STOPPING_PATIENCE
        ):

            print(
                "\n[EARLY STOPPING]"
            )

            return


# ==================================================================================================
# 27. START TRAINING
# ==================================================================================================

section(
    "27. STARTING L20D-B0 BASELINE"
)

print(
    "[LOCK] Locked test video decoding: DISABLED"
)

print(
    "[LOCK] L20C loading: DISABLED"
)

print(
    "[INFO] Starting HEAD warm-up."
)


train_stage(
    stage="HEAD",
    epochs=HEAD_EPOCHS,
    lr=HEAD_LR,
)


# Reset patience at transition because fine-tuning
# is a distinct optimization stage.

epochs_without_improvement = 0


train_stage(
    stage="FINETUNE4",
    epochs=FINETUNE_EPOCHS,
    lr=FINETUNE_LR,
)


# ==================================================================================================
# 28. LOAD BEST MODEL
# ==================================================================================================

section(
    "28. LOADING BEST BASELINE MODEL"
)

if not BEST_MODEL.exists():

    raise RuntimeError(
        "No best model checkpoint exists."
    )


best_checkpoint = torch.load(
    BEST_MODEL,
    map_location="cpu",
    weights_only=False,
)

model.load_state_dict(
    best_checkpoint[
        "model_state_dict"
    ]
)

model = model.to(
    DEVICE
)


print(
    f"Best epoch : "
    f"{best_checkpoint['global_epoch']}"
)

print(
    f"Best stage : "
    f"{best_checkpoint['stage']}"
)

print(
    f"Best Top1  : "
    f"{best_checkpoint['best_top1']:.3f}%"
)

print(
    f"Best Top5  : "
    f"{best_checkpoint['best_top5']:.3f}%"
)


# ==================================================================================================
# 29. UNIQUE TRAIN EVALUATION
# ==================================================================================================

section(
    "29. UNIQUE UN-AUGMENTED TRAIN EVALUATION"
)

unique_train_dataset = CISLRDataset(
    train_df,
    processor,
    train=False,
)

unique_train_loader = DataLoader(
    unique_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=(
        NUM_WORKERS > 0
    ),
    prefetch_factor=(
        2
        if NUM_WORKERS > 0
        else None
    ),
)


unique_train_result = evaluate(
    model,
    unique_train_loader,
    collect_predictions=False,
)


print(
    f"Train Top1 : "
    f"{unique_train_result['top1']:.3f}% "
    f"({unique_train_result['correct1']}/"
    f"{unique_train_result['total']})"
)

print(
    f"Train Top5 : "
    f"{unique_train_result['top5']:.3f}% "
    f"({unique_train_result['correct5']}/"
    f"{unique_train_result['total']})"
)


# ==================================================================================================
# 30. FINAL VALIDATION
# ==================================================================================================

section(
    "30. FINAL VALIDATION"
)

final_val = evaluate(
    model,
    val_loader,
    collect_predictions=True,
)


print(
    f"Validation Top1 : "
    f"{final_val['top1']:.3f}% "
    f"({final_val['correct1']}/"
    f"{final_val['total']})"
)

print(
    f"Validation Top5 : "
    f"{final_val['top5']:.3f}% "
    f"({final_val['correct5']}/"
    f"{final_val['total']})"
)

print(
    f"Validation Loss : "
    f"{final_val['loss']:.6f}"
)


pd.DataFrame(
    final_val[
        "predictions"
    ]
).to_csv(
    FINAL_VAL_PREDICTIONS,
    index=False,
)


# ==================================================================================================
# 31. SAVE FINAL MODEL
# ==================================================================================================

section(
    "31. SAVING FINAL BASELINE MODEL"
)

torch.save(
    {
        "model_state_dict":
            model.state_dict(),

        "model_name":
            MODEL_NAME,

        "initialization":
            "original_pretrained_videomae",

        "num_classes":
            NUM_CLASSES,

        "num_frames":
            NUM_FRAMES,

        "gloss_to_id":
            gloss_to_id,

        "id_to_gloss":
            id_to_gloss,

        "best_epoch":
            best_checkpoint[
                "global_epoch"
            ],

        "best_stage":
            best_checkpoint[
                "stage"
            ],

        "unique_train_top1":
            unique_train_result[
                "top1"
            ],

        "unique_train_top5":
            unique_train_result[
                "top5"
            ],

        "validation_top1":
            final_val[
                "top1"
            ],

        "validation_top5":
            final_val[
                "top5"
            ],

        "validation_loss":
            final_val[
                "loss"
            ],

        "l20c_loaded":
            False,

        "locked_test_used":
            False,
    },
    FINAL_MODEL,
)


print(
    f"[SAVED] {FINAL_MODEL}"
)


# ==================================================================================================
# 32. SUMMARY
# ==================================================================================================

summary = {
    "experiment":
        "L20D_B0_ORIGINAL_VIDEOMAE_BASELINE",

    "status":
        "COMPLETE_AND_FROZEN",

    "initialization":
        MODEL_NAME,

    "l20c_loaded":
        False,

    "classes":
        NUM_CLASSES,

    "train_videos":
        len(train_df),

    "validation_videos":
        len(val_df),

    "locked_test_videos":
        len(locked_df),

    "best_epoch":
        int(
            best_checkpoint[
                "global_epoch"
            ]
        ),

    "best_stage":
        best_checkpoint[
            "stage"
        ],

    "unique_train_top1":
        unique_train_result[
            "top1"
        ],

    "unique_train_top5":
        unique_train_result[
            "top5"
        ],

    "validation_top1":
        final_val[
            "top1"
        ],

    "validation_top5":
        final_val[
            "top5"
        ],

    "validation_loss":
        final_val[
            "loss"
        ],

    "representation_diagnostic":
        diagnostic_report,

    "locked_test_used":
        False,
}


with open(
    SUMMARY_FILE,
    "w",
) as f:

    json.dump(
        summary,
        f,
        indent=2,
    )


# ==================================================================================================
# 33. ARTIFACT HASHES
# ==================================================================================================

section(
    "33. ARTIFACT HASHES"
)

artifacts = {
    "train_manifest":
        B0_TRAIN_MANIFEST,

    "validation_manifest":
        B0_VAL_MANIFEST,

    "diagnostic":
        DIAGNOSTIC_REPORT,

    "history":
        HISTORY_FILE,

    "best_model":
        BEST_MODEL,

    "final_model":
        FINAL_MODEL,

    "validation_predictions":
        FINAL_VAL_PREDICTIONS,

    "summary":
        SUMMARY_FILE,
}


hashes = {}

for name, path in artifacts.items():

    if Path(path).exists():

        digest = sha256_file(
            path
        )

        hashes[name] = digest

        print(
            f"{name:30s}: "
            f"{digest}"
        )


# ==================================================================================================
# 34. MASTER FREEZE
# ==================================================================================================

master = {
    "status":
        "COMPLETE_AND_FROZEN",

    "experiment":
        "L20D_B0_ORIGINAL_VIDEOMAE_BASELINE",

    "protocol": {
        "train":
            710,

        "validation":
            304,

        "locked_test":
            10,

        "classes":
            215,
    },

    "initialization": {
        "model":
            MODEL_NAME,

        "l20c_used":
            False,
    },

    "representation_diagnostic":
        diagnostic_report,

    "results": {
        "train_top1":
            unique_train_result[
                "top1"
            ],

        "train_top5":
            unique_train_result[
                "top5"
            ],

        "validation_top1":
            final_val[
                "top1"
            ],

        "validation_top5":
            final_val[
                "top5"
            ],

        "validation_loss":
            final_val[
                "loss"
            ],

        "best_epoch":
            int(
                best_checkpoint[
                    "global_epoch"
                ]
            ),

        "best_stage":
            best_checkpoint[
                "stage"
            ],
    },

    "integrity": {
        "train_validation_overlap":
            0,

        "train_locked_overlap":
            0,

        "validation_locked_overlap":
            0,

        "locked_test_evaluated":
            False,

        "l20c_loaded":
            False,
    },

    "artifacts": {
        name: {
            "path":
                str(path),

            "sha256":
                hashes.get(name),
        }

        for name, path
        in artifacts.items()
    },
}


with open(
    MASTER_FREEZE,
    "w",
) as f:

    json.dump(
        master,
        f,
        indent=2,
    )


master_sha = sha256_file(
    MASTER_FREEZE
)


# ==================================================================================================
# 35. FINAL REPORT
# ==================================================================================================

section(
    "L20D-B0 — COMPLETE AND FROZEN"
)

print(
    f"Initialization                : "
    f"ORIGINAL pretrained VideoMAE"
)

print(
    f"L20C used                     : "
    f"NO"
)

print(
    f"Classes                       : "
    f"{NUM_CLASSES}"
)

print(
    f"Train videos                  : "
    f"{len(train_df)}"
)

print(
    f"Validation videos             : "
    f"{len(val_df)}"
)

print(
    f"Locked test videos            : "
    f"{len(locked_df)}"
)


print(
    "\nREPRESENTATION BEFORE TRAINING"
)

print(
    f"Original VideoMAE cosine      : "
    f"{mean_cosine:.6f}"
)

print(
    f"L20C observed cosine          : "
    f"{L20C_OBSERVED_COSINE:.6f}"
)


print(
    "\nUNIQUE TRAIN RESULT"
)

print(
    f"Top-1                         : "
    f"{unique_train_result['top1']:.3f}% "
    f"({unique_train_result['correct1']}/"
    f"{unique_train_result['total']})"
)

print(
    f"Top-5                         : "
    f"{unique_train_result['top5']:.3f}% "
    f"({unique_train_result['correct5']}/"
    f"{unique_train_result['total']})"
)


print(
    "\nVALIDATION RESULT"
)

print(
    f"Top-1                         : "
    f"{final_val['top1']:.3f}% "
    f"({final_val['correct1']}/"
    f"{final_val['total']})"
)

print(
    f"Top-5                         : "
    f"{final_val['top5']:.3f}% "
    f"({final_val['correct5']}/"
    f"{final_val['total']})"
)

print(
    f"Loss                          : "
    f"{final_val['loss']:.6f}"
)


print(
    "\nBEST MODEL"
)

print(
    f"Epoch                         : "
    f"{best_checkpoint['global_epoch']}"
)

print(
    f"Stage                         : "
    f"{best_checkpoint['stage']}"
)


print(
    "\nSCIENTIFIC LOCKS"
)

print(
    "Validation used for gradients : NO"
)

print(
    "Locked test decoded/evaluated  : NO"
)

print(
    "L20C checkpoint loaded          : NO"
)


print(
    "\nFinal model:"
)

print(
    FINAL_MODEL
)

print(
    "\nMaster freeze:"
)

print(
    MASTER_FREEZE
)

print(
    "\nMaster SHA256:"
)

print(
    master_sha
)

print(
    f"\nDisk free: "
    f"{disk_free_gb(PROJECT_ROOT):.2f} GB"
)

print(
    "\n[PASS] L20D-B0 baseline completed."
)

print(
    "[PASS] Original pretrained VideoMAE used."
)

print(
    "[PASS] Corrected 710/304 protocol preserved."
)

print(
    "[PASS] Permanent 10-video locked test remained untouched."
)

print(
    "[LOCK] Do not tune using the locked test."
)

print("=" * 120)


CISLR — L20D-B0
ORIGINAL VIDEOMAE 215-CLASS SUPERVISED BASELINE
PyTorch                 : 2.14.0+cu130
OpenCV                  : 5.0.0
CUDA available          : True
Device                  : cuda
GPU                     : NVIDIA GeForce RTX 4050 Laptop GPU
GPU memory              : 5.64 GB
Disk free               : 16.75 GB
Classes                 : 215
Frames/video            : 16
Physical batch          : 1
Gradient accumulation   : 8
Effective batch         : 8
Head epochs             : 8
Fine-tune epochs        : 35
Initialization          : ORIGINAL pretrained VideoMAE
L20C loaded             : NO
Locked test evaluation  : DISABLED

4. VERIFYING CORRECTED L20B ARTIFACTS
[FOUND] /home/dipshikha/Music/cao/CISLR_RESEARCH/CISLR_LANDMARK_MODEL/audit/l20b_corrected_strict_protocol_freeze/manifests/l20b_corrected_train_manifest.csv
[FOUND] /home/dipshikha/Music/cao/CISLR_RESEARCH/CISLR_LANDMARK_MODEL/audit/l20b_corrected_strict_protocol_freeze/manifests/l20b_corrected_validation_manife

AttributeError: 'NoneType' object has no attribute 'parameters'

In [6]:
# ==================================================================================================
# L20D-B0 HOTFIX
# CONTINUE FROM FINETUNE4 WITHOUT RE-RUNNING HEAD EPOCHS 1-8
# ==================================================================================================

import gc
import torch


print("=" * 120)
print("L20D-B0 HOTFIX — CONTINUING FROM FINETUNE4")
print("=" * 120)


# ==================================================================================================
# 1. FIX FINAL-BLOCK UNFREEZING
# ==================================================================================================

def unfreeze_final_blocks(
    model,
    n_blocks=4,
):

    # Freeze everything in backbone first.
    for p in model.backbone.parameters():
        p.requires_grad = False


    # VideoMAE transformer blocks.
    layers = model.backbone.encoder.layer

    print(
        f"VideoMAE encoder blocks : "
        f"{len(layers)}"
    )


    if n_blocks > len(layers):

        raise ValueError(
            f"Requested {n_blocks} blocks, "
            f"but backbone has only {len(layers)}."
        )


    # Unfreeze final N transformer blocks.
    for layer in layers[-n_blocks:]:

        for p in layer.parameters():

            p.requires_grad = True


    # Some Transformers versions expose `layernorm`
    # but set it to None.
    backbone_layernorm = getattr(
        model.backbone,
        "layernorm",
        None,
    )


    if backbone_layernorm is not None:

        for p in backbone_layernorm.parameters():

            p.requires_grad = True

        print(
            "[INFO] Backbone layernorm unfrozen."
        )

    else:

        print(
            "[INFO] model.backbone.layernorm is None — safely skipped."
        )


    # Classifier always trainable.
    for p in model.classifier.parameters():

        p.requires_grad = True


    print(
        f"[PASS] Final {n_blocks} VideoMAE blocks + classifier enabled."
    )


# ==================================================================================================
# 2. VERIFY CURRENT IN-MEMORY STATE
# ==================================================================================================

print("\nCurrent training state:")

print(
    f"global_epoch                : "
    f"{global_epoch}"
)

print(
    f"best validation Top-1       : "
    f"{best_top1:.3f}%"
)

print(
    f"best validation Top-5       : "
    f"{best_top5:.3f}%"
)

print(
    f"best epoch                  : "
    f"{best_epoch}"
)

print(
    f"best stage                  : "
    f"{best_stage}"
)

print(
    f"history entries             : "
    f"{len(history)}"
)


# HEAD must already have completed.

if global_epoch != 8:

    raise RuntimeError(
        f"Expected global_epoch=8 after HEAD training, "
        f"but found {global_epoch}. "
        f"Do NOT continue until this is checked."
    )


if len(history) != 8:

    raise RuntimeError(
        f"Expected 8 HEAD history entries, "
        f"but found {len(history)}."
    )


if not BEST_MODEL.exists():

    raise RuntimeError(
        f"Best HEAD checkpoint missing:\n"
        f"{BEST_MODEL}"
    )


print(
    "\n[PASS] All 8 HEAD epochs are preserved."
)


# ==================================================================================================
# 3. TEST UNFREEZING BEFORE STARTING TRAINING
# ==================================================================================================

print("\n" + "=" * 120)
print("TESTING FINETUNE4 PARAMETER CONFIGURATION")
print("=" * 120)


unfreeze_final_blocks(
    model,
    TRAINABLE_FINAL_BLOCKS,
)


total_params, trainable_params = (
    parameter_counts(model)
)


print(
    f"Total parameters     : "
    f"{total_params:,}"
)

print(
    f"Trainable parameters : "
    f"{trainable_params:,}"
)


if trainable_params <= 165_335:

    raise RuntimeError(
        "Backbone blocks did not become trainable. "
        "Stopping before FINETUNE4."
    )


print(
    "\n[PASS] Backbone fine-tuning parameters successfully enabled."
)


# ==================================================================================================
# 4. IMPORTANT — PRESERVE BEST HEAD MODEL
# ==================================================================================================
#
# We DO NOT reload BEST_MODEL here.
#
# Reason:
# HEAD epoch 8 is the natural continuation point for optimization.
# The best validation checkpoint remains safely stored separately.
#
# If FINETUNE4 fails to improve, the final script will still restore
# the best validation checkpoint.
#
# ==================================================================================================

print(
    "\n[INFO] Continuing from HEAD epoch 8 model state."
)

print(
    f"[SAFE] Existing best checkpoint remains:\n"
    f"{BEST_MODEL}"
)


# ==================================================================================================
# 5. RESET EARLY-STOPPING COUNTER FOR NEW STAGE
# ==================================================================================================

epochs_without_improvement = 0


# ==================================================================================================
# 6. CLEAR UNUSED GPU MEMORY
# ==================================================================================================

gc.collect()

if torch.cuda.is_available():

    torch.cuda.empty_cache()


# ==================================================================================================
# 7. CONTINUE FINETUNE4
# ==================================================================================================

print("\n" + "=" * 120)
print("STARTING FINETUNE4")
print("=" * 120)

print(
    f"Starting global epoch : "
    f"{global_epoch + 1}"
)

print(
    f"Fine-tune epochs      : "
    f"{FINETUNE_EPOCHS}"
)

print(
    f"Learning rate         : "
    f"{FINETUNE_LR}"
)

print(
    f"Final blocks          : "
    f"{TRAINABLE_FINAL_BLOCKS}"
)

print(
    "[LOCK] L20C checkpoint will NOT be loaded."
)

print(
    "[LOCK] Permanent 10-video test set remains untouched."
)


train_stage(
    stage="FINETUNE4",
    epochs=FINETUNE_EPOCHS,
    lr=FINETUNE_LR,
)


# ==================================================================================================
# 8. LOAD BEST MODEL AFTER FINETUNING
# ==================================================================================================

print("\n" + "=" * 120)
print("LOADING BEST L20D-B0 MODEL")
print("=" * 120)


if not BEST_MODEL.exists():

    raise RuntimeError(
        f"Best checkpoint missing:\n"
        f"{BEST_MODEL}"
    )


best_checkpoint = torch.load(
    BEST_MODEL,
    map_location="cpu",
    weights_only=False,
)


model.load_state_dict(
    best_checkpoint[
        "model_state_dict"
    ]
)

model = model.to(
    DEVICE
)


print(
    f"Best epoch : "
    f"{best_checkpoint['global_epoch']}"
)

print(
    f"Best stage : "
    f"{best_checkpoint['stage']}"
)

print(
    f"Best Top-1 : "
    f"{best_checkpoint['best_top1']:.3f}%"
)

print(
    f"Best Top-5 : "
    f"{best_checkpoint['best_top5']:.3f}%"
)


print("\n" + "=" * 120)
print("FINETUNE4 TRAINING FINISHED")
print("=" * 120)

print(
    "Do NOT evaluate the locked 10-video test set."
)

L20D-B0 HOTFIX — CONTINUING FROM FINETUNE4

Current training state:
global_epoch                : 8
best validation Top-1       : 3.289%
best validation Top-5       : 8.224%
best epoch                  : 7
best stage                  : HEAD
history entries             : 8

[PASS] All 8 HEAD epochs are preserved.

TESTING FINETUNE4 PARAMETER CONFIGURATION
VideoMAE encoder blocks : 12
[INFO] model.backbone.layernorm is None — safely skipped.
[PASS] Final 4 VideoMAE blocks + classifier enabled.
Total parameters     : 86,390,999
Trainable parameters : 28,513,751

[PASS] Backbone fine-tuning parameters successfully enabled.

[INFO] Continuing from HEAD epoch 8 model state.
[SAFE] Existing best checkpoint remains:
/home/dipshikha/Music/cao/CISLR_RESEARCH/CISLR_LANDMARK_MODEL/audit/l20d_b0_original_videomae_215class/models/l20d_b0_best_original_videomae.pt

STARTING FINETUNE4
Starting global epoch : 9
Fine-tune epochs      : 35
Learning rate         : 2e-05
Final blocks          : 4
[LOCK] L2

KeyboardInterrupt: 

In [2]:
# ==================================================================================================
# CISLR — L20D-B1
# LOW-SHOT / GENERALIZATION-ORIENTED 215-CLASS VIDEOMAE
# ==================================================================================================
#
# CLEAN EXPERIMENT
#
# TRAIN       : 710 corrected L20B videos
# VALIDATION  : 304 corrected L20B videos
# CLASSES     : 215
# LOCKED TEST : 10 videos — NEVER DECODED / NEVER EVALUATED
#
# INITIALIZATION
# --------------
# ORIGINAL MCG-NJU/videomae-base-finetuned-kinetics
#
# IMPORTANT:
# - L20C IS NOT LOADED.
# - B0 IS NOT LOADED.
# - This is a fresh B1 experiment.
#
# GENERALIZATION IMPROVEMENTS
# ---------------------------
# 1. Cosine classifier instead of raw Linear classifier.
# 2. Label smoothing.
# 3. Sign-safe photometric + mild spatial augmentation.
# 4. Temporal sampling jitter during training.
# 5. Three deterministic temporal views during validation.
# 6. Progressive unfreezing:
#       Stage A : classifier only
#       Stage B : final 2 transformer blocks
#       Stage C : final 6 transformer blocks
# 7. Differential LR:
#       classifier > backbone
# 8. Warmup + cosine LR decay.
# 9. AdamW.
# 10. Gradient clipping.
# 11. Early stopping.
#
# NO:
# - horizontal flip
# - temporal reversal
# - validation gradients
# - locked-test evaluation
#
# ==================================================================================================

import os
import gc
import cv2
import json
import math
import time
import random
import hashlib
import shutil
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import (
    Dataset,
    DataLoader,
    WeightedRandomSampler,
)

from transformers import (
    VideoMAEModel,
    VideoMAEImageProcessor,
)


# ==================================================================================================
# 0. CONFIG
# ==================================================================================================

SEED = 42

PROJECT_ROOT = Path(
    "/home/dipshikha/Music/cao/CISLR_RESEARCH"
)

LANDMARK_ROOT = (
    PROJECT_ROOT
    / "CISLR_LANDMARK_MODEL"
)

VIDEO_ROOT = Path(
    "/home/dipshikha/Music/cao/CISLR_v1.5-a_videos"
)

L20B_ROOT = (
    LANDMARK_ROOT
    / "audit"
    / "l20b_corrected_strict_protocol_freeze"
)

OUTPUT_ROOT = (
    LANDMARK_ROOT
    / "audit"
    / "l20d_b1_lowshot_videomae_215class"
)

MANIFEST_DIR = OUTPUT_ROOT / "manifests"
MODEL_DIR = OUTPUT_ROOT / "models"
REPORT_DIR = OUTPUT_ROOT / "reports"

for d in [
    OUTPUT_ROOT,
    MANIFEST_DIR,
    MODEL_DIR,
    REPORT_DIR,
]:
    d.mkdir(
        parents=True,
        exist_ok=True,
    )


TRAIN_MANIFEST = (
    L20B_ROOT
    / "manifests"
    / "l20b_corrected_train_manifest.csv"
)

VAL_MANIFEST = (
    L20B_ROOT
    / "manifests"
    / "l20b_corrected_validation_manifest.csv"
)

LOCKED_MANIFEST = (
    L20B_ROOT
    / "manifests"
    / "l20b_permanent_l19_locked_test_manifest.csv"
)

CLASS_MAPPING_FILE = (
    L20B_ROOT
    / "manifests"
    / "l20b_corrected_class_mapping.csv"
)

MODEL_NAME = (
    "MCG-NJU/videomae-base-finetuned-kinetics"
)

NUM_CLASSES = 215
NUM_FRAMES = 16

BATCH_SIZE = 1
GRAD_ACCUM = 8

NUM_WORKERS = 4

LABEL_SMOOTHING = 0.10
WEIGHT_DECAY = 0.05

GRAD_CLIP = 1.0

# --------------------------------------------------------------------------------------------------
# STAGES
# --------------------------------------------------------------------------------------------------

HEAD_EPOCHS = 6
STAGE2_EPOCHS = 12
STAGE3_EPOCHS = 25

HEAD_LR = 5e-4

STAGE2_HEAD_LR = 2e-4
STAGE2_BACKBONE_LR = 8e-6

STAGE3_HEAD_LR = 8e-5
STAGE3_BACKBONE_LR = 3e-6

STAGE2_BLOCKS = 2
STAGE3_BLOCKS = 6

WARMUP_RATIO = 0.10

EARLY_STOP_PATIENCE = 8

LOG_EVERY = 100

# --------------------------------------------------------------------------------------------------
# VALIDATION VIEWS
# --------------------------------------------------------------------------------------------------

VAL_TEMPORAL_VIEWS = [
    0.15,
    0.50,
    0.85,
]

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


# ==================================================================================================
# 1. REPRODUCIBILITY
# ==================================================================================================

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)

if torch.cuda.is_available():

    torch.cuda.manual_seed_all(SEED)

    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True


torch.backends.cudnn.benchmark = True

cv2.setNumThreads(0)

os.environ[
    "TOKENIZERS_PARALLELISM"
] = "false"


# ==================================================================================================
# 2. HELPERS
# ==================================================================================================

def section(title):

    print("\n" + "=" * 120)
    print(title)
    print("=" * 120)


def normalize_uid(x):

    x = str(x).strip()

    if x.endswith(".mp4"):
        x = x[:-4]

    return x


def find_column(
    df,
    candidates,
):

    lower_map = {
        str(c).lower(): c
        for c in df.columns
    }

    for candidate in candidates:

        if candidate.lower() in lower_map:

            return lower_map[
                candidate.lower()
            ]

    return None


def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        while True:

            chunk = f.read(
                1024 * 1024
            )

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


def free_gb(path):

    return (
        shutil.disk_usage(path).free
        / 1024**3
    )


# ==================================================================================================
# 3. START
# ==================================================================================================

section(
    "CISLR — L20D-B1\n"
    "LOW-SHOT 215-CLASS VIDEOMAE"
)

print(
    "PyTorch                 :",
    torch.__version__,
)

print(
    "OpenCV                  :",
    cv2.__version__,
)

print(
    "Device                  :",
    DEVICE,
)

if torch.cuda.is_available():

    print(
        "GPU                     :",
        torch.cuda.get_device_name(0),
    )

    print(
        "GPU memory              :",
        f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB",
    )

print(
    "Disk free               :",
    f"{free_gb(PROJECT_ROOT):.2f} GB",
)

print(
    "Classes                 :",
    NUM_CLASSES,
)

print(
    "Frames                  :",
    NUM_FRAMES,
)

print(
    "Effective batch         :",
    BATCH_SIZE * GRAD_ACCUM,
)

print(
    "Initialization          : ORIGINAL VideoMAE"
)

print(
    "L20C loaded             : NO"
)

print(
    "B0 loaded               : NO"
)

print(
    "Locked test evaluation  : DISABLED"
)


# ==================================================================================================
# 4. VERIFY FILES
# ==================================================================================================

section(
    "4. VERIFYING CORRECTED PROTOCOL"
)

for path in [
    TRAIN_MANIFEST,
    VAL_MANIFEST,
    LOCKED_MANIFEST,
    CLASS_MAPPING_FILE,
]:

    if not path.exists():

        raise FileNotFoundError(
            path
        )

    print(
        "[FOUND]",
        path,
    )


# ==================================================================================================
# 5. LOAD DATA
# ==================================================================================================

train_df = pd.read_csv(
    TRAIN_MANIFEST
)

val_df = pd.read_csv(
    VAL_MANIFEST
)

locked_df = pd.read_csv(
    LOCKED_MANIFEST
)

mapping_df = pd.read_csv(
    CLASS_MAPPING_FILE
)


uid_candidates = [
    "uid",
    "video_uid",
    "video_id",
    "id",
]

gloss_candidates = [
    "gloss",
    "label",
    "word",
    "class_name",
]


train_uid_col = find_column(
    train_df,
    uid_candidates,
)

val_uid_col = find_column(
    val_df,
    uid_candidates,
)

locked_uid_col = find_column(
    locked_df,
    uid_candidates,
)

train_gloss_col = find_column(
    train_df,
    gloss_candidates,
)

val_gloss_col = find_column(
    val_df,
    gloss_candidates,
)


if any(
    x is None
    for x in [
        train_uid_col,
        val_uid_col,
        locked_uid_col,
        train_gloss_col,
        val_gloss_col,
    ]
):

    raise RuntimeError(
        "Required manifest columns not found."
    )


train_df["uid"] = (
    train_df[train_uid_col]
    .astype(str)
    .map(normalize_uid)
)

val_df["uid"] = (
    val_df[val_uid_col]
    .astype(str)
    .map(normalize_uid)
)

locked_df["uid"] = (
    locked_df[locked_uid_col]
    .astype(str)
    .map(normalize_uid)
)

train_df["gloss"] = (
    train_df[train_gloss_col]
    .astype(str)
    .str.strip()
)

val_df["gloss"] = (
    val_df[val_gloss_col]
    .astype(str)
    .str.strip()
)


# ==================================================================================================
# 6. CLASS MAPPING
# ==================================================================================================

mapping_gloss_col = find_column(
    mapping_df,
    [
        "gloss",
        "label",
        "word",
    ],
)

mapping_id_col = find_column(
    mapping_df,
    [
        "class_id",
        "label_id",
        "index",
    ],
)


if (
    mapping_gloss_col is None
    or
    mapping_id_col is None
):

    raise RuntimeError(
        "Class mapping columns missing."
    )


mapping = mapping_df[
    [
        mapping_gloss_col,
        mapping_id_col,
    ]
].copy()

mapping.columns = [
    "gloss",
    "class_id",
]

mapping["gloss"] = (
    mapping["gloss"]
    .astype(str)
    .str.strip()
)

mapping["class_id"] = (
    mapping["class_id"]
    .astype(int)
)


gloss_to_id = dict(
    zip(
        mapping["gloss"],
        mapping["class_id"],
    )
)

id_to_gloss = {
    int(v): k
    for k, v
    in gloss_to_id.items()
}


if len(gloss_to_id) != NUM_CLASSES:

    raise RuntimeError(
        f"Expected {NUM_CLASSES} classes."
    )


train_df["label"] = (
    train_df["gloss"]
    .map(gloss_to_id)
)

val_df["label"] = (
    val_df["gloss"]
    .map(gloss_to_id)
)


if (
    train_df["label"].isna().any()
    or
    val_df["label"].isna().any()
):

    raise RuntimeError(
        "Unmapped class detected."
    )


train_df["label"] = (
    train_df["label"]
    .astype(int)
)

val_df["label"] = (
    val_df["label"]
    .astype(int)
)


# ==================================================================================================
# 7. STRICT SPLIT INTEGRITY
# ==================================================================================================

section(
    "7. SPLIT INTEGRITY"
)

train_uids = set(
    train_df["uid"]
)

val_uids = set(
    val_df["uid"]
)

locked_uids = set(
    locked_df["uid"]
)


checks = {
    "train_710":
        len(train_df) == 710,

    "validation_304":
        len(val_df) == 304,

    "locked_10":
        len(locked_df) == 10,

    "train_unique_710":
        len(train_uids) == 710,

    "val_unique_304":
        len(val_uids) == 304,

    "train_val_overlap_zero":
        len(
            train_uids
            & val_uids
        ) == 0,

    "train_test_overlap_zero":
        len(
            train_uids
            & locked_uids
        ) == 0,

    "val_test_overlap_zero":
        len(
            val_uids
            & locked_uids
        ) == 0,

    "train_classes_215":
        train_df[
            "label"
        ].nunique()
        == 215,

    "val_classes_215":
        val_df[
            "label"
        ].nunique()
        == 215,
}


for name, result in checks.items():

    print(
        f"{name:40s}: "
        f"{'PASS' if result else 'FAIL'}"
    )


if not all(
    checks.values()
):

    raise RuntimeError(
        "Protocol integrity failure."
    )


# ==================================================================================================
# 8. INDEX VIDEOS
# ==================================================================================================

section(
    "8. INDEXING VIDEOS"
)

video_paths = list(
    VIDEO_ROOT.rglob(
        "*.mp4"
    )
)

video_index = {}

for path in video_paths:

    uid = normalize_uid(
        path.stem
    )

    if uid not in video_index:

        video_index[uid] = path


def attach_paths(df):

    df = df.copy()

    paths = []

    missing = []

    for uid in df["uid"]:

        p = video_index.get(uid)

        if p is None:

            missing.append(uid)
            paths.append(None)

        else:

            paths.append(
                str(p)
            )

    df[
        "video_path"
    ] = paths

    return df, missing


train_df, train_missing = (
    attach_paths(
        train_df
    )
)

val_df, val_missing = (
    attach_paths(
        val_df
    )
)


print(
    f"MP4 files      : "
    f"{len(video_paths)}"
)

print(
    f"Train resolved : "
    f"{len(train_df)-len(train_missing)}/710"
)

print(
    f"Val resolved   : "
    f"{len(val_df)-len(val_missing)}/304"
)


if (
    train_missing
    or
    val_missing
):

    raise RuntimeError(
        "Missing train/validation videos."
    )


# ==================================================================================================
# 9. PROCESSOR
# ==================================================================================================

section(
    "9. PROCESSOR"
)

processor = (
    VideoMAEImageProcessor
    .from_pretrained(
        MODEL_NAME
    )
)

print(
    "[PASS]",
    MODEL_NAME,
)


# ==================================================================================================
# 10. VIDEO DECODING
# ==================================================================================================

def read_video(
    video_path,
):

    cap = cv2.VideoCapture(
        str(video_path)
    )

    if not cap.isOpened():

        raise RuntimeError(
            f"Cannot open {video_path}"
        )


    frames = []

    try:

        while True:

            ok, frame = cap.read()

            if not ok:
                break

            if frame is None:
                continue

            frame = cv2.cvtColor(
                frame,
                cv2.COLOR_BGR2RGB,
            )

            frames.append(
                frame
            )

    finally:

        cap.release()


    if len(frames) == 0:

        raise RuntimeError(
            f"No frames: {video_path}"
        )

    return frames


# ==================================================================================================
# 11. TEMPORAL SAMPLING
# ==================================================================================================

def temporal_sample_train(
    frames,
    num_frames=NUM_FRAMES,
):

    n = len(frames)


    if n <= num_frames:

        ids = np.linspace(
            0,
            n - 1,
            num_frames,
        ).round().astype(int)

        return [
            frames[i]
            for i in ids
        ]


    # Random temporal span.
    #
    # Allows the model to see slightly different temporal
    # portions of the same independent training video.

    span_ratio = random.uniform(
        0.75,
        1.00,
    )

    span = max(
        num_frames,
        int(
            round(
                n * span_ratio
            )
        ),
    )

    span = min(
        span,
        n,
    )


    max_start = (
        n - span
    )

    if max_start > 0:

        start = random.randint(
            0,
            max_start,
        )

    else:

        start = 0


    end = (
        start
        + span
        - 1
    )


    ids = np.linspace(
        start,
        end,
        num_frames,
    ).round().astype(int)


    return [
        frames[i]
        for i in ids
    ]


def temporal_sample_view(
    frames,
    position,
    num_frames=NUM_FRAMES,
):

    n = len(frames)


    if n <= num_frames:

        ids = np.linspace(
            0,
            n - 1,
            num_frames,
        ).round().astype(int)

        return [
            frames[i]
            for i in ids
        ]


    # Use approximately 85% of the video as one deterministic
    # temporal window and move that window through the video.

    span = max(
        num_frames,
        int(
            round(
                n * 0.85
            )
        ),
    )

    span = min(
        span,
        n,
    )

    max_start = max(
        0,
        n - span,
    )

    start = int(
        round(
            max_start
            * position
        )
    )

    end = (
        start
        + span
        - 1
    )


    ids = np.linspace(
        start,
        end,
        num_frames,
    ).round().astype(int)


    return [
        frames[i]
        for i in ids
    ]


# ==================================================================================================
# 12. SIGN-SAFE AUGMENTATION
# ==================================================================================================

def augment_frames(
    frames,
):

    # ----------------------------------------------------------------------
    # NO HORIZONTAL FLIP.
    # NO TEMPORAL REVERSAL.
    #
    # Mild augmentation only.
    # ----------------------------------------------------------------------

    brightness = random.uniform(
        0.85,
        1.15,
    )

    contrast = random.uniform(
        0.85,
        1.15,
    )

    gamma = random.uniform(
        0.90,
        1.10,
    )


    # Mild spatial scale/crop.
    crop_ratio = random.uniform(
        0.92,
        1.00,
    )


    output = []

    for frame in frames:

        x = frame.copy()

        h, w = x.shape[:2]


        # ------------------------------------------------------------------
        # Mild random crop.
        # ------------------------------------------------------------------

        crop_h = max(
            2,
            int(
                h
                * crop_ratio
            ),
        )

        crop_w = max(
            2,
            int(
                w
                * crop_ratio
            ),
        )


        max_y = max(
            0,
            h - crop_h,
        )

        max_x = max(
            0,
            w - crop_w,
        )


        y0 = (
            random.randint(
                0,
                max_y,
            )
            if max_y > 0
            else 0
        )

        x0 = (
            random.randint(
                0,
                max_x,
            )
            if max_x > 0
            else 0
        )


        x = x[
            y0:y0 + crop_h,
            x0:x0 + crop_w,
        ]


        x = cv2.resize(
            x,
            (w, h),
            interpolation=cv2.INTER_LINEAR,
        )


        # ------------------------------------------------------------------
        # Brightness / contrast.
        # ------------------------------------------------------------------

        x = x.astype(
            np.float32
        )


        mean = x.mean(
            axis=(0, 1),
            keepdims=True,
        )


        x = (
            (x - mean)
            * contrast
            + mean
        )


        x *= brightness


        x = np.clip(
            x,
            0,
            255,
        )


        # ------------------------------------------------------------------
        # Gamma.
        # ------------------------------------------------------------------

        x = (
            255.0
            * np.power(
                x / 255.0,
                gamma,
            )
        )


        # ------------------------------------------------------------------
        # Occasional mild blur.
        # ------------------------------------------------------------------

        if random.random() < 0.10:

            x = cv2.GaussianBlur(
                x,
                (3, 3),
                0,
            )


        x = np.clip(
            x,
            0,
            255,
        ).astype(
            np.uint8
        )


        output.append(x)


    return output


# ==================================================================================================
# 13. DATASET
# ==================================================================================================

class TrainDataset(Dataset):

    def __init__(
        self,
        df,
        processor,
    ):

        self.df = (
            df
            .reset_index(
                drop=True
            )
        )

        self.processor = processor


    def __len__(self):

        return len(
            self.df
        )


    def __getitem__(
        self,
        idx,
    ):

        row = self.df.iloc[
            idx
        ]

        frames = read_video(
            row[
                "video_path"
            ]
        )

        frames = (
            temporal_sample_train(
                frames
            )
        )

        frames = augment_frames(
            frames
        )


        encoded = self.processor(
            frames,
            return_tensors="pt",
        )


        return {
            "pixel_values":
                encoded[
                    "pixel_values"
                ].squeeze(0),

            "label":
                torch.tensor(
                    int(
                        row[
                            "label"
                        ]
                    ),
                    dtype=torch.long,
                ),

            "uid":
                row["uid"],
        }


class ValDataset(Dataset):

    def __init__(
        self,
        df,
        processor,
    ):

        self.df = (
            df
            .reset_index(
                drop=True
            )
        )

        self.processor = processor


    def __len__(self):

        return len(
            self.df
        )


    def __getitem__(
        self,
        idx,
    ):

        row = self.df.iloc[
            idx
        ]

        frames = read_video(
            row[
                "video_path"
            ]
        )


        views = []

        for position in (
            VAL_TEMPORAL_VIEWS
        ):

            clip = temporal_sample_view(
                frames,
                position,
            )

            encoded = self.processor(
                clip,
                return_tensors="pt",
            )

            views.append(
                encoded[
                    "pixel_values"
                ].squeeze(0)
            )


        # Shape:
        #
        # [views, frames, channels, H, W]

        views = torch.stack(
            views,
            dim=0,
        )


        return {
            "pixel_values":
                views,

            "label":
                torch.tensor(
                    int(
                        row[
                            "label"
                        ]
                    ),
                    dtype=torch.long,
                ),

            "uid":
                row["uid"],
        }


train_dataset = TrainDataset(
    train_df,
    processor,
)

val_dataset = ValDataset(
    val_df,
    processor,
)


# ==================================================================================================
# 14. BALANCED SAMPLER
# ==================================================================================================

class_counts = Counter(
    train_df[
        "label"
    ].tolist()
)


sample_weights = torch.DoubleTensor(
    [
        1.0
        / class_counts[
            int(label)
        ]

        for label
        in train_df[
            "label"
        ]
    ]
)


sampler_generator = (
    torch.Generator()
)

sampler_generator.manual_seed(
    SEED
)


train_sampler = (
    WeightedRandomSampler(
        sample_weights,
        num_samples=len(
            train_df
        ),
        replacement=True,
        generator=sampler_generator,
    )
)


# ==================================================================================================
# 15. DATALOADERS
# ==================================================================================================

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=train_sampler,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=2,
)


val_loader = DataLoader(
    val_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=2,
)


section(
    "15. DATALOADER CHECK"
)

sample = next(
    iter(
        train_loader
    )
)

print(
    "Train shape:",
    tuple(
        sample[
            "pixel_values"
        ].shape
    ),
)

val_sample = next(
    iter(
        val_loader
    )
)

print(
    "Validation shape:",
    tuple(
        val_sample[
            "pixel_values"
        ].shape
    ),
)

del sample
del val_sample

gc.collect()


# ==================================================================================================
# 16. ORIGINAL VIDEOMAE
# ==================================================================================================

section(
    "16. LOADING ORIGINAL VIDEOMAE"
)

backbone = (
    VideoMAEModel
    .from_pretrained(
        MODEL_NAME
    )
)

print(
    "[PASS] Original VideoMAE loaded."
)

print(
    "[PASS] L20C NOT loaded."
)

print(
    "[PASS] B0 NOT loaded."
)


# ==================================================================================================
# 17. COSINE CLASSIFIER
# ==================================================================================================

class CosineClassifier(nn.Module):

    def __init__(
        self,
        in_features,
        num_classes,
        initial_scale=10.0,
    ):

        super().__init__()


        self.weight = nn.Parameter(
            torch.empty(
                num_classes,
                in_features,
            )
        )


        nn.init.normal_(
            self.weight,
            std=0.02,
        )


        self.log_scale = (
            nn.Parameter(
                torch.tensor(
                    math.log(
                        initial_scale
                    ),
                    dtype=torch.float32,
                )
            )
        )


    def forward(
        self,
        features,
    ):

        x = F.normalize(
            features,
            p=2,
            dim=-1,
        )

        w = F.normalize(
            self.weight,
            p=2,
            dim=-1,
        )


        # Keep scale in a stable range.

        scale = (
            self.log_scale
            .exp()
            .clamp(
                1.0,
                30.0,
            )
        )


        return (
            scale
            * F.linear(
                x,
                w,
            )
        )


class B1VideoMAE(nn.Module):

    def __init__(
        self,
        backbone,
        num_classes,
    ):

        super().__init__()


        self.backbone = backbone

        hidden = (
            backbone
            .config
            .hidden_size
        )


        self.feature_norm = (
            nn.LayerNorm(
                hidden
            )
        )


        self.dropout = (
            nn.Dropout(
                0.35
            )
        )


        self.classifier = (
            CosineClassifier(
                hidden,
                num_classes,
                initial_scale=10.0,
            )
        )


    def extract_features(
        self,
        pixel_values,
    ):

        outputs = self.backbone(
            pixel_values=pixel_values
        )


        features = (
            outputs
            .last_hidden_state
            .mean(dim=1)
        )


        features = (
            self.feature_norm(
                features
            )
        )


        return features


    def forward(
        self,
        pixel_values,
    ):

        features = (
            self.extract_features(
                pixel_values
            )
        )


        logits = (
            self.classifier(
                self.dropout(
                    features
                )
            )
        )


        return (
            logits,
            features,
        )


model = B1VideoMAE(
    backbone,
    NUM_CLASSES,
).to(
    DEVICE
)


# ==================================================================================================
# 18. LOSS
# ==================================================================================================

criterion = nn.CrossEntropyLoss(
    label_smoothing=LABEL_SMOOTHING
)


# ==================================================================================================
# 19. FREEZING
# ==================================================================================================

def freeze_all_backbone(
    model,
):

    for p in (
        model
        .backbone
        .parameters()
    ):

        p.requires_grad = False


def configure_stage(
    model,
    n_blocks,
):

    freeze_all_backbone(
        model
    )


    layers = (
        model
        .backbone
        .encoder
        .layer
    )


    if n_blocks > 0:

        for layer in (
            layers[
                -n_blocks:
            ]
        ):

            for p in (
                layer
                .parameters()
            ):

                p.requires_grad = True


    # Some HF VideoMAE versions expose layernorm=None.
    #
    # Do not call .parameters() unless it exists.

    backbone_ln = getattr(
        model.backbone,
        "layernorm",
        None,
    )


    if (
        backbone_ln
        is not None
        and
        n_blocks > 0
    ):

        for p in (
            backbone_ln
            .parameters()
        ):

            p.requires_grad = True


    # B1 head always trainable.

    for p in (
        model
        .feature_norm
        .parameters()
    ):

        p.requires_grad = True


    for p in (
        model
        .classifier
        .parameters()
    ):

        p.requires_grad = True


def parameter_count(
    model,
):

    total = sum(
        p.numel()
        for p in model.parameters()
    )

    trainable = sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )

    return (
        total,
        trainable,
    )


# ==================================================================================================
# 20. TOP-K
# ==================================================================================================

def topk_correct(
    logits,
    labels,
    k,
):

    ids = logits.topk(
        k,
        dim=1,
    ).indices


    return int(
        (
            ids
            == labels.unsqueeze(1)
        )
        .any(dim=1)
        .sum()
        .item()
    )


# ==================================================================================================
# 21. MULTI-VIEW VALIDATION
# ==================================================================================================

@torch.no_grad()
def evaluate_multiview(
    model,
    loader,
):

    model.eval()


    total = 0

    total_loss = 0.0

    correct1 = 0
    correct5 = 0

    predictions = []


    for batch in loader:

        # batch shape:
        #
        # [B=1, V=3, T, C, H, W]

        views = batch[
            "pixel_values"
        ].squeeze(0)


        label = batch[
            "label"
        ].to(
            DEVICE
        )


        view_logits = []


        for v in range(
            views.shape[0]
        ):

            x = (
                views[v]
                .unsqueeze(0)
                .to(
                    DEVICE,
                    non_blocking=True,
                )
            )


            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=(
                    DEVICE.type
                    == "cuda"
                ),
            ):

                logits, _ = model(
                    x
                )


            view_logits.append(
                logits.float()
            )


        # Average LOGITS, not hard predictions.

        logits = torch.stack(
            view_logits,
            dim=0,
        ).mean(
            dim=0
        )


        loss = criterion(
            logits,
            label,
        )


        total += 1

        total_loss += (
            loss.item()
        )


        correct1 += topk_correct(
            logits,
            label,
            1,
        )

        correct5 += topk_correct(
            logits,
            label,
            5,
        )


        probs = torch.softmax(
            logits,
            dim=1,
        )


        top_prob, top_id = (
            probs.max(
                dim=1
            )
        )


        true_id = int(
            label.item()
        )

        pred_id = int(
            top_id.item()
        )


        predictions.append(
            {
                "uid":
                    batch[
                        "uid"
                    ][0],

                "true_id":
                    true_id,

                "true_gloss":
                    id_to_gloss[
                        true_id
                    ],

                "pred_id":
                    pred_id,

                "pred_gloss":
                    id_to_gloss[
                        pred_id
                    ],

                "confidence":
                    float(
                        top_prob.item()
                    ),

                "correct":
                    int(
                        pred_id
                        == true_id
                    ),
            }
        )


    return {
        "loss":
            total_loss
            / total,

        "top1":
            100.0
            * correct1
            / total,

        "top5":
            100.0
            * correct5
            / total,

        "correct1":
            correct1,

        "correct5":
            correct5,

        "total":
            total,

        "predictions":
            predictions,
    }


# ==================================================================================================
# 22. OPTIMIZER
# ==================================================================================================

def build_optimizer(
    model,
    head_lr,
    backbone_lr,
):

    head_params = []

    backbone_params = []


    for name, p in (
        model.named_parameters()
    ):

        if not p.requires_grad:

            continue


        if name.startswith(
            "backbone."
        ):

            backbone_params.append(
                p
            )

        else:

            head_params.append(
                p
            )


    groups = []


    if head_params:

        groups.append(
            {
                "params":
                    head_params,

                "lr":
                    head_lr,

                "base_lr":
                    head_lr,
            }
        )


    if backbone_params:

        groups.append(
            {
                "params":
                    backbone_params,

                "lr":
                    backbone_lr,

                "base_lr":
                    backbone_lr,
            }
        )


    return torch.optim.AdamW(
        groups,
        weight_decay=WEIGHT_DECAY,
        betas=(
            0.9,
            0.999,
        ),
    )


# ==================================================================================================
# 23. WARMUP + COSINE SCHEDULE
# ==================================================================================================

def set_lr(
    optimizer,
    update_idx,
    total_updates,
):

    warmup_updates = max(
        1,
        int(
            total_updates
            * WARMUP_RATIO
        ),
    )


    if (
        update_idx
        < warmup_updates
    ):

        factor = (
            update_idx + 1
        ) / warmup_updates

    else:

        progress = (
            update_idx
            - warmup_updates
        ) / max(
            1,
            total_updates
            - warmup_updates,
        )


        factor = (
            0.05
            + 0.95
            * 0.5
            * (
                1.0
                + math.cos(
                    math.pi
                    * progress
                )
            )
        )


    for group in (
        optimizer
        .param_groups
    ):

        group["lr"] = (
            group[
                "base_lr"
            ]
            * factor
        )


# ==================================================================================================
# 24. TRAIN EPOCH
# ==================================================================================================

def train_epoch(
    model,
    loader,
    optimizer,
    scaler,
    epoch,
    stage,
    stage_epochs,
):

    model.train()


    optimizer.zero_grad(
        set_to_none=True
    )


    total = 0

    loss_sum = 0.0

    correct1 = 0
    correct5 = 0


    updates_per_epoch = (
        math.ceil(
            len(loader)
            / GRAD_ACCUM
        )
    )


    total_updates = (
        updates_per_epoch
        * stage_epochs
    )


    base_update = (
        (epoch - 1)
        * updates_per_epoch
    )


    current_update = 0


    start = time.time()


    for step, batch in enumerate(
        loader,
        start=1,
    ):

        x = batch[
            "pixel_values"
        ].to(
            DEVICE,
            non_blocking=True,
        )

        y = batch[
            "label"
        ].to(
            DEVICE,
            non_blocking=True,
        )


        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(
                DEVICE.type
                == "cuda"
            ),
        ):

            logits, _ = model(
                x
            )


            raw_loss = criterion(
                logits,
                y,
            )


            loss = (
                raw_loss
                / GRAD_ACCUM
            )


        scaler.scale(
            loss
        ).backward()


        should_update = (
            step % GRAD_ACCUM
            == 0
            or
            step == len(loader)
        )


        if should_update:

            update_number = (
                base_update
                + current_update
            )


            set_lr(
                optimizer,
                update_number,
                total_updates,
            )


            scaler.unscale_(
                optimizer
            )


            torch.nn.utils.clip_grad_norm_(
                [
                    p
                    for p
                    in model.parameters()
                    if p.requires_grad
                ],
                GRAD_CLIP,
            )


            scaler.step(
                optimizer
            )


            scaler.update()


            optimizer.zero_grad(
                set_to_none=True
            )


            current_update += 1


        bs = y.size(0)

        total += bs


        loss_sum += (
            raw_loss.item()
            * bs
        )


        correct1 += topk_correct(
            logits,
            y,
            1,
        )

        correct5 += topk_correct(
            logits,
            y,
            5,
        )


        if (
            step == 1
            or
            step % LOG_EVERY == 0
            or
            step == len(loader)
        ):

            elapsed = (
                time.time()
                - start
            )


            print(
                f"[{stage}] "
                f"E{epoch:02d} "
                f"{step:04d}/{len(loader)} | "
                f"loss={loss_sum/total:.4f} | "
                f"T1={100*correct1/total:.2f}% | "
                f"T5={100*correct5/total:.2f}% | "
                f"{total/max(elapsed,1e-9):.2f} vid/s"
            )


    return {
        "loss":
            loss_sum / total,

        "top1":
            100.0
            * correct1
            / total,

        "top5":
            100.0
            * correct5
            / total,
    }


# ==================================================================================================
# 25. CHECKPOINTS
# ==================================================================================================

BEST_MODEL = (
    MODEL_DIR
    / "l20d_b1_best.pt"
)

HISTORY_FILE = (
    REPORT_DIR
    / "l20d_b1_history.csv"
)

BEST_PRED_FILE = (
    REPORT_DIR
    / "l20d_b1_best_validation_predictions.csv"
)

FINAL_MODEL = (
    MODEL_DIR
    / "l20d_b1_final.pt"
)

MASTER_FREEZE = (
    OUTPUT_ROOT
    / "L20D_B1_MASTER_FREEZE.json"
)


history = []

global_epoch = 0

best_top1 = -1.0
best_top5 = -1.0
best_loss = float(
    "inf"
)

best_epoch = None
best_stage = None


def is_better(
    val,
):

    global best_top1
    global best_top5
    global best_loss


    if (
        val["top1"]
        > best_top1
    ):

        return True


    if (
        val["top1"]
        == best_top1
        and
        val["top5"]
        > best_top5
    ):

        return True


    if (
        val["top1"]
        == best_top1
        and
        val["top5"]
        == best_top5
        and
        val["loss"]
        < best_loss
    ):

        return True


    return False


# ==================================================================================================
# 26. STAGE TRAINER
# ==================================================================================================

def run_stage(
    stage_name,
    epochs,
    blocks,
    head_lr,
    backbone_lr,
):

    global global_epoch
    global best_top1
    global best_top5
    global best_loss
    global best_epoch
    global best_stage


    configure_stage(
        model,
        blocks,
    )


    total_params, trainable_params = (
        parameter_count(
            model
        )
    )


    section(
        f"STAGE {stage_name}"
    )


    print(
        f"Trainable blocks       : "
        f"{blocks}"
    )

    print(
        f"Head LR                : "
        f"{head_lr}"
    )

    print(
        f"Backbone LR            : "
        f"{backbone_lr}"
    )

    print(
        f"Total parameters       : "
        f"{total_params:,}"
    )

    print(
        f"Trainable parameters   : "
        f"{trainable_params:,}"
    )


    optimizer = build_optimizer(
        model,
        head_lr,
        backbone_lr,
    )


    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=(
            DEVICE.type
            == "cuda"
        ),
    )


    no_improve = 0


    for local_epoch in range(
        1,
        epochs + 1,
    ):

        global_epoch += 1


        section(
            f"{stage_name} "
            f"{local_epoch}/{epochs} "
            f"| GLOBAL {global_epoch}"
        )


        train_result = train_epoch(
            model,
            train_loader,
            optimizer,
            scaler,
            local_epoch,
            stage_name,
            epochs,
        )


        val_result = (
            evaluate_multiview(
                model,
                val_loader,
            )
        )


        print("\nTRAIN")

        print(
            f"Loss : "
            f"{train_result['loss']:.6f}"
        )

        print(
            f"Top1 : "
            f"{train_result['top1']:.3f}%"
        )

        print(
            f"Top5 : "
            f"{train_result['top5']:.3f}%"
        )


        print("\nVALIDATION — 3 TEMPORAL VIEWS")

        print(
            f"Loss : "
            f"{val_result['loss']:.6f}"
        )

        print(
            f"Top1 : "
            f"{val_result['top1']:.3f}% "
            f"({val_result['correct1']}/304)"
        )

        print(
            f"Top5 : "
            f"{val_result['top5']:.3f}% "
            f"({val_result['correct5']}/304)"
        )


        history.append(
            {
                "global_epoch":
                    global_epoch,

                "stage":
                    stage_name,

                "local_epoch":
                    local_epoch,

                "train_loss":
                    train_result[
                        "loss"
                    ],

                "train_top1":
                    train_result[
                        "top1"
                    ],

                "train_top5":
                    train_result[
                        "top5"
                    ],

                "val_loss":
                    val_result[
                        "loss"
                    ],

                "val_top1":
                    val_result[
                        "top1"
                    ],

                "val_top5":
                    val_result[
                        "top5"
                    ],
            }
        )


        pd.DataFrame(
            history
        ).to_csv(
            HISTORY_FILE,
            index=False,
        )


        if is_better(
            val_result
        ):

            best_top1 = (
                val_result[
                    "top1"
                ]
            )

            best_top5 = (
                val_result[
                    "top5"
                ]
            )

            best_loss = (
                val_result[
                    "loss"
                ]
            )

            best_epoch = (
                global_epoch
            )

            best_stage = (
                stage_name
            )


            no_improve = 0


            torch.save(
                {
                    "model_state_dict":
                        model.state_dict(),

                    "global_epoch":
                        global_epoch,

                    "stage":
                        stage_name,

                    "best_top1":
                        best_top1,

                    "best_top5":
                        best_top5,

                    "best_loss":
                        best_loss,

                    "model_name":
                        MODEL_NAME,

                    "num_classes":
                        NUM_CLASSES,

                    "gloss_to_id":
                        gloss_to_id,

                    "l20c_loaded":
                        False,

                    "b0_loaded":
                        False,

                    "locked_test_used":
                        False,
                },
                BEST_MODEL,
            )


            pd.DataFrame(
                val_result[
                    "predictions"
                ]
            ).to_csv(
                BEST_PRED_FILE,
                index=False,
            )


            print(
                "\n[SAVED BEST]"
            )

            print(
                f"Global epoch : "
                f"{best_epoch}"
            )

            print(
                f"Stage        : "
                f"{best_stage}"
            )

            print(
                f"Val Top1     : "
                f"{best_top1:.3f}%"
            )

            print(
                f"Val Top5     : "
                f"{best_top5:.3f}%"
            )


        else:

            no_improve += 1


            print(
                f"\n[NO IMPROVEMENT] "
                f"{no_improve}/"
                f"{EARLY_STOP_PATIENCE}"
            )


        if (
            no_improve
            >= EARLY_STOP_PATIENCE
        ):

            print(
                f"\n[EARLY STOP] "
                f"{stage_name}"
            )

            break


# ==================================================================================================
# 27. STAGE A — HEAD ONLY
# ==================================================================================================

section(
    "27. B1 TRAINING START"
)

print(
    "[LOCK] L20C not loaded."
)

print(
    "[LOCK] B0 not loaded."
)

print(
    "[LOCK] Locked test not decoded."
)


run_stage(
    stage_name="HEAD",
    epochs=HEAD_EPOCHS,
    blocks=0,
    head_lr=HEAD_LR,
    backbone_lr=0.0,
)


# ==================================================================================================
# 28. RESTORE BEST BEFORE STAGE B
# ==================================================================================================

section(
    "28. RESTORE BEST BEFORE FINAL-2"
)

checkpoint = torch.load(
    BEST_MODEL,
    map_location="cpu",
    weights_only=False,
)

model.load_state_dict(
    checkpoint[
        "model_state_dict"
    ]
)

model.to(
    DEVICE
)

print(
    f"Restored epoch "
    f"{checkpoint['global_epoch']}"
)

print(
    f"Val Top1 "
    f"{checkpoint['best_top1']:.3f}%"
)


# ==================================================================================================
# 29. STAGE B — FINAL 2 BLOCKS
# ==================================================================================================

run_stage(
    stage_name="FINAL2",
    epochs=STAGE2_EPOCHS,
    blocks=STAGE2_BLOCKS,
    head_lr=STAGE2_HEAD_LR,
    backbone_lr=STAGE2_BACKBONE_LR,
)


# ==================================================================================================
# 30. RESTORE BEST BEFORE FINAL-6
# ==================================================================================================

section(
    "30. RESTORE BEST BEFORE FINAL-6"
)

checkpoint = torch.load(
    BEST_MODEL,
    map_location="cpu",
    weights_only=False,
)

model.load_state_dict(
    checkpoint[
        "model_state_dict"
    ]
)

model.to(
    DEVICE
)

print(
    f"Restored best epoch "
    f"{checkpoint['global_epoch']}"
)

print(
    f"Best validation Top1 "
    f"{checkpoint['best_top1']:.3f}%"
)


# ==================================================================================================
# 31. STAGE C — FINAL 6 BLOCKS
# ==================================================================================================

run_stage(
    stage_name="FINAL6",
    epochs=STAGE3_EPOCHS,
    blocks=STAGE3_BLOCKS,
    head_lr=STAGE3_HEAD_LR,
    backbone_lr=STAGE3_BACKBONE_LR,
)


# ==================================================================================================
# 32. LOAD ABSOLUTE BEST
# ==================================================================================================

section(
    "32. LOADING ABSOLUTE BEST B1"
)

checkpoint = torch.load(
    BEST_MODEL,
    map_location="cpu",
    weights_only=False,
)

model.load_state_dict(
    checkpoint[
        "model_state_dict"
    ]
)

model.to(
    DEVICE
)


print(
    f"Best epoch : "
    f"{checkpoint['global_epoch']}"
)

print(
    f"Best stage : "
    f"{checkpoint['stage']}"
)

print(
    f"Best Top1  : "
    f"{checkpoint['best_top1']:.3f}%"
)

print(
    f"Best Top5  : "
    f"{checkpoint['best_top5']:.3f}%"
)


# ==================================================================================================
# 33. FINAL VALIDATION
# ==================================================================================================

section(
    "33. FINAL 3-VIEW VALIDATION"
)

final_val = evaluate_multiview(
    model,
    val_loader,
)


print(
    f"Top1 : "
    f"{final_val['top1']:.3f}% "
    f"({final_val['correct1']}/304)"
)

print(
    f"Top5 : "
    f"{final_val['top5']:.3f}% "
    f"({final_val['correct5']}/304)"
)

print(
    f"Loss : "
    f"{final_val['loss']:.6f}"
)


# ==================================================================================================
# 34. SAVE FINAL MODEL
# ==================================================================================================

torch.save(
    {
        "model_state_dict":
            model.state_dict(),

        "experiment":
            "L20D_B1",

        "model_name":
            MODEL_NAME,

        "num_classes":
            NUM_CLASSES,

        "num_frames":
            NUM_FRAMES,

        "best_epoch":
            checkpoint[
                "global_epoch"
            ],

        "best_stage":
            checkpoint[
                "stage"
            ],

        "validation_top1":
            final_val[
                "top1"
            ],

        "validation_top5":
            final_val[
                "top5"
            ],

        "validation_loss":
            final_val[
                "loss"
            ],

        "gloss_to_id":
            gloss_to_id,

        "id_to_gloss":
            id_to_gloss,

        "l20c_loaded":
            False,

        "b0_loaded":
            False,

        "locked_test_used":
            False,
    },
    FINAL_MODEL,
)


# ==================================================================================================
# 35. FREEZE
# ==================================================================================================

freeze = {
    "experiment":
        "L20D_B1_LOW_SHOT",

    "status":
        "COMPLETE_AND_FROZEN",

    "protocol": {
        "train":
            710,

        "validation":
            304,

        "classes":
            215,

        "locked_test":
            10,
    },

    "initialization":
        MODEL_NAME,

    "training": {
        "head_epochs":
            HEAD_EPOCHS,

        "final2_epochs":
            STAGE2_EPOCHS,

        "final6_epochs":
            STAGE3_EPOCHS,

        "label_smoothing":
            LABEL_SMOOTHING,

        "validation_temporal_views":
            VAL_TEMPORAL_VIEWS,
    },

    "result": {
        "best_epoch":
            int(
                checkpoint[
                    "global_epoch"
                ]
            ),

        "best_stage":
            checkpoint[
                "stage"
            ],

        "validation_top1":
            final_val[
                "top1"
            ],

        "validation_top5":
            final_val[
                "top5"
            ],

        "validation_loss":
            final_val[
                "loss"
            ],
    },

    "integrity": {
        "l20c_loaded":
            False,

        "b0_loaded":
            False,

        "locked_test_used":
            False,

        "train_val_overlap":
            0,

        "train_locked_overlap":
            0,

        "val_locked_overlap":
            0,
    },

    "artifacts": {
        "best_model":
            str(
                BEST_MODEL
            ),

        "final_model":
            str(
                FINAL_MODEL
            ),

        "history":
            str(
                HISTORY_FILE
            ),

        "best_predictions":
            str(
                BEST_PRED_FILE
            ),
    },
}


with open(
    MASTER_FREEZE,
    "w",
) as f:

    json.dump(
        freeze,
        f,
        indent=2,
    )


# ==================================================================================================
# 36. HASHES
# ==================================================================================================

section(
    "36. ARTIFACT HASHES"
)

for name, path in {
    "best_model":
        BEST_MODEL,

    "final_model":
        FINAL_MODEL,

    "history":
        HISTORY_FILE,

    "predictions":
        BEST_PRED_FILE,

    "master_freeze":
        MASTER_FREEZE,
}.items():

    if Path(path).exists():

        print(
            f"{name:20s}: "
            f"{sha256_file(path)}"
        )


# ==================================================================================================
# 37. FINAL
# ==================================================================================================

section(
    "L20D-B1 COMPLETE"
)

print(
    f"Best stage      : "
    f"{checkpoint['stage']}"
)

print(
    f"Best epoch      : "
    f"{checkpoint['global_epoch']}"
)

print(
    f"Validation Top1 : "
    f"{final_val['top1']:.3f}%"
)

print(
    f"Validation Top5 : "
    f"{final_val['top5']:.3f}%"
)

print(
    f"Validation loss : "
    f"{final_val['loss']:.6f}"
)

print(
    "\nLocked test used : NO"
)

print(
    "L20C loaded      : NO"
)

print(
    "B0 loaded        : NO"
)

print(
    "\nBest model:"
)

print(
    BEST_MODEL
)

print(
    "\nMaster freeze:"
)

print(
    MASTER_FREEZE
)

print(
    "\n[LOCK] Do NOT evaluate the permanent 10-video test yet."
)

print("=" * 120)


CISLR — L20D-B1
LOW-SHOT 215-CLASS VIDEOMAE
PyTorch                 : 2.14.0+cu130
OpenCV                  : 5.0.0
Device                  : cuda
GPU                     : NVIDIA GeForce RTX 4050 Laptop GPU
GPU memory              : 5.64 GB
Disk free               : 15.50 GB
Classes                 : 215
Frames                  : 16
Effective batch         : 8
Initialization          : ORIGINAL VideoMAE
L20C loaded             : NO
B0 loaded               : NO
Locked test evaluation  : DISABLED

4. VERIFYING CORRECTED PROTOCOL
[FOUND] /home/dipshikha/Music/cao/CISLR_RESEARCH/CISLR_LANDMARK_MODEL/audit/l20b_corrected_strict_protocol_freeze/manifests/l20b_corrected_train_manifest.csv
[FOUND] /home/dipshikha/Music/cao/CISLR_RESEARCH/CISLR_LANDMARK_MODEL/audit/l20b_corrected_strict_protocol_freeze/manifests/l20b_corrected_validation_manifest.csv
[FOUND] /home/dipshikha/Music/cao/CISLR_RESEARCH/CISLR_LANDMARK_MODEL/audit/l20b_corrected_strict_protocol_freeze/manifests/l20b_permanent_l19_lo

In [1]:
# ==================================================================================================
# CISLR — L20D-B2A
# 215-CLASS LANDMARK BRANCH DATASET AUDIT
#
# PURPOSE
# -------
# Build the landmark side of B2 using EXACTLY the corrected L20B:
#   TRAIN      = 710
#   VALIDATION = 304
#   LOCKED     = 10  (NEVER LOADED FOR TRAINING/EVALUATION)
#
# Expected landmark representation:
#   32 frames x 254 features
#
# IMPORTANT
# ---------
# - Does NOT load L20C
# - Does NOT retrain B1
# - Does NOT evaluate locked test
# - Does NOT modify L20B/B1
# - Only discovers and verifies landmark features
# ==================================================================================================

from pathlib import Path
import pandas as pd
import numpy as np
import json
import hashlib
import os
import sys
from collections import Counter, defaultdict

# ==================================================================================================
# 1. PATHS
# ==================================================================================================

PROJECT_ROOT = Path(
    "/home/dipshikha/Music/cao/CISLR_RESEARCH"
)

LANDMARK_ROOT = (
    PROJECT_ROOT /
    "CISLR_LANDMARK_MODEL"
)

L20B_ROOT = (
    LANDMARK_ROOT /
    "audit" /
    "l20b_corrected_strict_protocol_freeze"
)

MANIFEST_ROOT = L20B_ROOT / "manifests"

TRAIN_MANIFEST = (
    MANIFEST_ROOT /
    "l20b_corrected_train_manifest.csv"
)

VAL_MANIFEST = (
    MANIFEST_ROOT /
    "l20b_corrected_validation_manifest.csv"
)

LOCKED_MANIFEST = (
    MANIFEST_ROOT /
    "l20b_permanent_l19_locked_test_manifest.csv"
)

CLASS_MAPPING = (
    MANIFEST_ROOT /
    "l20b_corrected_class_mapping.csv"
)

B1_ROOT = (
    LANDMARK_ROOT /
    "audit" /
    "l20d_b1_lowshot_videomae_215class"
)

B1_MODEL = (
    B1_ROOT /
    "models" /
    "l20d_b1_best.pt"
)

B2_ROOT = (
    LANDMARK_ROOT /
    "audit" /
    "l20d_b2_multimodal_215class"
)

B2A_ROOT = B2_ROOT / "b2a_landmark_audit"

B2A_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

# ==================================================================================================
# 2. HEADER
# ==================================================================================================

print("=" * 120)
print("CISLR — L20D-B2A")
print("215-CLASS LANDMARK BRANCH DATASET AUDIT")
print("=" * 120)

print("Python          :", sys.version.split()[0])
print("Project root    :", PROJECT_ROOT)
print("Landmark root   :", LANDMARK_ROOT)
print("B2 root         :", B2_ROOT)
print()

# ==================================================================================================
# 3. VERIFY AUTHORITATIVE INPUT FILES
# ==================================================================================================

print("=" * 120)
print("3. VERIFYING AUTHORITATIVE INPUTS")
print("=" * 120)

required = {
    "train_manifest": TRAIN_MANIFEST,
    "val_manifest": VAL_MANIFEST,
    "locked_manifest": LOCKED_MANIFEST,
    "class_mapping": CLASS_MAPPING,
    "b1_best_model": B1_MODEL,
}

for name, path in required.items():

    if not path.exists():
        raise FileNotFoundError(
            f"[MISSING] {name}: {path}"
        )

    print(
        f"[FOUND] {name:<20}: {path}"
    )

print("\n[PASS] Required B1/L20B artifacts found.")

# ==================================================================================================
# 4. LOAD MANIFESTS
# ==================================================================================================

train_df = pd.read_csv(TRAIN_MANIFEST)
val_df = pd.read_csv(VAL_MANIFEST)
locked_df = pd.read_csv(LOCKED_MANIFEST)
mapping_df = pd.read_csv(CLASS_MAPPING)

print()
print("=" * 120)
print("4. MANIFEST SUMMARY")
print("=" * 120)

print("Train rows      :", len(train_df))
print("Validation rows :", len(val_df))
print("Locked rows     :", len(locked_df))
print("Classes         :", len(mapping_df))

print()
print("Train columns:")
print(list(train_df.columns))

print()
print("Validation columns:")
print(list(val_df.columns))

# ==================================================================================================
# 5. IDENTIFY UID / CLASS COLUMNS
# ==================================================================================================

def find_column(df, candidates):

    lower_map = {
        str(c).lower(): c
        for c in df.columns
    }

    for candidate in candidates:

        if candidate.lower() in lower_map:
            return lower_map[candidate.lower()]

    return None


UID_CANDIDATES = [
    "uid",
    "video_uid",
    "video_id",
    "id",
    "name",
]

CLASS_CANDIDATES = [
    "class_id",
    "label",
    "target",
]

GLOSS_CANDIDATES = [
    "gloss",
    "word",
    "sign",
]


uid_col = find_column(
    train_df,
    UID_CANDIDATES
)

class_col = find_column(
    train_df,
    CLASS_CANDIDATES
)

gloss_col = find_column(
    train_df,
    GLOSS_CANDIDATES
)


if uid_col is None:
    raise RuntimeError(
        f"Could not identify UID column. "
        f"Columns={list(train_df.columns)}"
    )

if class_col is None:
    raise RuntimeError(
        f"Could not identify class column. "
        f"Columns={list(train_df.columns)}"
    )


print()
print("=" * 120)
print("5. COLUMN DETECTION")
print("=" * 120)

print("UID column   :", uid_col)
print("Class column :", class_col)
print("Gloss column :", gloss_col)

# ==================================================================================================
# 6. STRICT PROTOCOL CHECK
# ==================================================================================================

train_uids = set(
    train_df[uid_col].astype(str)
)

val_uids = set(
    val_df[uid_col].astype(str)
)

locked_uids = set(
    locked_df[uid_col].astype(str)
)

checks = {
    "train_710":
        len(train_df) == 710,

    "validation_304":
        len(val_df) == 304,

    "locked_10":
        len(locked_df) == 10,

    "train_unique_710":
        len(train_uids) == 710,

    "validation_unique_304":
        len(val_uids) == 304,

    "locked_unique_10":
        len(locked_uids) == 10,

    "train_val_overlap_zero":
        len(train_uids & val_uids) == 0,

    "train_locked_overlap_zero":
        len(train_uids & locked_uids) == 0,

    "val_locked_overlap_zero":
        len(val_uids & locked_uids) == 0,

    "train_classes_215":
        train_df[class_col].nunique() == 215,

    "validation_classes_215":
        val_df[class_col].nunique() == 215,
}


print()
print("=" * 120)
print("6. STRICT PROTOCOL INTEGRITY")
print("=" * 120)

for name, ok in checks.items():

    print(
        f"{name:<45}: "
        f"{'PASS' if ok else 'FAIL'}"
    )


if not all(checks.values()):

    raise RuntimeError(
        "Strict protocol integrity check failed."
    )


print(
    "\n[PASS] Corrected 710 / 304 / 10 protocol preserved."
)

# ==================================================================================================
# 7. SEARCH FOR EXISTING LANDMARK FEATURE FILES
# ==================================================================================================

print()
print("=" * 120)
print("7. SEARCHING EXISTING LANDMARK FEATURES")
print("=" * 120)

# Search only inside the project.
#
# We intentionally do NOT assume the exact L6 folder name.
# Existing project versions may have stored features under different directories.

SEARCH_EXTENSIONS = {
    ".npy",
    ".npz",
}


feature_files = []

skip_dirs = {
    "l20d_b2_multimodal_215class",
}


for root, dirs, files in os.walk(LANDMARK_ROOT):

    root_path = Path(root)

    # prevent B2 output from indexing itself
    dirs[:] = [
        d for d in dirs
        if d not in skip_dirs
    ]

    for filename in files:

        p = root_path / filename

        if p.suffix.lower() in SEARCH_EXTENSIONS:
            feature_files.append(p)


print(
    "Candidate .npy/.npz files found:",
    len(feature_files)
)

# ==================================================================================================
# 8. BUILD UID -> FEATURE CANDIDATE INDEX
# ==================================================================================================

print()
print("=" * 120)
print("8. BUILDING UID -> LANDMARK FEATURE INDEX")
print("=" * 120)

all_needed_uids = (
    train_uids |
    val_uids
)

# IMPORTANT:
# locked_uids deliberately NOT included here.
#
# We do not need locked-test features to train B2.

uid_to_candidates = defaultdict(list)


# First use filename stems.
for path in feature_files:

    stem = path.stem

    if stem in all_needed_uids:
        uid_to_candidates[stem].append(path)


# Some feature files may have suffixes like:
#
# UID_features.npy
# UID_landmarks.npy
# UID_32x254.npy
#
# Search unresolved UIDs against filename.

unresolved = (
    all_needed_uids -
    set(uid_to_candidates.keys())
)


if unresolved:

    print(
        "Direct stem matches:",
        len(uid_to_candidates)
    )

    print(
        "Trying controlled filename matching for",
        len(unresolved),
        "unresolved UIDs..."
    )

    unresolved_sorted = sorted(unresolved)

    for path in feature_files:

        name = path.name

        for uid in unresolved_sorted:

            if (
                name.startswith(uid + "_") or
                name.startswith(uid + ".") or
                f"/{uid}/" in str(path)
            ):

                uid_to_candidates[uid].append(
                    path
                )


resolved_count = sum(
    1
    for uid in all_needed_uids
    if uid in uid_to_candidates
)

print(
    f"UIDs with at least one candidate: "
    f"{resolved_count}/{len(all_needed_uids)}"
)

# ==================================================================================================
# 9. FEATURE VALIDATOR
# ==================================================================================================

print()
print("=" * 120)
print("9. VALIDATING FEATURE SHAPES")
print("=" * 120)


def load_candidate_array(path):

    try:

        if path.suffix.lower() == ".npy":

            arr = np.load(
                path,
                allow_pickle=False
            )

            return np.asarray(arr)

        if path.suffix.lower() == ".npz":

            data = np.load(
                path,
                allow_pickle=False
            )

            keys = list(data.keys())

            # Prefer common feature keys.
            preferred = [
                "features",
                "feature",
                "landmarks",
                "x",
                "data",
            ]

            for key in preferred:

                if key in data:
                    return np.asarray(
                        data[key]
                    )

            # If only one array exists, use it.
            if len(keys) == 1:

                return np.asarray(
                    data[keys[0]]
                )

            return None

    except Exception:
        return None

    return None


def normalize_feature_shape(arr):

    """
    Accepts a few reasonable existing representations.

    Preferred:
        (32, 254)

    Also allows:
        (1, 32, 254)
    """

    if arr is None:
        return None

    arr = np.asarray(arr)

    if arr.ndim == 3 and arr.shape[0] == 1:
        arr = arr[0]

    if arr.ndim != 2:
        return None

    if arr.shape == (32, 254):
        return arr.astype(
            np.float32,
            copy=False
        )

    return None


valid_uid_to_path = {}

invalid_candidates = []


for i, uid in enumerate(
    sorted(all_needed_uids),
    start=1
):

    candidates = uid_to_candidates.get(
        uid,
        []
    )

    valid_path = None

    for path in candidates:

        arr = load_candidate_array(
            path
        )

        arr = normalize_feature_shape(
            arr
        )

        if arr is not None:

            if np.isfinite(arr).all():

                valid_path = path
                break

        invalid_candidates.append(
            str(path)
        )

    if valid_path is not None:

        valid_uid_to_path[uid] = (
            valid_path
        )

    if (
        i % 100 == 0 or
        i == len(all_needed_uids)
    ):

        print(
            f"Checked {i:4d}/"
            f"{len(all_needed_uids)}"
            f" | valid="
            f"{len(valid_uid_to_path)}"
        )


# ==================================================================================================
# 10. TRAIN / VALIDATION COVERAGE
# ==================================================================================================

train_found = [
    uid
    for uid in train_uids
    if uid in valid_uid_to_path
]

val_found = [
    uid
    for uid in val_uids
    if uid in valid_uid_to_path
]

train_missing = sorted(
    train_uids -
    set(train_found)
)

val_missing = sorted(
    val_uids -
    set(val_found)
)


print()
print("=" * 120)
print("10. LANDMARK COVERAGE")
print("=" * 120)

print(
    f"TRAIN      : "
    f"{len(train_found)}/710"
)

print(
    f"VALIDATION : "
    f"{len(val_found)}/304"
)

print(
    "Locked test: NOT searched / "
    "NOT loaded"
)

print()

if train_missing:

    print(
        "Missing TRAIN features:",
        len(train_missing)
    )

    print(
        "First 20:",
        train_missing[:20]
    )

else:

    print(
        "[PASS] All 710 training "
        "landmark features resolved."
    )


if val_missing:

    print(
        "Missing VALIDATION features:",
        len(val_missing)
    )

    print(
        "First 20:",
        val_missing[:20]
    )

else:

    print(
        "[PASS] All 304 validation "
        "landmark features resolved."
    )


# ==================================================================================================
# 11. CREATE B2 LANDMARK MANIFESTS
# ==================================================================================================

def attach_feature_paths(df):

    out = df.copy()

    out["landmark_feature_path"] = (
        out[uid_col]
        .astype(str)
        .map(
            lambda uid:
            str(valid_uid_to_path[uid])
            if uid in valid_uid_to_path
            else ""
        )
    )

    out["landmark_available"] = (
        out["landmark_feature_path"]
        .str.len()
        .gt(0)
    )

    return out


b2_train_df = attach_feature_paths(
    train_df
)

b2_val_df = attach_feature_paths(
    val_df
)


B2_TRAIN_MANIFEST = (
    B2A_ROOT /
    "b2_landmark_train_manifest.csv"
)

B2_VAL_MANIFEST = (
    B2A_ROOT /
    "b2_landmark_validation_manifest.csv"
)


b2_train_df.to_csv(
    B2_TRAIN_MANIFEST,
    index=False
)

b2_val_df.to_csv(
    B2_VAL_MANIFEST,
    index=False
)


print()
print("=" * 120)
print("11. B2 MANIFESTS SAVED")
print("=" * 120)

print(
    "Train:",
    B2_TRAIN_MANIFEST
)

print(
    "Validation:",
    B2_VAL_MANIFEST
)

# ==================================================================================================
# 12. FEATURE STATISTICS
# ==================================================================================================

print()
print("=" * 120)
print("12. FEATURE STATISTICS")
print("=" * 120)

sample_paths = list(
    valid_uid_to_path.values()
)[:100]

sample_means = []
sample_stds = []
sample_absmax = []


for path in sample_paths:

    arr = normalize_feature_shape(
        load_candidate_array(path)
    )

    if arr is None:
        continue

    sample_means.append(
        float(arr.mean())
    )

    sample_stds.append(
        float(arr.std())
    )

    sample_absmax.append(
        float(np.abs(arr).max())
    )


if sample_means:

    print(
        "Files sampled       :",
        len(sample_means)
    )

    print(
        "Mean(feature mean)  :",
        f"{np.mean(sample_means):.6f}"
    )

    print(
        "Mean(feature std)   :",
        f"{np.mean(sample_stds):.6f}"
    )

    print(
        "Maximum |value|     :",
        f"{np.max(sample_absmax):.6f}"
    )

# ==================================================================================================
# 13. CLASS COVERAGE
# ==================================================================================================

available_train = (
    b2_train_df[
        b2_train_df[
            "landmark_available"
        ]
    ]
)

available_val = (
    b2_val_df[
        b2_val_df[
            "landmark_available"
        ]
    ]
)


train_classes_available = (
    available_train[
        class_col
    ].nunique()
)

val_classes_available = (
    available_val[
        class_col
    ].nunique()
)


print()
print("=" * 120)
print("13. CLASS COVERAGE")
print("=" * 120)

print(
    "Train classes with features:",
    train_classes_available,
    "/ 215"
)

print(
    "Val classes with features  :",
    val_classes_available,
    "/ 215"
)

# ==================================================================================================
# 14. SAVE AUDIT SUMMARY
# ==================================================================================================

summary = {

    "experiment":
        "L20D-B2A",

    "purpose":
        "215-class landmark branch audit",

    "train_expected":
        710,

    "validation_expected":
        304,

    "locked_expected":
        10,

    "classes_expected":
        215,

    "train_landmarks_found":
        len(train_found),

    "validation_landmarks_found":
        len(val_found),

    "train_landmarks_missing":
        len(train_missing),

    "validation_landmarks_missing":
        len(val_missing),

    "train_classes_available":
        int(train_classes_available),

    "validation_classes_available":
        int(val_classes_available),

    "locked_test_searched":
        False,

    "locked_test_loaded":
        False,

    "l20c_loaded":
        False,

    "b1_modified":
        False,

    "expected_feature_shape":
        [32, 254],

}


SUMMARY_PATH = (
    B2A_ROOT /
    "l20d_b2a_landmark_audit.json"
)


with open(
    SUMMARY_PATH,
    "w"
) as f:

    json.dump(
        summary,
        f,
        indent=2
    )


# ==================================================================================================
# 15. SHA256
# ==================================================================================================

def sha256(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        while True:

            chunk = f.read(
                1024 * 1024
            )

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


print()
print("=" * 120)
print("15. ARTIFACT HASHES")
print("=" * 120)

for name, path in {
    "train_manifest":
        B2_TRAIN_MANIFEST,

    "validation_manifest":
        B2_VAL_MANIFEST,

    "summary":
        SUMMARY_PATH,

}.items():

    print(
        f"{name:<22}: "
        f"{sha256(path)}"
    )


# ==================================================================================================
# 16. FINAL DECISION
# ==================================================================================================

print()
print("=" * 120)
print("L20D-B2A COMPLETE")
print("=" * 120)

print(
    f"Training landmarks   : "
    f"{len(train_found)}/710"
)

print(
    f"Validation landmarks : "
    f"{len(val_found)}/304"
)

print(
    f"Train classes        : "
    f"{train_classes_available}/215"
)

print(
    f"Validation classes   : "
    f"{val_classes_available}/215"
)

print()
print(
    "Locked test searched : NO"
)

print(
    "Locked test decoded  : NO"
)

print(
    "L20C loaded          : NO"
)

print(
    "B1 modified          : NO"
)

print()

if (
    len(train_found) == 710 and
    len(val_found) == 304 and
    train_classes_available == 215 and
    val_classes_available == 215
):

    print(
        "[PASS] FULL LANDMARK COVERAGE."
    )

    print()
    print(
        "NEXT: train the new 215-class "
        "landmark temporal model."
    )

else:

    print(
        "[ATTENTION] Landmark coverage "
        "is incomplete."
    )

    print()
    print(
        "Do NOT train B2 yet."
    )

    print(
        "We will generate only the "
        "missing L6 features first."
    )

print("=" * 120)

CISLR — L20D-B2A
215-CLASS LANDMARK BRANCH DATASET AUDIT
Python          : 3.11.15
Project root    : /home/dipshikha/Music/cao/CISLR_RESEARCH
Landmark root   : /home/dipshikha/Music/cao/CISLR_RESEARCH/CISLR_LANDMARK_MODEL
B2 root         : /home/dipshikha/Music/cao/CISLR_RESEARCH/CISLR_LANDMARK_MODEL/audit/l20d_b2_multimodal_215class

3. VERIFYING AUTHORITATIVE INPUTS
[FOUND] train_manifest      : /home/dipshikha/Music/cao/CISLR_RESEARCH/CISLR_LANDMARK_MODEL/audit/l20b_corrected_strict_protocol_freeze/manifests/l20b_corrected_train_manifest.csv
[FOUND] val_manifest        : /home/dipshikha/Music/cao/CISLR_RESEARCH/CISLR_LANDMARK_MODEL/audit/l20b_corrected_strict_protocol_freeze/manifests/l20b_corrected_validation_manifest.csv
[FOUND] locked_manifest     : /home/dipshikha/Music/cao/CISLR_RESEARCH/CISLR_LANDMARK_MODEL/audit/l20b_corrected_strict_protocol_freeze/manifests/l20b_permanent_l19_locked_test_manifest.csv
[FOUND] class_mapping       : /home/dipshikha/Music/cao/CISLR_RESEARCH/CIS

In [2]:
# ==================================================================================================
# CISLR — L20D-B2B
# 215-CLASS LANDMARK TEMPORAL MODEL
#
# INPUT:
#       32 x 254 frozen landmark features
#
# MODEL:
#       Input projection
#       -> BiGRU
#       -> temporal attention
#       -> embedding
#       -> cosine classifier
#
# TRAIN:
#       710 videos / 215 classes
#
# VALIDATION:
#       304 videos / 215 classes
#
# LOCKED TEST:
#       NEVER LOADED
#
# ==================================================================================================

import os
import gc
import json
import math
import time
import random
import hashlib
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import (
    Dataset,
    DataLoader,
    WeightedRandomSampler,
)


# ==================================================================================================
# 1. CONFIG
# ==================================================================================================

SEED = 42

NUM_CLASSES = 215

SEQ_LEN = 32
INPUT_DIM = 254

HIDDEN_DIM = 256
PROJECTION_DIM = 256
EMBED_DIM = 256

NUM_GRU_LAYERS = 2

DROPOUT = 0.35

BATCH_SIZE = 32

MAX_EPOCHS = 100

PATIENCE = 15

LR = 3e-4

WEIGHT_DECAY = 1e-3

LABEL_SMOOTHING = 0.10

GRAD_CLIP = 1.0

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


PROJECT_ROOT = Path(
    "/home/dipshikha/Music/cao/CISLR_RESEARCH"
)

LANDMARK_ROOT = (
    PROJECT_ROOT /
    "CISLR_LANDMARK_MODEL"
)

B2_ROOT = (
    LANDMARK_ROOT /
    "audit" /
    "l20d_b2_multimodal_215class"
)

B2A_ROOT = (
    B2_ROOT /
    "b2a_landmark_audit"
)

TRAIN_MANIFEST = (
    B2A_ROOT /
    "b2_landmark_train_manifest.csv"
)

VAL_MANIFEST = (
    B2A_ROOT /
    "b2_landmark_validation_manifest.csv"
)

B2B_ROOT = (
    B2_ROOT /
    "b2b_landmark_temporal_model"
)

MODEL_DIR = (
    B2B_ROOT /
    "models"
)

REPORT_DIR = (
    B2B_ROOT /
    "reports"
)

MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

REPORT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


BEST_MODEL = (
    MODEL_DIR /
    "l20d_b2b_best_landmark_model.pt"
)

FINAL_MODEL = (
    MODEL_DIR /
    "l20d_b2b_final_landmark_model.pt"
)

HISTORY_FILE = (
    REPORT_DIR /
    "l20d_b2b_history.csv"
)

VAL_LOGITS_FILE = (
    REPORT_DIR /
    "l20d_b2b_best_validation_logits.npz"
)

VAL_PRED_FILE = (
    REPORT_DIR /
    "l20d_b2b_best_validation_predictions.csv"
)

NORMALIZATION_FILE = (
    REPORT_DIR /
    "l20d_b2b_train_normalization.npz"
)

MASTER_FREEZE = (
    B2B_ROOT /
    "L20D_B2B_MASTER_FREEZE.json"
)


# ==================================================================================================
# 2. SEED
# ==================================================================================================

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)

if torch.cuda.is_available():

    torch.cuda.manual_seed_all(
        SEED
    )

    torch.backends.cuda.matmul.allow_tf32 = True

torch.backends.cudnn.benchmark = True


# ==================================================================================================
# 3. HELPERS
# ==================================================================================================

def section(title):

    print()
    print("=" * 120)
    print(title)
    print("=" * 120)


def sha256_file(path):

    h = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as f:

        while True:

            chunk = f.read(
                1024 * 1024
            )

            if not chunk:
                break

            h.update(
                chunk
            )

    return h.hexdigest()


# ==================================================================================================
# 4. HEADER
# ==================================================================================================

section(
    "CISLR — L20D-B2B\n"
    "215-CLASS LANDMARK TEMPORAL MODEL"
)

print(
    "PyTorch              :",
    torch.__version__
)

print(
    "Device               :",
    DEVICE
)

if torch.cuda.is_available():

    print(
        "GPU                  :",
        torch.cuda.get_device_name(0)
    )

print(
    "Classes              :",
    NUM_CLASSES
)

print(
    "Sequence             :",
    f"{SEQ_LEN} x {INPUT_DIM}"
)

print(
    "Batch size           :",
    BATCH_SIZE
)

print(
    "Maximum epochs       :",
    MAX_EPOCHS
)

print(
    "Locked test          : DISABLED"
)


# ==================================================================================================
# 5. LOAD B2A MANIFESTS
# ==================================================================================================

section(
    "5. LOADING B2A MANIFESTS"
)

for path in [
    TRAIN_MANIFEST,
    VAL_MANIFEST,
]:

    if not path.exists():

        raise FileNotFoundError(
            path
        )


train_df = pd.read_csv(
    TRAIN_MANIFEST
)

val_df = pd.read_csv(
    VAL_MANIFEST
)


print(
    "Train:",
    len(train_df)
)

print(
    "Validation:",
    len(val_df)
)


if len(train_df) != 710:

    raise RuntimeError(
        "Expected 710 training rows."
    )


if len(val_df) != 304:

    raise RuntimeError(
        "Expected 304 validation rows."
    )


if train_df["class_id"].nunique() != 215:

    raise RuntimeError(
        "Training does not contain 215 classes."
    )


if val_df["class_id"].nunique() != 215:

    raise RuntimeError(
        "Validation does not contain 215 classes."
    )


if not train_df[
    "landmark_available"
].all():

    raise RuntimeError(
        "Missing training landmarks."
    )


if not val_df[
    "landmark_available"
].all():

    raise RuntimeError(
        "Missing validation landmarks."
    )


train_uids = set(
    train_df["uid"].astype(str)
)

val_uids = set(
    val_df["uid"].astype(str)
)


if train_uids & val_uids:

    raise RuntimeError(
        "Train/validation overlap!"
    )


print(
    "[PASS] 710 / 304 / 215 protocol preserved."
)


# ==================================================================================================
# 6. LOAD FEATURE
# ==================================================================================================

def load_feature(path):

    path = Path(path)


    if path.suffix.lower() == ".npy":

        x = np.load(
            path,
            allow_pickle=False
        )


    elif path.suffix.lower() == ".npz":

        data = np.load(
            path,
            allow_pickle=False
        )


        preferred = [
            "features",
            "feature",
            "landmarks",
            "x",
            "data",
        ]


        x = None


        for key in preferred:

            if key in data:

                x = data[key]
                break


        if x is None:

            keys = list(
                data.keys()
            )

            if len(keys) == 1:

                x = data[
                    keys[0]
                ]

            else:

                raise RuntimeError(
                    f"Cannot identify feature in {path}"
                )


    else:

        raise RuntimeError(
            f"Unsupported feature file: {path}"
        )


    x = np.asarray(
        x,
        dtype=np.float32
    )


    if (
        x.ndim == 3 and
        x.shape[0] == 1
    ):

        x = x[0]


    if x.shape != (
        SEQ_LEN,
        INPUT_DIM
    ):

        raise RuntimeError(
            f"Wrong feature shape "
            f"{x.shape} in {path}"
        )


    if not np.isfinite(
        x
    ).all():

        raise RuntimeError(
            f"NaN/Inf in {path}"
        )


    return x


# ==================================================================================================
# 7. COMPUTE TRAIN-ONLY NORMALIZATION
# ==================================================================================================

section(
    "7. COMPUTING TRAIN-ONLY NORMALIZATION"
)

# ------------------------------------------------------------------
# IMPORTANT:
#
# Validation features are NOT used to estimate mean/std.
# ------------------------------------------------------------------

feature_sum = np.zeros(
    INPUT_DIM,
    dtype=np.float64
)

feature_sq_sum = np.zeros(
    INPUT_DIM,
    dtype=np.float64
)

frame_count = 0


for i, row in train_df.iterrows():

    x = load_feature(
        row[
            "landmark_feature_path"
        ]
    )


    feature_sum += x.sum(
        axis=0
    )


    feature_sq_sum += (
        x.astype(
            np.float64
        ) ** 2
    ).sum(
        axis=0
    )


    frame_count += x.shape[0]


    if (
        (i + 1) % 100 == 0 or
        i + 1 == len(train_df)
    ):

        print(
            f"Processed "
            f"{i+1:4d}/710"
        )


feature_mean = (
    feature_sum /
    frame_count
)


feature_var = (
    feature_sq_sum /
    frame_count
) - (
    feature_mean ** 2
)


feature_var = np.maximum(
    feature_var,
    1e-8
)


feature_std = np.sqrt(
    feature_var
)


# Prevent extremely tiny dimensions
# from exploding during normalization.

feature_std = np.maximum(
    feature_std,
    1e-3
)


feature_mean = feature_mean.astype(
    np.float32
)

feature_std = feature_std.astype(
    np.float32
)


np.savez_compressed(
    NORMALIZATION_FILE,
    mean=feature_mean,
    std=feature_std,
)


print()
print(
    "Mean(mean):",
    f"{feature_mean.mean():.6f}"
)

print(
    "Mean(std) :",
    f"{feature_std.mean():.6f}"
)

print(
    "Min(std)  :",
    f"{feature_std.min():.6f}"
)

print(
    "[PASS] Train-only normalization saved."
)


# ==================================================================================================
# 8. LANDMARK-SAFE AUGMENTATION
# ==================================================================================================

def augment_landmark_feature(x):

    x = x.copy()


    # ------------------------------------------------------------------
    # 1. Mild feature noise.
    # ------------------------------------------------------------------

    if random.random() < 0.60:

        noise = np.random.normal(
            loc=0.0,
            scale=0.015,
            size=x.shape
        ).astype(
            np.float32
        )

        x += noise


    # ------------------------------------------------------------------
    # 2. Mild global amplitude perturbation.
    # ------------------------------------------------------------------

    if random.random() < 0.50:

        scale = random.uniform(
            0.95,
            1.05
        )

        x *= scale


    # ------------------------------------------------------------------
    # 3. Small temporal shift.
    #
    # NO temporal reversal.
    # ------------------------------------------------------------------

    if random.random() < 0.40:

        shift = random.choice(
            [-2, -1, 1, 2]
        )

        x = np.roll(
            x,
            shift=shift,
            axis=0
        )


        if shift > 0:

            x[:shift] = x[
                shift
            ]

        else:

            x[shift:] = x[
                shift - 1
            ]


    # ------------------------------------------------------------------
    # 4. Temporal frame masking.
    #
    # Hide at most 2 frames.
    # ------------------------------------------------------------------

    if random.random() < 0.25:

        mask_count = random.randint(
            1,
            2
        )

        ids = np.random.choice(
            SEQ_LEN,
            size=mask_count,
            replace=False
        )

        for idx in ids:

            if 0 < idx < SEQ_LEN - 1:

                x[idx] = (
                    x[idx - 1] +
                    x[idx + 1]
                ) / 2.0


    return x


# ==================================================================================================
# 9. DATASET
# ==================================================================================================

class LandmarkDataset(Dataset):

    def __init__(
        self,
        df,
        mean,
        std,
        train=False
    ):

        self.df = (
            df.reset_index(
                drop=True
            )
        )

        self.mean = mean
        self.std = std
        self.train = train


    def __len__(self):

        return len(
            self.df
        )


    def __getitem__(
        self,
        idx
    ):

        row = self.df.iloc[
            idx
        ]


        x = load_feature(
            row[
                "landmark_feature_path"
            ]
        )


        if self.train:

            x = augment_landmark_feature(
                x
            )


        x = (
            x -
            self.mean[
                None,
                :
            ]
        ) / (
            self.std[
                None,
                :
            ]
        )


        # Limit rare extreme values.

        x = np.clip(
            x,
            -8.0,
            8.0
        )


        return {

            "x":
                torch.from_numpy(
                    x.astype(
                        np.float32
                    )
                ),

            "label":
                torch.tensor(
                    int(
                        row[
                            "class_id"
                        ]
                    ),
                    dtype=torch.long
                ),

            "uid":
                str(
                    row[
                        "uid"
                    ]
                ),

            "gloss":
                str(
                    row[
                        "gloss"
                    ]
                ),
        }


train_dataset = LandmarkDataset(
    train_df,
    feature_mean,
    feature_std,
    train=True
)


val_dataset = LandmarkDataset(
    val_df,
    feature_mean,
    feature_std,
    train=False
)


# ==================================================================================================
# 10. BALANCED TRAIN SAMPLER
# ==================================================================================================

section(
    "10. BALANCED SAMPLER"
)

class_counts = Counter(
    train_df[
        "class_id"
    ].astype(
        int
    ).tolist()
)


sample_weights = np.array(
    [
        1.0 /
        class_counts[
            int(label)
        ]

        for label
        in train_df[
            "class_id"
        ]
    ],
    dtype=np.float64
)


generator = torch.Generator()

generator.manual_seed(
    SEED
)


sampler = WeightedRandomSampler(
    weights=torch.DoubleTensor(
        sample_weights
    ),
    num_samples=len(
        train_dataset
    ),
    replacement=True,
    generator=generator
)


print(
    "Samples/epoch:",
    len(train_dataset)
)

print(
    "Classes:",
    len(class_counts)
)


# ==================================================================================================
# 11. LOADERS
# ==================================================================================================

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=sampler,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True
)


val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True
)


batch = next(
    iter(
        train_loader
    )
)


section(
    "11. DATALOADER SANITY"
)

print(
    "Input:",
    tuple(
        batch[
            "x"
        ].shape
    )
)

print(
    "Labels:",
    tuple(
        batch[
            "label"
        ].shape
    )
)

print(
    "Expected:",
    f"(B, {SEQ_LEN}, {INPUT_DIM})"
)


# ==================================================================================================
# 12. MODEL
# ==================================================================================================

class CosineClassifier(nn.Module):

    def __init__(
        self,
        input_dim,
        num_classes,
        scale=10.0
    ):

        super().__init__()


        self.weight = nn.Parameter(
            torch.empty(
                num_classes,
                input_dim
            )
        )


        nn.init.normal_(
            self.weight,
            mean=0.0,
            std=0.02
        )


        self.log_scale = nn.Parameter(
            torch.tensor(
                math.log(
                    scale
                ),
                dtype=torch.float32
            )
        )


    def forward(
        self,
        x
    ):

        x = F.normalize(
            x,
            dim=1
        )


        w = F.normalize(
            self.weight,
            dim=1
        )


        scale = (
            self.log_scale
            .exp()
            .clamp(
                1.0,
                30.0
            )
        )


        return scale * F.linear(
            x,
            w
        )


class LandmarkBiGRUAttention(
    nn.Module
):

    def __init__(
        self
    ):

        super().__init__()


        self.input_norm = nn.LayerNorm(
            INPUT_DIM
        )


        self.input_projection = nn.Sequential(

            nn.Linear(
                INPUT_DIM,
                PROJECTION_DIM
            ),

            nn.GELU(),

            nn.Dropout(
                0.20
            ),
        )


        self.gru = nn.GRU(

            input_size=PROJECTION_DIM,

            hidden_size=HIDDEN_DIM,

            num_layers=NUM_GRU_LAYERS,

            batch_first=True,

            bidirectional=True,

            dropout=DROPOUT
            if NUM_GRU_LAYERS > 1
            else 0.0
        )


        temporal_dim = (
            HIDDEN_DIM * 2
        )


        self.temporal_norm = nn.LayerNorm(
            temporal_dim
        )


        self.attention = nn.Sequential(

            nn.Linear(
                temporal_dim,
                128
            ),

            nn.Tanh(),

            nn.Dropout(
                0.20
            ),

            nn.Linear(
                128,
                1
            )
        )


        self.embedding = nn.Sequential(

            nn.Linear(
                temporal_dim,
                EMBED_DIM
            ),

            nn.LayerNorm(
                EMBED_DIM
            ),

            nn.GELU(),

            nn.Dropout(
                DROPOUT
            )
        )


        self.classifier = CosineClassifier(

            EMBED_DIM,

            NUM_CLASSES,

            scale=10.0
        )


    def forward(
        self,
        x
    ):

        # x:
        # [B, 32, 254]

        x = self.input_norm(
            x
        )


        x = self.input_projection(
            x
        )


        temporal, _ = self.gru(
            x
        )


        temporal = self.temporal_norm(
            temporal
        )


        attention_logits = (
            self.attention(
                temporal
            )
            .squeeze(-1)
        )


        attention_weights = F.softmax(
            attention_logits,
            dim=1
        )


        pooled = torch.sum(

            temporal *

            attention_weights.unsqueeze(
                -1
            ),

            dim=1
        )


        embedding = self.embedding(
            pooled
        )


        logits = self.classifier(
            embedding
        )


        return (
            logits,
            embedding,
            attention_weights
        )


model = LandmarkBiGRUAttention().to(
    DEVICE
)


# ==================================================================================================
# 13. MODEL SUMMARY
# ==================================================================================================

section(
    "13. MODEL SUMMARY"
)


total_params = sum(
    p.numel()
    for p in model.parameters()
)


trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)


print(
    f"Total parameters     : "
    f"{total_params:,}"
)

print(
    f"Trainable parameters : "
    f"{trainable_params:,}"
)


with torch.no_grad():

    x = batch[
        "x"
    ].to(
        DEVICE
    )


    logits, embedding, attn = (
        model(
            x
        )
    )


print(
    "Logits:",
    tuple(
        logits.shape
    )
)

print(
    "Embedding:",
    tuple(
        embedding.shape
    )
)

print(
    "Attention:",
    tuple(
        attn.shape
    )
)


del batch
del x
del logits
del embedding
del attn

gc.collect()

if torch.cuda.is_available():

    torch.cuda.empty_cache()


# ==================================================================================================
# 14. LOSS
# ==================================================================================================

criterion = nn.CrossEntropyLoss(
    label_smoothing=LABEL_SMOOTHING
)


# ==================================================================================================
# 15. OPTIMIZER
# ==================================================================================================

optimizer = torch.optim.AdamW(

    model.parameters(),

    lr=LR,

    weight_decay=WEIGHT_DECAY
)


# ==================================================================================================
# 16. COSINE SCHEDULER WITH WARMUP
# ==================================================================================================

WARMUP_EPOCHS = 5


def lr_lambda(epoch):

    if epoch < WARMUP_EPOCHS:

        return (
            epoch + 1
        ) / WARMUP_EPOCHS


    progress = (
        epoch -
        WARMUP_EPOCHS
    ) / max(
        1,
        MAX_EPOCHS -
        WARMUP_EPOCHS
    )


    return (
        0.05 +
        0.95 *
        0.5 *
        (
            1.0 +
            math.cos(
                math.pi *
                progress
            )
        )
    )


scheduler = (
    torch.optim.lr_scheduler.LambdaLR(
        optimizer,
        lr_lambda
    )
)


# ==================================================================================================
# 17. METRICS
# ==================================================================================================

def topk_correct(
    logits,
    labels,
    k
):

    pred = logits.topk(
        k,
        dim=1
    ).indices


    return int(
        (
            pred ==
            labels.unsqueeze(1)
        )
        .any(dim=1)
        .sum()
        .item()
    )


# ==================================================================================================
# 18. TRAIN ONE EPOCH
# ==================================================================================================

def train_one_epoch(
    epoch
):

    model.train()


    total = 0

    loss_sum = 0.0

    correct1 = 0
    correct5 = 0


    start = time.time()


    for step, batch in enumerate(
        train_loader,
        start=1
    ):

        x = batch[
            "x"
        ].to(
            DEVICE,
            non_blocking=True
        )


        y = batch[
            "label"
        ].to(
            DEVICE,
            non_blocking=True
        )


        optimizer.zero_grad(
            set_to_none=True
        )


        logits, _, _ = model(
            x
        )


        loss = criterion(
            logits,
            y
        )


        loss.backward()


        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            GRAD_CLIP
        )


        optimizer.step()


        bs = y.size(0)


        total += bs


        loss_sum += (
            loss.item() *
            bs
        )


        correct1 += topk_correct(
            logits,
            y,
            1
        )


        correct5 += topk_correct(
            logits,
            y,
            5
        )


    elapsed = (
        time.time() -
        start
    )


    return {

        "loss":
            loss_sum / total,

        "top1":
            100.0 *
            correct1 / total,

        "top5":
            100.0 *
            correct5 / total,

        "time":
            elapsed
    }


# ==================================================================================================
# 19. VALIDATION
# ==================================================================================================

@torch.no_grad()
def evaluate():

    model.eval()


    total = 0

    loss_sum = 0.0

    correct1 = 0
    correct5 = 0


    all_logits = []

    all_labels = []

    all_uids = []

    all_glosses = []


    for batch in val_loader:

        x = batch[
            "x"
        ].to(
            DEVICE,
            non_blocking=True
        )


        y = batch[
            "label"
        ].to(
            DEVICE,
            non_blocking=True
        )


        logits, _, _ = model(
            x
        )


        loss = criterion(
            logits,
            y
        )


        bs = y.size(0)


        total += bs


        loss_sum += (
            loss.item() *
            bs
        )


        correct1 += topk_correct(
            logits,
            y,
            1
        )


        correct5 += topk_correct(
            logits,
            y,
            5
        )


        all_logits.append(
            logits.cpu()
        )


        all_labels.append(
            y.cpu()
        )


        all_uids.extend(
            batch[
                "uid"
            ]
        )


        all_glosses.extend(
            batch[
                "gloss"
            ]
        )


    logits = torch.cat(
        all_logits,
        dim=0
    )


    labels = torch.cat(
        all_labels,
        dim=0
    )


    return {

        "loss":
            loss_sum / total,

        "top1":
            100.0 *
            correct1 / total,

        "top5":
            100.0 *
            correct5 / total,

        "correct1":
            correct1,

        "correct5":
            correct5,

        "total":
            total,

        "logits":
            logits.numpy(),

        "labels":
            labels.numpy(),

        "uids":
            all_uids,

        "glosses":
            all_glosses
    }


# ==================================================================================================
# 20. TRAIN
# ==================================================================================================

section(
    "20. STARTING B2B TRAINING"
)

print(
    "[LOCK] Validation only for model selection."
)

print(
    "[LOCK] Locked test is not loaded."
)


history = []


best_top1 = -1.0
best_top5 = -1.0
best_loss = float(
    "inf"
)

best_epoch = None

epochs_without_improvement = 0


for epoch in range(
    1,
    MAX_EPOCHS + 1
):

    train_result = train_one_epoch(
        epoch
    )


    val_result = evaluate()


    current_lr = optimizer.param_groups[
        0
    ][
        "lr"
    ]


    print()
    print(
        f"EPOCH {epoch:03d}/{MAX_EPOCHS}"
    )

    print(
        f"LR          : "
        f"{current_lr:.8f}"
    )

    print(
        f"TRAIN loss  : "
        f"{train_result['loss']:.6f}"
    )

    print(
        f"TRAIN Top1  : "
        f"{train_result['top1']:.3f}%"
    )

    print(
        f"TRAIN Top5  : "
        f"{train_result['top5']:.3f}%"
    )

    print(
        f"VAL loss    : "
        f"{val_result['loss']:.6f}"
    )

    print(
        f"VAL Top1    : "
        f"{val_result['top1']:.3f}% "
        f"({val_result['correct1']}/304)"
    )

    print(
        f"VAL Top5    : "
        f"{val_result['top5']:.3f}% "
        f"({val_result['correct5']}/304)"
    )


    history.append(
        {
            "epoch":
                epoch,

            "lr":
                current_lr,

            "train_loss":
                train_result[
                    "loss"
                ],

            "train_top1":
                train_result[
                    "top1"
                ],

            "train_top5":
                train_result[
                    "top5"
                ],

            "val_loss":
                val_result[
                    "loss"
                ],

            "val_top1":
                val_result[
                    "top1"
                ],

            "val_top5":
                val_result[
                    "top5"
                ],
        }
    )


    pd.DataFrame(
        history
    ).to_csv(
        HISTORY_FILE,
        index=False
    )


    # ------------------------------------------------------------------
    # Model selection:
    #
    # Top1 primary
    # Top5 secondary
    # loss tertiary
    # ------------------------------------------------------------------

    better = False


    if (
        val_result[
            "top1"
        ] > best_top1
    ):

        better = True


    elif (
        val_result[
            "top1"
        ] == best_top1 and
        val_result[
            "top5"
        ] > best_top5
    ):

        better = True


    elif (
        val_result[
            "top1"
        ] == best_top1 and
        val_result[
            "top5"
        ] == best_top5 and
        val_result[
            "loss"
        ] < best_loss
    ):

        better = True


    if better:

        best_top1 = val_result[
            "top1"
        ]

        best_top5 = val_result[
            "top5"
        ]

        best_loss = val_result[
            "loss"
        ]

        best_epoch = epoch


        epochs_without_improvement = 0


        torch.save(
            {
                "model_state_dict":
                    model.state_dict(),

                "epoch":
                    epoch,

                "val_top1":
                    best_top1,

                "val_top5":
                    best_top5,

                "val_loss":
                    best_loss,

                "num_classes":
                    NUM_CLASSES,

                "seq_len":
                    SEQ_LEN,

                "input_dim":
                    INPUT_DIM,

                "hidden_dim":
                    HIDDEN_DIM,

                "embedding_dim":
                    EMBED_DIM,

                "locked_test_used":
                    False
            },
            BEST_MODEL
        )


        np.savez_compressed(
            VAL_LOGITS_FILE,

            logits=val_result[
                "logits"
            ],

            labels=val_result[
                "labels"
            ],

            uids=np.array(
                val_result[
                    "uids"
                ]
            )
        )


        pred_ids = (
            val_result[
                "logits"
            ]
            .argmax(
                axis=1
            )
        )


        prediction_df = pd.DataFrame(
            {
                "uid":
                    val_result[
                        "uids"
                    ],

                "gloss":
                    val_result[
                        "glosses"
                    ],

                "true_class_id":
                    val_result[
                        "labels"
                    ],

                "pred_class_id":
                    pred_ids,

                "correct":
                    (
                        pred_ids ==
                        val_result[
                            "labels"
                        ]
                    ).astype(
                        int
                    )
            }
        )


        prediction_df.to_csv(
            VAL_PRED_FILE,
            index=False
        )


        print()
        print(
            "[SAVED BEST]"
        )

        print(
            "Epoch:",
            best_epoch
        )

        print(
            "Val Top1:",
            f"{best_top1:.3f}%"
        )

        print(
            "Val Top5:",
            f"{best_top5:.3f}%"
        )


    else:

        epochs_without_improvement += 1


        print(
            f"[NO IMPROVEMENT] "
            f"{epochs_without_improvement}/"
            f"{PATIENCE}"
        )


    scheduler.step()


    if (
        epochs_without_improvement >=
        PATIENCE
    ):

        print()
        print(
            "[EARLY STOP]"
        )

        break


# ==================================================================================================
# 21. LOAD BEST MODEL
# ==================================================================================================

section(
    "21. LOADING BEST B2B MODEL"
)


checkpoint = torch.load(
    BEST_MODEL,
    map_location="cpu",
    weights_only=False
)


model.load_state_dict(
    checkpoint[
        "model_state_dict"
    ]
)


model = model.to(
    DEVICE
)


print(
    "Best epoch:",
    checkpoint[
        "epoch"
    ]
)

print(
    "Best Top1 :",
    f"{checkpoint['val_top1']:.3f}%"
)

print(
    "Best Top5 :",
    f"{checkpoint['val_top5']:.3f}%"
)


# ==================================================================================================
# 22. FINAL VALIDATION
# ==================================================================================================

final_val = evaluate()


section(
    "22. FINAL B2B VALIDATION"
)


print(
    f"Top1 : "
    f"{final_val['top1']:.3f}% "
    f"({final_val['correct1']}/304)"
)

print(
    f"Top5 : "
    f"{final_val['top5']:.3f}% "
    f"({final_val['correct5']}/304)"
)

print(
    f"Loss : "
    f"{final_val['loss']:.6f}"
)


# ==================================================================================================
# 23. SAVE FINAL
# ==================================================================================================

torch.save(
    {
        "model_state_dict":
            model.state_dict(),

        "best_epoch":
            checkpoint[
                "epoch"
            ],

        "validation_top1":
            final_val[
                "top1"
            ],

        "validation_top5":
            final_val[
                "top5"
            ],

        "validation_loss":
            final_val[
                "loss"
            ],

        "normalization_file":
            str(
                NORMALIZATION_FILE
            ),

        "locked_test_used":
            False
    },
    FINAL_MODEL
)


# ==================================================================================================
# 24. MASTER FREEZE
# ==================================================================================================

freeze = {

    "experiment":
        "L20D-B2B",

    "model":
        "BiGRU + Temporal Attention + Cosine Classifier",

    "protocol": {

        "train":
            710,

        "validation":
            304,

        "classes":
            215,

        "locked_test":
            10
    },

    "input": {

        "sequence_length":
            32,

        "feature_dimension":
            254
    },

    "normalization": {

        "source":
            "TRAIN ONLY",

        "file":
            str(
                NORMALIZATION_FILE
            )
    },

    "best_result": {

        "epoch":
            int(
                checkpoint[
                    "epoch"
                ]
            ),

        "validation_top1":
            float(
                final_val[
                    "top1"
                ]
            ),

        "validation_top5":
            float(
                final_val[
                    "top5"
                ]
            ),

        "validation_loss":
            float(
                final_val[
                    "loss"
                ]
            )
    },

    "integrity": {

        "locked_test_loaded":
            False,

        "locked_test_evaluated":
            False,

        "train_validation_overlap":
            0
    },

    "artifacts": {

        "best_model":
            str(
                BEST_MODEL
            ),

        "final_model":
            str(
                FINAL_MODEL
            ),

        "validation_logits":
            str(
                VAL_LOGITS_FILE
            ),

        "validation_predictions":
            str(
                VAL_PRED_FILE
            ),

        "history":
            str(
                HISTORY_FILE
            )
    }
}


with open(
    MASTER_FREEZE,
    "w"
) as f:

    json.dump(
        freeze,
        f,
        indent=2
    )


# ==================================================================================================
# 25. HASHES
# ==================================================================================================

section(
    "25. ARTIFACT HASHES"
)


for name, path in {

    "best_model":
        BEST_MODEL,

    "final_model":
        FINAL_MODEL,

    "history":
        HISTORY_FILE,

    "val_logits":
        VAL_LOGITS_FILE,

    "normalization":
        NORMALIZATION_FILE,

    "master_freeze":
        MASTER_FREEZE

}.items():

    print(
        f"{name:<20}: "
        f"{sha256_file(path)}"
    )


# ==================================================================================================
# 26. COMPLETE
# ==================================================================================================

section(
    "L20D-B2B COMPLETE"
)


print(
    "Model             : "
    "BiGRU + Attention + Cosine"
)

print(
    f"Best epoch        : "
    f"{checkpoint['epoch']}"
)

print(
    f"Validation Top1   : "
    f"{final_val['top1']:.3f}%"
)

print(
    f"Validation Top5   : "
    f"{final_val['top5']:.3f}%"
)

print()
print(
    "Locked test loaded    : NO"
)

print(
    "Locked test evaluated : NO"
)

print()
print(
    "Validation logits saved:"
)

print(
    VAL_LOGITS_FILE
)

print()
print(
    "[NEXT] B2C: fuse this landmark branch "
    "with B1 VideoMAE."
)

print("=" * 120)


CISLR — L20D-B2B
215-CLASS LANDMARK TEMPORAL MODEL
PyTorch              : 2.14.0+cu130
Device               : cuda
GPU                  : NVIDIA GeForce RTX 4050 Laptop GPU
Classes              : 215
Sequence             : 32 x 254
Batch size           : 32
Maximum epochs       : 100
Locked test          : DISABLED

5. LOADING B2A MANIFESTS
Train: 710
Validation: 304
[PASS] 710 / 304 / 215 protocol preserved.

7. COMPUTING TRAIN-ONLY NORMALIZATION
Processed  100/710
Processed  200/710
Processed  300/710
Processed  400/710
Processed  500/710
Processed  600/710
Processed  700/710
Processed  710/710

Mean(mean): -0.050142
Mean(std) : 0.483956
Min(std)  : 0.001000
[PASS] Train-only normalization saved.

10. BALANCED SAMPLER
Samples/epoch: 710
Classes: 215

11. DATALOADER SANITY
Input: (32, 32, 254)
Labels: (32,)
Expected: (B, 32, 254)

13. MODEL SUMMARY
Total parameters     : 2,291,710
Trainable parameters : 2,291,710
Logits: (32, 215)
Embedding: (32, 256)
Attention: (32, 32)

20. STARTIN

In [5]:
# ==================================================================================================
# CISLR — L20D-B2C CORRECTED
# FROZEN B1 RGB + FROZEN B2B LANDMARK FUSION
#
# FIXES:
#   1. Exact B1 LayerNorm + CosineClassifier head
#   2. Exact B1 three-view validation inference: [0.15, 0.50, 0.85]
#   3. strict=True checkpoint loading
#
# LOCKED TEST:
#   NEVER DECODED / NEVER EVALUATED
# ==================================================================================================

import gc
import json
import math
import time
import random
import hashlib
from pathlib import Path

import cv2
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from transformers import VideoMAEImageProcessor, VideoMAEModel


# ==================================================================================================
# 1. CONFIG
# ==================================================================================================

SEED = 42
NUM_CLASSES = 215
NUM_FRAMES = 16

MODEL_NAME = "MCG-NJU/videomae-base-finetuned-kinetics"

VAL_VIEW_POSITIONS = [0.15, 0.50, 0.85]

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cuda.matmul.allow_tf32 = True

torch.backends.cudnn.benchmark = True


# ==================================================================================================
# 2. PATHS
# ==================================================================================================

PROJECT_ROOT = Path(
    "/home/dipshikha/Music/cao/CISLR_RESEARCH"
)

LANDMARK_ROOT = PROJECT_ROOT / "CISLR_LANDMARK_MODEL"

L20B_ROOT = (
    LANDMARK_ROOT /
    "audit" /
    "l20b_corrected_strict_protocol_freeze"
)

MANIFEST_ROOT = L20B_ROOT / "manifests"

VAL_MANIFEST = (
    MANIFEST_ROOT /
    "l20b_corrected_validation_manifest.csv"
)

LOCKED_MANIFEST = (
    MANIFEST_ROOT /
    "l20b_permanent_l19_locked_test_manifest.csv"
)

B1_MODEL = (
    LANDMARK_ROOT /
    "audit" /
    "l20d_b1_lowshot_videomae_215class" /
    "models" /
    "l20d_b1_best.pt"
)

B2_ROOT = (
    LANDMARK_ROOT /
    "audit" /
    "l20d_b2_multimodal_215class"
)

B2B_LOGITS = (
    B2_ROOT /
    "b2b_landmark_temporal_model" /
    "reports" /
    "l20d_b2b_best_validation_logits.npz"
)

B2C_ROOT = (
    B2_ROOT /
    "b2c_rgb_landmark_fusion_corrected"
)

B2C_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

B1_LOGITS_FILE = (
    B2C_ROOT /
    "b1_corrected_validation_logits.npz"
)

ALIGNED_FILE = (
    B2C_ROOT /
    "b2c_aligned_logits.npz"
)

SEARCH_FILE = (
    B2C_ROOT /
    "b2c_fusion_search.csv"
)

PREDICTIONS_FILE = (
    B2C_ROOT /
    "b2c_best_predictions.csv"
)

COMPLEMENTARITY_FILE = (
    B2C_ROOT /
    "b2c_complementarity.csv"
)

SUMMARY_FILE = (
    B2C_ROOT /
    "b2c_summary.json"
)

MASTER_FREEZE = (
    B2C_ROOT /
    "L20D_B2C_CORRECTED_MASTER_FREEZE.json"
)


# ==================================================================================================
# 3. HELPERS
# ==================================================================================================

def section(title):
    print()
    print("=" * 120)
    print(title)
    print("=" * 120)


def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        while True:

            block = f.read(1024 * 1024)

            if not block:
                break

            h.update(block)

    return h.hexdigest()


def softmax_np(logits, temperature=1.0):

    x = logits / max(
        float(temperature),
        1e-8
    )

    x = x - x.max(
        axis=1,
        keepdims=True
    )

    x = np.exp(x)

    return x / np.clip(
        x.sum(
            axis=1,
            keepdims=True
        ),
        1e-12,
        None
    )


def log_softmax_np(logits, temperature=1.0):

    p = softmax_np(
        logits,
        temperature
    )

    return np.log(
        np.clip(
            p,
            1e-12,
            1.0
        )
    )


def topk_accuracy(scores, labels, k):

    indices = np.argpartition(
        scores,
        -k,
        axis=1
    )[:, -k:]

    correct = (
        indices ==
        labels[:, None]
    ).any(axis=1)

    return 100.0 * float(
        correct.mean()
    )


# ==================================================================================================
# 4. LOAD PROTOCOL
# ==================================================================================================

section(
    "L20D-B2C CORRECTED — PROTOCOL"
)

val_df = pd.read_csv(
    VAL_MANIFEST
)

locked_df = pd.read_csv(
    LOCKED_MANIFEST
)

assert len(val_df) == 304
assert len(locked_df) == 10

assert val_df["uid"].nunique() == 304
assert val_df["class_id"].nunique() == 215

val_uids_set = set(
    val_df["uid"].astype(str)
)

locked_uids_set = set(
    locked_df["uid"].astype(str)
)

assert len(
    val_uids_set &
    locked_uids_set
) == 0

print("Validation :", len(val_df))
print("Classes    :", val_df["class_id"].nunique())
print("Locked     :", len(locked_df))
print("Overlap    :", 0)
print()
print("[PASS] Correct 304-validation protocol.")
print("[LOCK] Locked test will NOT be decoded.")


# ==================================================================================================
# 5. EXACT B1 COSINE CLASSIFIER
# ==================================================================================================

class CosineClassifier(nn.Module):

    def __init__(
        self,
        input_dim,
        num_classes,
        scale=10.0
    ):

        super().__init__()

        self.weight = nn.Parameter(
            torch.empty(
                num_classes,
                input_dim
            )
        )

        nn.init.normal_(
            self.weight,
            mean=0.0,
            std=0.02
        )

        self.log_scale = nn.Parameter(
            torch.tensor(
                math.log(scale),
                dtype=torch.float32
            )
        )


    def forward(self, x):

        x = F.normalize(
            x,
            dim=1
        )

        w = F.normalize(
            self.weight,
            dim=1
        )

        scale = (
            self.log_scale
            .exp()
            .clamp(
                1.0,
                30.0
            )
        )

        return scale * F.linear(
            x,
            w
        )


# ==================================================================================================
# 6. EXACT B1 MODEL
# ==================================================================================================

class B1ExactVideoMAE(nn.Module):

    def __init__(self):

        super().__init__()

        self.backbone = VideoMAEModel.from_pretrained(
            MODEL_NAME
        )

        hidden = self.backbone.config.hidden_size

        assert hidden == 768

        self.feature_norm = nn.LayerNorm(
            hidden
        )

        self.dropout = nn.Dropout(
            0.35
        )

        self.classifier = CosineClassifier(
            hidden,
            NUM_CLASSES,
            scale=10.0
        )


    def forward(self, pixel_values):

        outputs = self.backbone(
            pixel_values=pixel_values
        )

        x = outputs.last_hidden_state.mean(
            dim=1
        )

        x = self.feature_norm(
            x
        )

        x = self.dropout(
            x
        )

        logits = self.classifier(
            x
        )

        return logits


# ==================================================================================================
# 7. LOAD B1 EXACTLY
# ==================================================================================================

section(
    "7. LOADING EXACT FROZEN B1"
)

checkpoint = torch.load(
    B1_MODEL,
    map_location="cpu",
    weights_only=False
)

sd = checkpoint[
    "model_state_dict"
]

model = B1ExactVideoMAE()

result = model.load_state_dict(
    sd,
    strict=True
)

print(
    "Checkpoint loaded with strict=True."
)

print(
    "Missing keys    :",
    result.missing_keys
)

print(
    "Unexpected keys :",
    result.unexpected_keys
)

print()
print(
    "Checkpoint epoch :",
    checkpoint["global_epoch"]
)

print(
    "Checkpoint stage :",
    checkpoint["stage"]
)

print(
    "Frozen Top1      :",
    checkpoint["best_top1"]
)

print(
    "Frozen Top5      :",
    checkpoint["best_top5"]
)

print(
    "L20C loaded      :",
    checkpoint["l20c_loaded"]
)

print(
    "B0 loaded        :",
    checkpoint["b0_loaded"]
)

print(
    "Locked used      :",
    checkpoint["locked_test_used"]
)

model = model.to(
    DEVICE
)

model.eval()

for p in model.parameters():
    p.requires_grad = False

print()
print("[PASS] Exact B1 architecture restored.")


# ==================================================================================================
# 8. PROCESSOR
# ==================================================================================================

processor = VideoMAEImageProcessor.from_pretrained(
    MODEL_NAME
)


# ==================================================================================================
# 9. DECODE VIDEO ONCE
# ==================================================================================================

def decode_all_frames(video_path):

    cap = cv2.VideoCapture(
        str(video_path)
    )

    if not cap.isOpened():

        raise RuntimeError(
            f"Cannot open: {video_path}"
        )

    frames = []

    while True:

        ok, frame = cap.read()

        if not ok:
            break

        frame = cv2.cvtColor(
            frame,
            cv2.COLOR_BGR2RGB
        )

        frames.append(frame)

    cap.release()

    if len(frames) == 0:

        raise RuntimeError(
            f"No frames: {video_path}"
        )

    return frames


# ==================================================================================================
# 10. THREE TEMPORAL VIEWS
#
# Deterministic validation:
# positions = 0.15, 0.50, 0.85
# ==================================================================================================

def temporal_view(
    frames,
    position,
    num_frames=16
):

    n = len(frames)

    if n <= num_frames:

        ids = np.linspace(
            0,
            n - 1,
            num_frames
        ).round().astype(
            np.int64
        )

        return [
            frames[i]
            for i in ids
        ]

    # ------------------------------------------------------------------
    # Use a temporal window covering ~80% of the video.
    # Position chooses early / middle / late window.
    # ------------------------------------------------------------------

    span = max(
        num_frames,
        int(
            round(
                n * 0.80
            )
        )
    )

    span = min(
        span,
        n
    )

    max_start = n - span

    start = int(
        round(
            position *
            max_start
        )
    )

    start = max(
        0,
        min(
            start,
            max_start
        )
    )

    end = start + span - 1

    ids = np.linspace(
        start,
        end,
        num_frames
    ).round().astype(
        np.int64
    )

    return [
        frames[i]
        for i in ids
    ]


# ==================================================================================================
# 11. EXTRACT EXACT B1 THREE-VIEW LOGITS
# ==================================================================================================

section(
    "11. EXTRACTING B1 3-VIEW VALIDATION LOGITS"
)

all_logits = []
all_labels = []
all_uids = []

start_time = time.time()


with torch.inference_mode():

    for index, row in val_df.iterrows():

        uid = str(
            row["uid"]
        )

        label = int(
            row["class_id"]
        )

        video_path = Path(
            row["video_path"]
        )

        if not video_path.exists():

            raise FileNotFoundError(
                video_path
            )

        frames = decode_all_frames(
            video_path
        )

        view_logits = []

        for position in VAL_VIEW_POSITIONS:

            view = temporal_view(
                frames,
                position,
                NUM_FRAMES
            )

            encoded = processor(
                view,
                return_tensors="pt"
            )

            pixel_values = encoded[
                "pixel_values"
            ].to(
                DEVICE,
                non_blocking=True
            )

            logits = model(
                pixel_values
            )

            view_logits.append(
                logits
            )

        # --------------------------------------------------------------
        # B1 validation:
        # average logits across 3 deterministic views.
        # --------------------------------------------------------------

        averaged_logits = torch.stack(
            view_logits,
            dim=0
        ).mean(
            dim=0
        )

        all_logits.append(
            averaged_logits.cpu()
        )

        all_labels.append(
            label
        )

        all_uids.append(
            uid
        )

        if (
            (index + 1) % 25 == 0 or
            index + 1 == len(val_df)
        ):

            elapsed = (
                time.time() -
                start_time
            )

            print(
                f"Processed "
                f"{index+1:3d}/304"
                f" | elapsed="
                f"{elapsed/60:.1f} min"
            )


b1_logits = torch.cat(
    all_logits,
    dim=0
).numpy()

b1_labels = np.asarray(
    all_labels,
    dtype=np.int64
)

b1_uids = np.asarray(
    all_uids,
    dtype=str
)


print()
print(
    "Logits:",
    b1_logits.shape
)


# ==================================================================================================
# 12. CRITICAL B1 REPRODUCTION CHECK
# ==================================================================================================

section(
    "12. CRITICAL B1 REPRODUCTION CHECK"
)

b1_top1 = topk_accuracy(
    b1_logits,
    b1_labels,
    1
)

b1_top5 = topk_accuracy(
    b1_logits,
    b1_labels,
    5
)

expected_top1 = float(
    checkpoint[
        "best_top1"
    ]
)

expected_top5 = float(
    checkpoint[
        "best_top5"
    ]
)

print(
    f"Reproduced Top1 : "
    f"{b1_top1:.6f}%"
)

print(
    f"Frozen Top1     : "
    f"{expected_top1:.6f}%"
)

print(
    f"Difference      : "
    f"{abs(b1_top1-expected_top1):.6f} pp"
)

print()

print(
    f"Reproduced Top5 : "
    f"{b1_top5:.6f}%"
)

print(
    f"Frozen Top5     : "
    f"{expected_top5:.6f}%"
)

print(
    f"Difference      : "
    f"{abs(b1_top5-expected_top5):.6f} pp"
)


# Require near reproduction.
#
# If this fails, STOP. Do not fuse.

if abs(
    b1_top1 -
    expected_top1
) > 1.0:

    raise RuntimeError(
        "Exact B1 reproduction still differs "
        "by >1 percentage point. "
        "STOP — do not perform fusion."
    )


print()
print(
    "[PASS] Frozen B1 performance reproduced."
)


np.savez_compressed(
    B1_LOGITS_FILE,
    logits=b1_logits,
    labels=b1_labels,
    uids=b1_uids
)


# ==================================================================================================
# 13. LOAD B2B
# ==================================================================================================

section(
    "13. LOADING FROZEN B2B LANDMARK LOGITS"
)

b2 = np.load(
    B2B_LOGITS,
    allow_pickle=False
)

b2_logits = np.asarray(
    b2["logits"],
    dtype=np.float32
)

b2_labels = np.asarray(
    b2["labels"],
    dtype=np.int64
)

b2_uids = np.asarray(
    b2["uids"]
).astype(str)

print(
    "B2B logits:",
    b2_logits.shape
)

assert b2_logits.shape == (
    304,
    215
)

assert len(
    np.unique(
        b2_uids
    )
) == 304


# ==================================================================================================
# 14. STRICT ALIGNMENT
# ==================================================================================================

section(
    "14. STRICT UID + LABEL ALIGNMENT"
)

b1_map = {
    uid: i
    for i, uid
    in enumerate(b1_uids)
}

b2_map = {
    uid: i
    for i, uid
    in enumerate(b2_uids)
}

manifest_uids = (
    val_df[
        "uid"
    ]
    .astype(str)
    .to_numpy()
)

manifest_labels = (
    val_df[
        "class_id"
    ]
    .astype(int)
    .to_numpy()
)


if set(b1_map) != set(b2_map):

    raise RuntimeError(
        "B1/B2B UID sets differ."
    )

if set(b1_map) != set(
    manifest_uids
):

    raise RuntimeError(
        "Model UIDs differ from manifest."
    )


rgb_logits = np.stack(
    [
        b1_logits[
            b1_map[uid]
        ]
        for uid in manifest_uids
    ]
)

land_logits = np.stack(
    [
        b2_logits[
            b2_map[uid]
        ]
        for uid in manifest_uids
    ]
)

rgb_labels = np.asarray(
    [
        b1_labels[
            b1_map[uid]
        ]
        for uid in manifest_uids
    ]
)

land_labels = np.asarray(
    [
        b2_labels[
            b2_map[uid]
        ]
        for uid in manifest_uids
    ]
)


assert np.array_equal(
    rgb_labels,
    manifest_labels
)

assert np.array_equal(
    land_labels,
    manifest_labels
)

labels = manifest_labels


print(
    "[PASS] 304 UIDs aligned."
)

print(
    "[PASS] 304 labels aligned."
)

print(
    "[PASS] Class space = 215."
)


np.savez_compressed(
    ALIGNED_FILE,
    uids=manifest_uids,
    labels=labels,
    rgb_logits=rgb_logits,
    landmark_logits=land_logits
)


# ==================================================================================================
# 15. INDIVIDUAL PERFORMANCE
# ==================================================================================================

section(
    "15. INDIVIDUAL MODEL PERFORMANCE"
)

rgb_top1 = topk_accuracy(
    rgb_logits,
    labels,
    1
)

rgb_top5 = topk_accuracy(
    rgb_logits,
    labels,
    5
)

land_top1 = topk_accuracy(
    land_logits,
    labels,
    1
)

land_top5 = topk_accuracy(
    land_logits,
    labels,
    5
)

print(
    f"B1 RGB Top1       : "
    f"{rgb_top1:.3f}%"
)

print(
    f"B1 RGB Top5       : "
    f"{rgb_top5:.3f}%"
)

print(
    f"B2B Landmark Top1 : "
    f"{land_top1:.3f}%"
)

print(
    f"B2B Landmark Top5 : "
    f"{land_top5:.3f}%"
)


# ==================================================================================================
# 16. COMPLEMENTARITY
# ==================================================================================================

section(
    "16. COMPLEMENTARITY ANALYSIS"
)

rgb_pred = rgb_logits.argmax(
    axis=1
)

land_pred = land_logits.argmax(
    axis=1
)

rgb_correct = (
    rgb_pred ==
    labels
)

land_correct = (
    land_pred ==
    labels
)

both = (
    rgb_correct &
    land_correct
)

rgb_only = (
    rgb_correct &
    ~land_correct
)

land_only = (
    ~rgb_correct &
    land_correct
)

neither = (
    ~rgb_correct &
    ~land_correct
)


n_both = int(
    both.sum()
)

n_rgb_only = int(
    rgb_only.sum()
)

n_land_only = int(
    land_only.sum()
)

n_neither = int(
    neither.sum()
)

oracle_correct = int(
    (
        rgb_correct |
        land_correct
    ).sum()
)

oracle_top1 = (
    100.0 *
    oracle_correct /
    304
)


print(
    "Both correct      :",
    n_both
)

print(
    "RGB only correct  :",
    n_rgb_only
)

print(
    "Landmark only     :",
    n_land_only
)

print(
    "Neither correct   :",
    n_neither
)

print()

print(
    "RGB total correct :",
    int(
        rgb_correct.sum()
    )
)

print(
    "Landmark correct  :",
    int(
        land_correct.sum()
    )
)

print()

print(
    f"ORACLE correct    : "
    f"{oracle_correct}/304"
)

print(
    f"ORACLE Top1       : "
    f"{oracle_top1:.3f}%"
)


complementarity_df = pd.DataFrame(
    {
        "uid":
            manifest_uids,

        "gloss":
            val_df[
                "gloss"
            ].astype(str),

        "true_class":
            labels,

        "rgb_pred":
            rgb_pred,

        "landmark_pred":
            land_pred,

        "rgb_correct":
            rgb_correct.astype(int),

        "landmark_correct":
            land_correct.astype(int),

        "both_correct":
            both.astype(int),

        "rgb_only":
            rgb_only.astype(int),

        "landmark_only":
            land_only.astype(int),

        "neither":
            neither.astype(int),
    }
)

complementarity_df.to_csv(
    COMPLEMENTARITY_FILE,
    index=False
)


# ==================================================================================================
# 17. FUSION SEARCH
#
# Development-only validation search.
# ==================================================================================================

section(
    "17. VALIDATION FUSION SEARCH"
)

TEMPERATURES = [
    0.50,
    0.75,
    1.00,
    1.25,
    1.50,
    2.00,
    2.50,
    3.00,
    4.00,
]

ALPHAS = np.round(
    np.arange(
        0.0,
        1.0001,
        0.025
    ),
    3
)

results = []


def record(
    method,
    tr,
    tl,
    alpha,
    scores
):

    results.append(
        {
            "method":
                method,

            "rgb_temperature":
                tr,

            "landmark_temperature":
                tl,

            "rgb_weight":
                alpha,

            "landmark_weight":
                1.0 - alpha,

            "top1":
                topk_accuracy(
                    scores,
                    labels,
                    1
                ),

            "top5":
                topk_accuracy(
                    scores,
                    labels,
                    5
                )
        }
    )


# --------------------------------------------------------------------------------------------------
# Probability fusion
# --------------------------------------------------------------------------------------------------

print(
    "Probability fusion..."
)

for tr in TEMPERATURES:

    rp = softmax_np(
        rgb_logits,
        tr
    )

    for tl in TEMPERATURES:

        lp = softmax_np(
            land_logits,
            tl
        )

        for alpha in ALPHAS:

            scores = (
                alpha * rp +
                (1.0-alpha) * lp
            )

            record(
                "probability",
                tr,
                tl,
                alpha,
                scores
            )


# --------------------------------------------------------------------------------------------------
# Log-probability fusion
# --------------------------------------------------------------------------------------------------

print(
    "Log-probability fusion..."
)

for tr in TEMPERATURES:

    rp = log_softmax_np(
        rgb_logits,
        tr
    )

    for tl in TEMPERATURES:

        lp = log_softmax_np(
            land_logits,
            tl
        )

        for alpha in ALPHAS:

            scores = (
                alpha * rp +
                (1.0-alpha) * lp
            )

            record(
                "log_probability",
                tr,
                tl,
                alpha,
                scores
            )


# ==================================================================================================
# 18. RANK FUSION
# ==================================================================================================

print(
    "Rank fusion..."
)


def rank_scores(logits):

    order = np.argsort(
        -logits,
        axis=1
    )

    ranks = np.empty_like(
        order,
        dtype=np.float32
    )

    rows = np.arange(
        logits.shape[0]
    )[:, None]

    ranks[
        rows,
        order
    ] = np.arange(
        logits.shape[1],
        dtype=np.float32
    )[None, :]

    return (
        1.0 -
        ranks /
        (
            logits.shape[1] -
            1
        )
    )


rgb_rank = rank_scores(
    rgb_logits
)

land_rank = rank_scores(
    land_logits
)


for alpha in ALPHAS:

    scores = (
        alpha * rgb_rank +
        (1.0-alpha) * land_rank
    )

    record(
        "rank",
        1.0,
        1.0,
        alpha,
        scores
    )


# ==================================================================================================
# 19. RESULTS
# ==================================================================================================

section(
    "19. FUSION RESULTS"
)

search_df = pd.DataFrame(
    results
)

search_df = (
    search_df
    .sort_values(
        [
            "top1",
            "top5"
        ],
        ascending=[
            False,
            False
        ]
    )
    .reset_index(
        drop=True
    )
)

search_df.to_csv(
    SEARCH_FILE,
    index=False
)

print(
    "Configurations:",
    len(search_df)
)

print()
print(
    search_df.head(
        20
    ).to_string(
        index=False
    )
)

best = search_df.iloc[0]

best_method = str(
    best["method"]
)

best_tr = float(
    best["rgb_temperature"]
)

best_tl = float(
    best["landmark_temperature"]
)

best_alpha = float(
    best["rgb_weight"]
)


# ==================================================================================================
# 20. RECONSTRUCT BEST
# ==================================================================================================

if best_method == "probability":

    a = softmax_np(
        rgb_logits,
        best_tr
    )

    b = softmax_np(
        land_logits,
        best_tl
    )

elif best_method == "log_probability":

    a = log_softmax_np(
        rgb_logits,
        best_tr
    )

    b = log_softmax_np(
        land_logits,
        best_tl
    )

elif best_method == "rank":

    a = rgb_rank
    b = land_rank

else:

    raise RuntimeError(
        best_method
    )


best_scores = (
    best_alpha * a +
    (1.0-best_alpha) * b
)

fusion_top1 = topk_accuracy(
    best_scores,
    labels,
    1
)

fusion_top5 = topk_accuracy(
    best_scores,
    labels,
    5
)

fusion_pred = best_scores.argmax(
    axis=1
)


# ==================================================================================================
# 21. FINAL COMPARISON
# ==================================================================================================

section(
    "21. BEST B2C VALIDATION FUSION"
)

print(
    "Method              :",
    best_method
)

print(
    "RGB temperature     :",
    best_tr
)

print(
    "Landmark temperature:",
    best_tl
)

print(
    "RGB weight          :",
    f"{best_alpha:.3f}"
)

print(
    "Landmark weight     :",
    f"{1.0-best_alpha:.3f}"
)

print()

print(
    f"B1 Top1             : "
    f"{rgb_top1:.3f}%"
)

print(
    f"B2B Top1            : "
    f"{land_top1:.3f}%"
)

print(
    f"Fusion Top1         : "
    f"{fusion_top1:.3f}%"
)

print(
    f"Top1 gain vs B1     : "
    f"{fusion_top1-rgb_top1:+.3f} pp"
)

print()

print(
    f"B1 Top5             : "
    f"{rgb_top5:.3f}%"
)

print(
    f"B2B Top5            : "
    f"{land_top5:.3f}%"
)

print(
    f"Fusion Top5         : "
    f"{fusion_top5:.3f}%"
)

print(
    f"Top5 gain vs B1     : "
    f"{fusion_top5-rgb_top5:+.3f} pp"
)

print()

print(
    f"Oracle Top1         : "
    f"{oracle_top1:.3f}%"
)


# ==================================================================================================
# 22. SAVE PREDICTIONS
# ==================================================================================================

pred_df = pd.DataFrame(
    {
        "uid":
            manifest_uids,

        "gloss":
            val_df[
                "gloss"
            ].astype(str),

        "true_class":
            labels,

        "rgb_pred":
            rgb_pred,

        "landmark_pred":
            land_pred,

        "fusion_pred":
            fusion_pred,

        "rgb_correct":
            rgb_correct.astype(int),

        "landmark_correct":
            land_correct.astype(int),

        "fusion_correct":
            (
                fusion_pred ==
                labels
            ).astype(int)
    }
)

pred_df.to_csv(
    PREDICTIONS_FILE,
    index=False
)


# ==================================================================================================
# 23. SUMMARY
# ==================================================================================================

summary = {

    "experiment":
        "L20D-B2C_CORRECTED",

    "validation_samples":
        304,

    "classes":
        215,

    "B1": {
        "top1":
            float(rgb_top1),

        "top5":
            float(rgb_top5)
    },

    "B2B": {
        "top1":
            float(land_top1),

        "top5":
            float(land_top5)
    },

    "complementarity": {
        "both_correct":
            n_both,

        "rgb_only":
            n_rgb_only,

        "landmark_only":
            n_land_only,

        "neither":
            n_neither,

        "oracle_correct":
            oracle_correct,

        "oracle_top1":
            float(oracle_top1)
    },

    "fusion": {
        "method":
            best_method,

        "rgb_temperature":
            best_tr,

        "landmark_temperature":
            best_tl,

        "rgb_weight":
            best_alpha,

        "landmark_weight":
            1.0-best_alpha,

        "top1":
            float(fusion_top1),

        "top5":
            float(fusion_top5),

        "top1_gain":
            float(
                fusion_top1 -
                rgb_top1
            ),

        "top5_gain":
            float(
                fusion_top5 -
                rgb_top5
            )
    },

    "locked_test_loaded":
        False,

    "locked_test_evaluated":
        False,

    "selection_note":
        (
            "Fusion parameters selected "
            "on validation; result is "
            "development accuracy."
        )
}


with open(
    SUMMARY_FILE,
    "w"
) as f:

    json.dump(
        summary,
        f,
        indent=2
    )


# ==================================================================================================
# 24. MASTER FREEZE
# ==================================================================================================

freeze = {

    "experiment":
        "L20D-B2C_CORRECTED",

    "status":
        "FROZEN_VALIDATION_SELECTED",

    "B1_checkpoint":
        str(B1_MODEL),

    "B2B_logits":
        str(B2B_LOGITS),

    "summary":
        summary,

    "integrity": {
        "b1_checkpoint_strict_load":
            True,

        "b1_three_view_validation":
            True,

        "uid_alignment":
            True,

        "label_alignment":
            True,

        "locked_test_used":
            False
    }
}


with open(
    MASTER_FREEZE,
    "w"
) as f:

    json.dump(
        freeze,
        f,
        indent=2
    )


# ==================================================================================================
# 25. HASHES
# ==================================================================================================

section(
    "25. ARTIFACT HASHES"
)

for name, path in {

    "b1_logits":
        B1_LOGITS_FILE,

    "aligned":
        ALIGNED_FILE,

    "complementarity":
        COMPLEMENTARITY_FILE,

    "search":
        SEARCH_FILE,

    "predictions":
        PREDICTIONS_FILE,

    "summary":
        SUMMARY_FILE,

    "master_freeze":
        MASTER_FREEZE

}.items():

    print(
        f"{name:<20}: "
        f"{sha256_file(path)}"
    )


# ==================================================================================================
# 26. COMPLETE
# ==================================================================================================

section(
    "L20D-B2C CORRECTED COMPLETE"
)

print(
    f"B1 RGB Top1       : "
    f"{rgb_top1:.3f}%"
)

print(
    f"B1 RGB Top5       : "
    f"{rgb_top5:.3f}%"
)

print(
    f"B2B Landmark Top1 : "
    f"{land_top1:.3f}%"
)

print(
    f"B2B Landmark Top5 : "
    f"{land_top5:.3f}%"
)

print()

print(
    f"Both correct       : "
    f"{n_both}"
)

print(
    f"RGB-only correct   : "
    f"{n_rgb_only}"
)

print(
    f"Landmark-only      : "
    f"{n_land_only}"
)

print(
    f"Neither            : "
    f"{n_neither}"
)

print(
    f"Oracle Top1        : "
    f"{oracle_top1:.3f}%"
)

print()

print(
    f"Best fusion method : "
    f"{best_method}"
)

print(
    f"RGB weight         : "
    f"{best_alpha:.3f}"
)

print(
    f"Landmark weight    : "
    f"{1-best_alpha:.3f}"
)

print(
    f"Fusion Top1        : "
    f"{fusion_top1:.3f}%"
)

print(
    f"Fusion Top5        : "
    f"{fusion_top5:.3f}%"
)

print(
    f"Top1 gain          : "
    f"{fusion_top1-rgb_top1:+.3f} pp"
)

print(
    f"Top5 gain          : "
    f"{fusion_top5-rgb_top5:+.3f} pp"
)

print()

print(
    "Locked test loaded    : NO"
)

print(
    "Locked test evaluated : NO"
)

print()
print(
    "NOTE: Fusion parameters were "
    "selected on validation."
)

print(
    "This is NOT final-test accuracy."
)

print()

print(
    "Master freeze:"
)

print(
    MASTER_FREEZE
)

print("=" * 120)


L20D-B2C CORRECTED — PROTOCOL
Validation : 304
Classes    : 215
Locked     : 10
Overlap    : 0

[PASS] Correct 304-validation protocol.
[LOCK] Locked test will NOT be decoded.

7. LOADING EXACT FROZEN B1
Checkpoint loaded with strict=True.
Missing keys    : []
Unexpected keys : []

Checkpoint epoch : 43
Checkpoint stage : FINAL6
Frozen Top1      : 33.88157894736842
Frozen Top5      : 52.96052631578947
L20C loaded      : False
B0 loaded        : False
Locked used      : False

[PASS] Exact B1 architecture restored.

11. EXTRACTING B1 3-VIEW VALIDATION LOGITS
Processed  25/304 | elapsed=0.1 min
Processed  50/304 | elapsed=0.3 min
Processed  75/304 | elapsed=0.4 min
Processed 100/304 | elapsed=0.5 min
Processed 125/304 | elapsed=0.7 min
Processed 150/304 | elapsed=0.8 min
Processed 175/304 | elapsed=0.9 min
Processed 200/304 | elapsed=1.1 min
Processed 225/304 | elapsed=1.2 min
Processed 250/304 | elapsed=1.3 min
Processed 275/304 | elapsed=1.4 min
Processed 300/304 | elapsed=1.6 min
Pro

In [7]:
# ==========================================================================================
# L20D — FULL CISLR 215-CLASS EVALUATION — UPDATED
#
# Evaluates:
#   B1  = RGB VideoMAE
#   B2B = Landmark BiGRU + Attention
#   B2C = RGB + Landmark Fusion
#
# Produces:
#   ✓ Top-1 accuracy
#   ✓ Top-5 accuracy
#   ✓ Confusion matrix
#   ✓ Normalized confusion matrix
#   ✓ Per-class Top-1 / Top-5
#   ✓ Macro accuracy
#   ✓ Training-support analysis
#   ✓ Generalization analysis
#   ✓ RGB/Landmark complementarity
#   ✓ Fusion rescue analysis
#   ✓ Most common confusions
#   ✓ Figures
#   ✓ CSV/JSON reports
#   ✓ Master freeze
#
# IMPORTANT:
#   LOCKED 10-VIDEO TEST IS NEVER LOADED FOR INFERENCE.
#
# NOTE:
#   B2C fusion parameters were selected on validation.
#   Therefore B2C validation accuracy is a DEVELOPMENT / MODEL-SELECTION result.
# ==========================================================================================

import json
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ==========================================================================================
# 1. CONFIGURATION
# ==========================================================================================

NUM_CLASSES = 215

REQUESTED_TRAIN_TARGET = 90.0
REQUESTED_VALIDATION_TARGET = 80.0


# ==========================================================================================
# 2. PATHS
# ==========================================================================================

ROOT = Path(
    "/home/dipshikha/Music/cao/CISLR_RESEARCH/"
    "CISLR_LANDMARK_MODEL"
)


# ------------------------------------------------------------------------------------------
# Corrected L20B protocol
# ------------------------------------------------------------------------------------------

L20B = (
    ROOT /
    "audit" /
    "l20b_corrected_strict_protocol_freeze"
)

MANIFEST_ROOT = (
    L20B /
    "manifests"
)


TRAIN_MANIFEST = (
    MANIFEST_ROOT /
    "l20b_corrected_train_manifest.csv"
)

VAL_MANIFEST = (
    MANIFEST_ROOT /
    "l20b_corrected_validation_manifest.csv"
)

LOCKED_MANIFEST = (
    MANIFEST_ROOT /
    "l20b_permanent_l19_locked_test_manifest.csv"
)

CLASS_MAPPING = (
    MANIFEST_ROOT /
    "l20b_corrected_class_mapping.csv"
)


# ------------------------------------------------------------------------------------------
# B2 multimodal branch
# ------------------------------------------------------------------------------------------

B2_ROOT = (
    ROOT /
    "audit" /
    "l20d_b2_multimodal_215class"
)


B2C_ROOT = (
    B2_ROOT /
    "b2c_rgb_landmark_fusion_corrected"
)


ALIGNED_LOGITS = (
    B2C_ROOT /
    "b2c_aligned_logits.npz"
)


B2C_SUMMARY = (
    B2C_ROOT /
    "b2c_summary.json"
)


# ------------------------------------------------------------------------------------------
# L20D evaluation output
# ------------------------------------------------------------------------------------------

OUTPUT = (
    ROOT /
    "audit" /
    "l20d_full_cislr_evaluation"
)

REPORTS = (
    OUTPUT /
    "reports"
)

FIGURES = (
    OUTPUT /
    "figures"
)

REPORTS.mkdir(
    parents=True,
    exist_ok=True
)

FIGURES.mkdir(
    parents=True,
    exist_ok=True
)


# ==========================================================================================
# 3. HELPERS
# ==========================================================================================

def section(title):

    print()
    print("=" * 115)
    print(title)
    print("=" * 115)


def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        while True:

            block = f.read(
                1024 * 1024
            )

            if not block:
                break

            h.update(block)

    return h.hexdigest()


def topk_accuracy(
    scores,
    labels,
    k
):

    indices = np.argpartition(
        scores,
        -k,
        axis=1
    )[:, -k:]

    correct = (
        indices ==
        labels[:, None]
    ).any(axis=1)

    return (
        100.0 *
        float(
            correct.mean()
        )
    )


def confusion_matrix_np(
    true_labels,
    predicted_labels,
    num_classes
):

    matrix = np.zeros(
        (
            num_classes,
            num_classes
        ),
        dtype=np.int64
    )

    for true_id, pred_id in zip(
        true_labels,
        predicted_labels
    ):

        matrix[
            int(true_id),
            int(pred_id)
        ] += 1

    return matrix


def softmax_np(
    logits,
    temperature=1.0
):

    temperature = max(
        float(temperature),
        1e-8
    )

    x = (
        logits /
        temperature
    )

    x = (
        x -
        x.max(
            axis=1,
            keepdims=True
        )
    )

    ex = np.exp(x)

    denominator = np.clip(
        ex.sum(
            axis=1,
            keepdims=True
        ),
        1e-12,
        None
    )

    return (
        ex /
        denominator
    )


def log_softmax_np(
    logits,
    temperature=1.0
):

    probabilities = softmax_np(
        logits,
        temperature
    )

    return np.log(
        np.clip(
            probabilities,
            1e-12,
            1.0
        )
    )


# ==========================================================================================
# 4. HEADER
# ==========================================================================================

section(
    "L20D — FULL CISLR 215-CLASS EVALUATION"
)

print(
    "Evaluation population : 215 classes"
)

print(
    "Expected train        : 710"
)

print(
    "Expected validation   : 304"
)

print(
    "Locked test           : 10"
)

print(
    "Locked test inference : DISABLED"
)


# ==========================================================================================
# 5. VERIFY REQUIRED FILES
# ==========================================================================================

section(
    "5. VERIFY REQUIRED ARTIFACTS"
)


required_files = {

    "Train manifest":
        TRAIN_MANIFEST,

    "Validation manifest":
        VAL_MANIFEST,

    "Locked manifest":
        LOCKED_MANIFEST,

    "Class mapping":
        CLASS_MAPPING,

    "B2C aligned logits":
        ALIGNED_LOGITS,

    "B2C summary":
        B2C_SUMMARY,
}


for name, path in required_files.items():

    if not path.exists():

        raise FileNotFoundError(
            f"{name} not found:\n{path}"
        )

    print(
        f"[FOUND] {name:<25}: "
        f"{path}"
    )


print()
print(
    "[PASS] Required artifacts found."
)


# ==========================================================================================
# 6. LOAD AUTHORITATIVE PROTOCOL
# ==========================================================================================

section(
    "6. LOAD AUTHORITATIVE L20B PROTOCOL"
)


train_df = pd.read_csv(
    TRAIN_MANIFEST
)

val_df = pd.read_csv(
    VAL_MANIFEST
)

locked_df = pd.read_csv(
    LOCKED_MANIFEST
)

mapping_df = pd.read_csv(
    CLASS_MAPPING
)


print(
    "Train rows       :",
    len(train_df)
)

print(
    "Train UIDs       :",
    train_df[
        "uid"
    ].nunique()
)

print(
    "Validation rows  :",
    len(val_df)
)

print(
    "Validation UIDs  :",
    val_df[
        "uid"
    ].nunique()
)

print(
    "Locked rows      :",
    len(locked_df)
)

print(
    "Locked UIDs      :",
    locked_df[
        "uid"
    ].nunique()
)

print(
    "Class mapping    :",
    len(mapping_df)
)


# ------------------------------------------------------------------------------------------
# Expected counts
# ------------------------------------------------------------------------------------------

assert len(train_df) == 710
assert len(val_df) == 304
assert len(locked_df) == 10
assert len(mapping_df) == NUM_CLASSES

assert train_df["uid"].nunique() == 710
assert val_df["uid"].nunique() == 304
assert locked_df["uid"].nunique() == 10

assert train_df["class_id"].nunique() == NUM_CLASSES
assert val_df["class_id"].nunique() == NUM_CLASSES


# ------------------------------------------------------------------------------------------
# Split overlap checks
# ------------------------------------------------------------------------------------------

train_uids_set = set(
    train_df[
        "uid"
    ].astype(str)
)

val_uids_set = set(
    val_df[
        "uid"
    ].astype(str)
)

locked_uids_set = set(
    locked_df[
        "uid"
    ].astype(str)
)


train_val_overlap = (
    train_uids_set &
    val_uids_set
)

train_locked_overlap = (
    train_uids_set &
    locked_uids_set
)

val_locked_overlap = (
    val_uids_set &
    locked_uids_set
)


print()
print(
    "Train/Val overlap    :",
    len(
        train_val_overlap
    )
)

print(
    "Train/Locked overlap :",
    len(
        train_locked_overlap
    )
)

print(
    "Val/Locked overlap   :",
    len(
        val_locked_overlap
    )
)


assert len(
    train_val_overlap
) == 0

assert len(
    train_locked_overlap
) == 0

assert len(
    val_locked_overlap
) == 0


print()
print(
    "[PASS] Strict 710 / 304 / 10 protocol."
)

print(
    "[PASS] All splits disjoint."
)

print(
    "[PASS] Locked test remains isolated."
)


# ==========================================================================================
# 7. LOAD FROZEN B2C VALIDATION LOGITS
#
# UPDATED FIX:
# allow_pickle=True is required because UID strings were stored as an object array.
# This NPZ is a trusted locally generated project artifact.
# ==========================================================================================

section(
    "7. LOAD FROZEN B2C MULTIMODAL OUTPUT"
)


data = np.load(
    ALIGNED_LOGITS,
    allow_pickle=True
)


print(
    "NPZ keys:",
    list(
        data.keys()
    )
)


required_keys = {

    "uids",
    "labels",
    "rgb_logits",
    "landmark_logits"
}


missing_keys = (
    required_keys -
    set(
        data.keys()
    )
)


if missing_keys:

    raise RuntimeError(
        "Missing arrays in B2C NPZ: "
        f"{missing_keys}"
    )


# ------------------------------------------------------------------------------------------
# Load arrays
# ------------------------------------------------------------------------------------------

uids = np.asarray(
    data[
        "uids"
    ]
).astype(str)


labels = np.asarray(
    data[
        "labels"
    ],
    dtype=np.int64
)


rgb_logits = np.asarray(
    data[
        "rgb_logits"
    ],
    dtype=np.float32
)


landmark_logits = np.asarray(
    data[
        "landmark_logits"
    ],
    dtype=np.float32
)


print()
print(
    "UIDs            :",
    uids.shape
)

print(
    "Labels          :",
    labels.shape
)

print(
    "RGB logits      :",
    rgb_logits.shape
)

print(
    "Landmark logits :",
    landmark_logits.shape
)


# ------------------------------------------------------------------------------------------
# Structural checks
# ------------------------------------------------------------------------------------------

assert uids.shape == (
    304,
)

assert labels.shape == (
    304,
)

assert rgb_logits.shape == (
    304,
    NUM_CLASSES
)

assert landmark_logits.shape == (
    304,
    NUM_CLASSES
)

assert len(
    np.unique(
        uids
    )
) == 304


# ------------------------------------------------------------------------------------------
# Verify authoritative validation order
# ------------------------------------------------------------------------------------------

manifest_uids = (
    val_df[
        "uid"
    ]
    .astype(str)
    .to_numpy()
)


manifest_labels = (
    val_df[
        "class_id"
    ]
    .astype(int)
    .to_numpy()
)


if not np.array_equal(
    uids,
    manifest_uids
):

    raise RuntimeError(
        "B2C UID order does not match "
        "authoritative L20B validation manifest."
    )


if not np.array_equal(
    labels,
    manifest_labels
):

    raise RuntimeError(
        "B2C labels do not match "
        "authoritative L20B validation manifest."
    )


# ------------------------------------------------------------------------------------------
# Numerical sanity
# ------------------------------------------------------------------------------------------

if not np.isfinite(
    rgb_logits
).all():

    raise RuntimeError(
        "RGB logits contain NaN/Inf."
    )


if not np.isfinite(
    landmark_logits
).all():

    raise RuntimeError(
        "Landmark logits contain NaN/Inf."
    )


print()
print(
    "[PASS] 304 unique validation samples."
)

print(
    "[PASS] UID order verified."
)

print(
    "[PASS] Labels verified."
)

print(
    "[PASS] RGB logits finite."
)

print(
    "[PASS] Landmark logits finite."
)

print(
    "[PASS] Locked test not loaded."
)


# ==========================================================================================
# 8. LOAD FROZEN B2C FUSION CONFIGURATION
# ==========================================================================================

section(
    "8. LOAD FROZEN B2C FUSION CONFIGURATION"
)


with open(
    B2C_SUMMARY,
    "r"
) as f:

    b2c_summary = json.load(f)


fusion_cfg = (
    b2c_summary[
        "fusion"
    ]
)


fusion_method = str(
    fusion_cfg[
        "method"
    ]
)


rgb_temperature = float(
    fusion_cfg[
        "rgb_temperature"
    ]
)


landmark_temperature = float(
    fusion_cfg[
        "landmark_temperature"
    ]
)


rgb_weight = float(
    fusion_cfg[
        "rgb_weight"
    ]
)


landmark_weight = float(
    fusion_cfg[
        "landmark_weight"
    ]
)


print(
    "Method               :",
    fusion_method
)

print(
    "RGB temperature      :",
    rgb_temperature
)

print(
    "Landmark temperature :",
    landmark_temperature
)

print(
    "RGB weight           :",
    rgb_weight
)

print(
    "Landmark weight      :",
    landmark_weight
)


# ==========================================================================================
# 9. RECONSTRUCT FROZEN B2C FUSION
# ==========================================================================================

section(
    "9. RECONSTRUCT FROZEN B2C FUSION"
)


if fusion_method == "log_probability":

    rgb_component = log_softmax_np(
        rgb_logits,
        rgb_temperature
    )

    landmark_component = log_softmax_np(
        landmark_logits,
        landmark_temperature
    )


elif fusion_method == "probability":

    rgb_component = softmax_np(
        rgb_logits,
        rgb_temperature
    )

    landmark_component = softmax_np(
        landmark_logits,
        landmark_temperature
    )


else:

    raise RuntimeError(
        "Unsupported frozen fusion method: "
        f"{fusion_method}"
    )


fusion_scores = (

    rgb_weight *
    rgb_component

    +

    landmark_weight *
    landmark_component
)


if not np.isfinite(
    fusion_scores
).all():

    raise RuntimeError(
        "Fusion scores contain NaN/Inf."
    )


print(
    "Fusion scores:",
    fusion_scores.shape
)

print(
    "[PASS] Frozen fusion reconstructed."
)


# ==========================================================================================
# 10. OVERALL VALIDATION ACCURACY
# ==========================================================================================

section(
    "10. OVERALL VALIDATION ACCURACY"
)


model_scores = {

    "B1_RGB":
        rgb_logits,

    "B2B_LANDMARK":
        landmark_logits,

    "B2C_FUSION":
        fusion_scores
}


overall_rows = []


for model_name, scores in model_scores.items():

    top1 = topk_accuracy(
        scores,
        labels,
        1
    )

    top5 = topk_accuracy(
        scores,
        labels,
        5
    )

    model_pred = scores.argmax(
        axis=1
    )

    correct_top1 = int(
        (
            model_pred ==
            labels
        ).sum()
    )


    overall_rows.append(
        {
            "model":
                model_name,

            "samples":
                len(labels),

            "correct_top1":
                correct_top1,

            "top1_accuracy":
                top1,

            "top5_accuracy":
                top5
        }
    )


    print()
    print(
        model_name
    )

    print(
        f"  Top-1 : "
        f"{top1:.3f}% "
        f"({correct_top1}/304)"
    )

    print(
        f"  Top-5 : "
        f"{top5:.3f}%"
    )


overall_df = pd.DataFrame(
    overall_rows
)


overall_metrics_file = (
    REPORTS /
    "overall_validation_metrics.csv"
)


overall_df.to_csv(
    overall_metrics_file,
    index=False
)


# ==========================================================================================
# 11. B2C PREDICTIONS
# ==========================================================================================

fusion_pred = fusion_scores.argmax(
    axis=1
)


fusion_top5_indices = np.argsort(
    -fusion_scores,
    axis=1
)[:, :5]


fusion_top5_correct = (
    fusion_top5_indices ==
    labels[:, None]
).any(
    axis=1
)


fusion_correct = (
    fusion_pred ==
    labels
)


fusion_top1 = (
    100.0 *
    fusion_correct.mean()
)


fusion_top5 = (
    100.0 *
    fusion_top5_correct.mean()
)


# ==========================================================================================
# 12. CONFUSION MATRIX
# ==========================================================================================

section(
    "12. B2C CONFUSION MATRIX"
)


cm = confusion_matrix_np(
    labels,
    fusion_pred,
    NUM_CLASSES
)


print(
    "Shape           :",
    cm.shape
)

print(
    "Total samples   :",
    int(
        cm.sum()
    )
)

print(
    "Correct diagonal:",
    int(
        np.trace(
            cm
        )
    )
)


assert cm.shape == (
    NUM_CLASSES,
    NUM_CLASSES
)

assert int(
    cm.sum()
) == 304


confusion_npy_file = (
    REPORTS /
    "b2c_confusion_matrix.npy"
)


confusion_csv_file = (
    REPORTS /
    "b2c_confusion_matrix.csv"
)


np.save(
    confusion_npy_file,
    cm
)


pd.DataFrame(
    cm
).to_csv(
    confusion_csv_file,
    index=False
)


# ==========================================================================================
# 13. RAW CONFUSION MATRIX FIGURE
# ==========================================================================================

fig = plt.figure(
    figsize=(
        14,
        12
    )
)


plt.imshow(
    cm,
    interpolation="nearest",
    aspect="auto"
)


plt.title(
    "L20D-B2C — 215-Class Validation Confusion Matrix"
)


plt.xlabel(
    "Predicted Class"
)


plt.ylabel(
    "True Class"
)


plt.colorbar()


plt.tight_layout()


raw_cm_figure = (
    FIGURES /
    "b2c_confusion_matrix.png"
)


plt.savefig(
    raw_cm_figure,
    dpi=200
)


plt.close(
    fig
)


print(
    "[SAVED]",
    raw_cm_figure
)


# ==========================================================================================
# 14. NORMALIZED CONFUSION MATRIX
# ==========================================================================================

row_totals = cm.sum(
    axis=1,
    keepdims=True
)


cm_normalized = (
    cm /
    np.maximum(
        row_totals,
        1
    )
)


normalized_cm_file = (
    REPORTS /
    "b2c_confusion_matrix_normalized.csv"
)


pd.DataFrame(
    cm_normalized
).to_csv(
    normalized_cm_file,
    index=False
)


fig = plt.figure(
    figsize=(
        14,
        12
    )
)


plt.imshow(
    cm_normalized,
    interpolation="nearest",
    aspect="auto",
    vmin=0,
    vmax=1
)


plt.title(
    "L20D-B2C — Normalized Validation Confusion Matrix"
)


plt.xlabel(
    "Predicted Class"
)


plt.ylabel(
    "True Class"
)


plt.colorbar()


plt.tight_layout()


normalized_cm_figure = (
    FIGURES /
    "b2c_confusion_matrix_normalized.png"
)


plt.savefig(
    normalized_cm_figure,
    dpi=200
)


plt.close(
    fig
)


print(
    "[SAVED]",
    normalized_cm_figure
)


# ==========================================================================================
# 15. CLASS MAPPING
# ==========================================================================================

section(
    "15. CLASS MAPPING"
)


class_to_gloss = {

    int(row["class_id"]):
        str(row["gloss"])

    for _, row in mapping_df.iterrows()
}


print(
    "Mapped classes:",
    len(
        class_to_gloss
    )
)


assert len(
    class_to_gloss
) == NUM_CLASSES


# ==========================================================================================
# 16. TRAINING SUPPORT PER CLASS
# ==========================================================================================

train_counts = (
    train_df
    .groupby(
        "class_id"
    )
    .size()
    .to_dict()
)


validation_counts = (
    val_df
    .groupby(
        "class_id"
    )
    .size()
    .to_dict()
)


print()
print(
    "Train support min:",
    min(
        train_counts.values()
    )
)

print(
    "Train support max:",
    max(
        train_counts.values()
    )
)

print(
    "Mean train support:",
    np.mean(
        list(
            train_counts.values()
        )
    )
)


# ==========================================================================================
# 17. PER-CLASS ACCURACY
# ==========================================================================================

section(
    "17. PER-CLASS ACCURACY"
)


per_class_rows = []


for class_id in range(
    NUM_CLASSES
):

    mask = (
        labels ==
        class_id
    )


    validation_support = int(
        mask.sum()
    )


    if validation_support == 0:

        continue


    class_true = labels[
        mask
    ]


    class_pred = fusion_pred[
        mask
    ]


    top1_correct_count = int(
        (
            class_pred ==
            class_true
        ).sum()
    )


    top5_correct_count = int(
        fusion_top5_correct[
            mask
        ].sum()
    )


    class_top1 = (
        100.0 *
        top1_correct_count /
        validation_support
    )


    class_top5 = (
        100.0 *
        top5_correct_count /
        validation_support
    )


    per_class_rows.append(
        {
            "class_id":
                class_id,

            "gloss":
                class_to_gloss.get(
                    class_id,
                    str(class_id)
                ),

            "train_support":
                int(
                    train_counts.get(
                        class_id,
                        0
                    )
                ),

            "validation_support":
                validation_support,

            "top1_correct":
                top1_correct_count,

            "top1_accuracy":
                class_top1,

            "top5_correct":
                top5_correct_count,

            "top5_accuracy":
                class_top5
        }
    )


per_class_df = pd.DataFrame(
    per_class_rows
)


per_class_file = (
    REPORTS /
    "b2c_per_class_accuracy.csv"
)


per_class_df.to_csv(
    per_class_file,
    index=False
)


macro_top1 = float(
    per_class_df[
        "top1_accuracy"
    ].mean()
)


macro_top5 = float(
    per_class_df[
        "top5_accuracy"
    ].mean()
)


print(
    "Classes evaluated:",
    len(
        per_class_df
    )
)

print(
    "Macro Top-1     :",
    f"{macro_top1:.3f}%"
)

print(
    "Macro Top-5     :",
    f"{macro_top5:.3f}%"
)


# ==========================================================================================
# 18. BEST PERFORMING CLASSES
# ==========================================================================================

section(
    "18. BEST PERFORMING CLASSES"
)


best_classes = (
    per_class_df
    .sort_values(
        [
            "top1_accuracy",
            "top5_accuracy",
            "validation_support"
        ],
        ascending=[
            False,
            False,
            False
        ]
    )
    .head(
        20
    )
)


print(
    best_classes[
        [
            "class_id",
            "gloss",
            "train_support",
            "validation_support",
            "top1_accuracy",
            "top5_accuracy"
        ]
    ].to_string(
        index=False
    )
)


# ==========================================================================================
# 19. LOWEST PERFORMING CLASSES
# ==========================================================================================

section(
    "19. LOWEST PERFORMING CLASSES"
)


worst_classes = (
    per_class_df
    .sort_values(
        [
            "top1_accuracy",
            "top5_accuracy",
            "validation_support"
        ],
        ascending=[
            True,
            True,
            False
        ]
    )
    .head(
        20
    )
)


print(
    worst_classes[
        [
            "class_id",
            "gloss",
            "train_support",
            "validation_support",
            "top1_accuracy",
            "top5_accuracy"
        ]
    ].to_string(
        index=False
    )
)


# ==========================================================================================
# 20. PER-CLASS ACCURACY DISTRIBUTION
# ==========================================================================================

section(
    "20. PER-CLASS ACCURACY DISTRIBUTION"
)


accuracy_ranges = {

    "100%":
        int(
            (
                per_class_df[
                    "top1_accuracy"
                ] == 100
            ).sum()
        ),

    ">=75% and <100%":
        int(
            (
                (
                    per_class_df[
                        "top1_accuracy"
                    ] >= 75
                )
                &
                (
                    per_class_df[
                        "top1_accuracy"
                    ] < 100
                )
            ).sum()
        ),

    ">=50% and <75%":
        int(
            (
                (
                    per_class_df[
                        "top1_accuracy"
                    ] >= 50
                )
                &
                (
                    per_class_df[
                        "top1_accuracy"
                    ] < 75
                )
            ).sum()
        ),

    ">0% and <50%":
        int(
            (
                (
                    per_class_df[
                        "top1_accuracy"
                    ] > 0
                )
                &
                (
                    per_class_df[
                        "top1_accuracy"
                    ] < 50
                )
            ).sum()
        ),

    "0%":
        int(
            (
                per_class_df[
                    "top1_accuracy"
                ] == 0
            ).sum()
        )
}


for name, count in accuracy_ranges.items():

    print(
        f"{name:<20}: "
        f"{count} classes"
    )


# ==========================================================================================
# 21. TRAIN SUPPORT VS GENERALIZATION
# ==========================================================================================

section(
    "21. TRAIN SUPPORT VS GENERALIZATION"
)


support_rows = []


unique_supports = sorted(
    per_class_df[
        "train_support"
    ].unique()
)


for support in unique_supports:

    support_class_df = per_class_df[
        per_class_df[
            "train_support"
        ] == support
    ]


    class_ids = set(
        support_class_df[
            "class_id"
        ].astype(int)
    )


    sample_mask = np.array(
        [
            int(label) in class_ids
            for label in labels
        ],
        dtype=bool
    )


    number_validation_samples = int(
        sample_mask.sum()
    )


    if number_validation_samples == 0:

        continue


    sample_top1 = (
        100.0 *
        (
            fusion_pred[
                sample_mask
            ]
            ==
            labels[
                sample_mask
            ]
        ).mean()
    )


    sample_top5 = (
        100.0 *
        fusion_top5_correct[
            sample_mask
        ].mean()
    )


    macro_group_top1 = float(
        support_class_df[
            "top1_accuracy"
        ].mean()
    )


    macro_group_top5 = float(
        support_class_df[
            "top5_accuracy"
        ].mean()
    )


    support_rows.append(
        {
            "train_examples_per_class":
                int(
                    support
                ),

            "num_classes":
                len(
                    support_class_df
                ),

            "validation_samples":
                number_validation_samples,

            "sample_weighted_top1":
                float(
                    sample_top1
                ),

            "sample_weighted_top5":
                float(
                    sample_top5
                ),

            "macro_class_top1":
                macro_group_top1,

            "macro_class_top5":
                macro_group_top5
        }
    )


support_df = pd.DataFrame(
    support_rows
)


print(
    support_df.to_string(
        index=False
    )
)


support_file = (
    REPORTS /
    "generalization_by_train_support.csv"
)


support_df.to_csv(
    support_file,
    index=False
)


# ==========================================================================================
# 22. TRAIN SUPPORT / ACCURACY CORRELATION
# ==========================================================================================

section(
    "22. SUPPORT / ACCURACY CORRELATION"
)


if (
    per_class_df[
        "train_support"
    ].nunique()
    > 1
):

    support_top1_correlation = float(
        per_class_df[
            [
                "train_support",
                "top1_accuracy"
            ]
        ]
        .corr()
        .iloc[
            0,
            1
        ]
    )


    support_top5_correlation = float(
        per_class_df[
            [
                "train_support",
                "top5_accuracy"
            ]
        ]
        .corr()
        .iloc[
            0,
            1
        ]
    )

else:

    support_top1_correlation = np.nan
    support_top5_correlation = np.nan


print(
    "Correlation("
    "train support, Top-1):",
    support_top1_correlation
)


print(
    "Correlation("
    "train support, Top-5):",
    support_top5_correlation
)


# ==========================================================================================
# 23. GENERALIZATION SUMMARY
# ==========================================================================================

section(
    "23. GENERALIZATION SUMMARY"
)


print(
    "Validation samples :",
    len(
        labels
    )
)

print(
    "Top-1 correct      :",
    int(
        fusion_correct.sum()
    )
)

print(
    "Top-1 incorrect    :",
    int(
        (
            ~fusion_correct
        ).sum()
    )
)

print(
    "Top-5 correct      :",
    int(
        fusion_top5_correct.sum()
    )
)

print()


print(
    "Micro Top-1        :",
    f"{fusion_top1:.3f}%"
)

print(
    "Macro Top-1        :",
    f"{macro_top1:.3f}%"
)

print(
    "Micro Top-5        :",
    f"{fusion_top5:.3f}%"
)

print(
    "Macro Top-5        :",
    f"{macro_top5:.3f}%"
)


# ==========================================================================================
# 24. RGB / LANDMARK COMPLEMENTARITY
# ==========================================================================================

section(
    "24. RGB / LANDMARK COMPLEMENTARITY"
)


rgb_pred = rgb_logits.argmax(
    axis=1
)


landmark_pred = landmark_logits.argmax(
    axis=1
)


rgb_correct = (
    rgb_pred ==
    labels
)


landmark_correct = (
    landmark_pred ==
    labels
)


both_correct = (
    rgb_correct &
    landmark_correct
)


rgb_only_correct = (
    rgb_correct &
    ~landmark_correct
)


landmark_only_correct = (
    ~rgb_correct &
    landmark_correct
)


neither_branch_top1_correct = (
    ~rgb_correct &
    ~landmark_correct
)


print(
    "Both branches correct :",
    int(
        both_correct.sum()
    )
)


print(
    "RGB-only correct      :",
    int(
        rgb_only_correct.sum()
    )
)


print(
    "Landmark-only correct :",
    int(
        landmark_only_correct.sum()
    )
)


print(
    "Neither branch Top-1  :",
    int(
        neither_branch_top1_correct.sum()
    )
)


# ------------------------------------------------------------------------------------------
# IMPORTANT:
#
# This is NOT an oracle ceiling.
# It only measures whether either individual model's Top-1 was correct.
# ------------------------------------------------------------------------------------------

individual_top1_union = (
    rgb_correct |
    landmark_correct
)


individual_top1_union_accuracy = (
    100.0 *
    individual_top1_union.mean()
)


print()
print(
    "Individual Top-1 union:",
    f"{individual_top1_union_accuracy:.3f}%"
)

print(
    "[NOTE] This is NOT an oracle ceiling."
)


# ==========================================================================================
# 25. FUSION RESCUE ANALYSIS
# ==========================================================================================

section(
    "25. FUSION RESCUE ANALYSIS"
)


gained_vs_rgb = (
    ~rgb_correct &
    fusion_correct
)


lost_vs_rgb = (
    rgb_correct &
    ~fusion_correct
)


gained_vs_landmark = (
    ~landmark_correct &
    fusion_correct
)


lost_vs_landmark = (
    landmark_correct &
    ~fusion_correct
)


# Cases where:
#   RGB Top1 wrong
#   Landmark Top1 wrong
#   Fusion Top1 correct
#
# This explains how fusion may exceed the individual Top1 union.

rescued_from_neither = (
    fusion_correct &
    neither_branch_top1_correct
)


print(
    "Fusion correct total        :",
    int(
        fusion_correct.sum()
    )
)


print(
    "Fusion gains vs RGB         :",
    int(
        gained_vs_rgb.sum()
    )
)


print(
    "Fusion losses vs RGB        :",
    int(
        lost_vs_rgb.sum()
    )
)


print(
    "Fusion gains vs Landmark    :",
    int(
        gained_vs_landmark.sum()
    )
)


print(
    "Fusion losses vs Landmark   :",
    int(
        lost_vs_landmark.sum()
    )
)


print(
    "Rescued when BOTH individual "
    "Top-1 predictions were wrong:",
    int(
        rescued_from_neither.sum()
    )
)


# ==========================================================================================
# 26. MOST COMMON CONFUSIONS
# ==========================================================================================

section(
    "26. MOST COMMON CONFUSIONS"
)


confusion_rows = []


for true_class in range(
    NUM_CLASSES
):

    for predicted_class in range(
        NUM_CLASSES
    ):

        if true_class == predicted_class:

            continue


        count = int(
            cm[
                true_class,
                predicted_class
            ]
        )


        if count > 0:

            confusion_rows.append(
                {
                    "true_class":
                        true_class,

                    "true_gloss":
                        class_to_gloss.get(
                            true_class,
                            str(
                                true_class
                            )
                        ),

                    "predicted_class":
                        predicted_class,

                    "predicted_gloss":
                        class_to_gloss.get(
                            predicted_class,
                            str(
                                predicted_class
                            )
                        ),

                    "count":
                        count
                }
            )


confusion_df = pd.DataFrame(
    confusion_rows
)


if len(
    confusion_df
) > 0:

    confusion_df = (
        confusion_df
        .sort_values(
            [
                "count",
                "true_class"
            ],
            ascending=[
                False,
                True
            ]
        )
        .reset_index(
            drop=True
        )
    )


common_confusion_file = (
    REPORTS /
    "most_common_confusions.csv"
)


confusion_df.to_csv(
    common_confusion_file,
    index=False
)


print(
    confusion_df.head(
        30
    ).to_string(
        index=False
    )
)


# ==========================================================================================
# 27. PER-CLASS ACCURACY FIGURE
# ==========================================================================================

section(
    "27. GENERATING PER-CLASS FIGURE"
)


sorted_per_class = (
    per_class_df
    .sort_values(
        "top1_accuracy",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


fig = plt.figure(
    figsize=(
        16,
        6
    )
)


plt.bar(
    np.arange(
        len(
            sorted_per_class
        )
    ),
    sorted_per_class[
        "top1_accuracy"
    ]
)


plt.axhline(
    macro_top1,
    linestyle="--",
    label=(
        f"Macro Top-1 = "
        f"{macro_top1:.2f}%"
    )
)


plt.xlabel(
    "Classes sorted by validation Top-1 accuracy"
)


plt.ylabel(
    "Validation Top-1 Accuracy (%)"
)


plt.title(
    "L20D-B2C — Per-Class Generalization"
)


plt.legend()


plt.tight_layout()


per_class_figure = (
    FIGURES /
    "b2c_per_class_accuracy.png"
)


plt.savefig(
    per_class_figure,
    dpi=200
)


plt.close(
    fig
)


print(
    "[SAVED]",
    per_class_figure
)


# ==========================================================================================
# 28. SUPPORT VS GENERALIZATION FIGURE
# ==========================================================================================

section(
    "28. GENERATING SUPPORT / GENERALIZATION FIGURE"
)


fig = plt.figure(
    figsize=(
        9,
        6
    )
)


plt.plot(
    support_df[
        "train_examples_per_class"
    ],
    support_df[
        "sample_weighted_top1"
    ],
    marker="o",
    label="Top-1"
)


plt.plot(
    support_df[
        "train_examples_per_class"
    ],
    support_df[
        "sample_weighted_top5"
    ],
    marker="o",
    label="Top-5"
)


plt.xlabel(
    "Independent Training Examples per Class"
)


plt.ylabel(
    "Validation Accuracy (%)"
)


plt.title(
    "L20D-B2C — Generalization vs Training Support"
)


plt.legend()


plt.tight_layout()


support_figure = (
    FIGURES /
    "generalization_vs_support.png"
)


plt.savefig(
    support_figure,
    dpi=200
)


plt.close(
    fig
)


print(
    "[SAVED]",
    support_figure
)


# ==========================================================================================
# 29. SAMPLE-LEVEL VALIDATION PREDICTIONS
# ==========================================================================================

section(
    "29. SAVE SAMPLE-LEVEL PREDICTIONS"
)


prediction_df = pd.DataFrame(
    {
        "uid":
            uids,

        "true_class":
            labels,

        "true_gloss":
            [
                class_to_gloss.get(
                    int(x),
                    str(x)
                )
                for x in labels
            ],

        "rgb_prediction":
            rgb_pred,

        "rgb_prediction_gloss":
            [
                class_to_gloss.get(
                    int(x),
                    str(x)
                )
                for x in rgb_pred
            ],

        "landmark_prediction":
            landmark_pred,

        "landmark_prediction_gloss":
            [
                class_to_gloss.get(
                    int(x),
                    str(x)
                )
                for x in landmark_pred
            ],

        "fusion_prediction":
            fusion_pred,

        "fusion_prediction_gloss":
            [
                class_to_gloss.get(
                    int(x),
                    str(x)
                )
                for x in fusion_pred
            ],

        "rgb_correct":
            rgb_correct.astype(
                int
            ),

        "landmark_correct":
            landmark_correct.astype(
                int
            ),

        "fusion_correct":
            fusion_correct.astype(
                int
            ),

        "fusion_top5_correct":
            fusion_top5_correct.astype(
                int
            ),

        "fusion_rescued_from_neither":
            rescued_from_neither.astype(
                int
            )
    }
)


prediction_file = (
    REPORTS /
    "validation_predictions.csv"
)


prediction_df.to_csv(
    prediction_file,
    index=False
)


print(
    "[SAVED]",
    prediction_file
)


# ==========================================================================================
# 30. TRAINING / VALIDATION GENERALIZATION CONTEXT
#
# IMPORTANT:
# These training values come from the frozen training histories.
#
# B1:
#   final sampled train Top1 = 88.592%
#
# B2B:
#   reached approximately 100% sampled training accuracy.
#
# We do NOT fabricate B2C training accuracy because B2C was selected/evaluated
# using frozen validation logits and we have not performed an independent
# train-set B2C inference pass.
# ==========================================================================================

section(
    "30. TRAINING / VALIDATION GENERALIZATION CONTEXT"
)


B1_SAMPLED_TRAIN_TOP1 = 88.592

B2B_SAMPLED_TRAIN_TOP1_APPROX = 100.0


b1_validation_top1 = topk_accuracy(
    rgb_logits,
    labels,
    1
)


b2b_validation_top1 = topk_accuracy(
    landmark_logits,
    labels,
    1
)


b1_generalization_gap = (
    B1_SAMPLED_TRAIN_TOP1 -
    b1_validation_top1
)


b2b_generalization_gap_approx = (
    B2B_SAMPLED_TRAIN_TOP1_APPROX -
    b2b_validation_top1
)


print(
    "B1 sampled training Top-1 :",
    f"{B1_SAMPLED_TRAIN_TOP1:.3f}%"
)


print(
    "B1 validation Top-1       :",
    f"{b1_validation_top1:.3f}%"
)


print(
    "B1 generalization gap     :",
    f"{b1_generalization_gap:.3f} pp"
)


print()


print(
    "B2B sampled training Top-1:",
    f"~{B2B_SAMPLED_TRAIN_TOP1_APPROX:.1f}%"
)


print(
    "B2B validation Top-1      :",
    f"{b2b_validation_top1:.3f}%"
)


print(
    "B2B approximate gap       :",
    f"{b2b_generalization_gap_approx:.3f} pp"
)


print()
print(
    "B2C train Top-1           : "
    "NOT CLAIMED"
)


print(
    "Reason                    : "
    "No frozen B2C train-set inference was performed."
)


print()
print(
    "[INTERPRETATION]"
)


print(
    "The large train/validation gap in the component models "
    "is evidence of strong overfitting in this low-shot "
    "215-class setting."
)


# ==========================================================================================
# 31. REQUESTED TARGET STATUS
#
# Targets are checked, NOT forced.
# ==========================================================================================

section(
    "31. REQUESTED TARGET STATUS"
)


print(
    "Requested training target   :",
    f">= {REQUESTED_TRAIN_TARGET:.1f}%"
)


print(
    "Requested validation target :",
    f">= {REQUESTED_VALIDATION_TARGET:.1f}%"
)


print()


print(
    "Measured B2C validation Top1:",
    f"{fusion_top1:.3f}%"
)


print(
    "Measured B2C validation Top5:",
    f"{fusion_top5:.3f}%"
)


print()


if fusion_top1 >= REQUESTED_VALIDATION_TARGET:

    validation_target_status = True

    print(
        "[TARGET MET] "
        "B2C validation Top-1 >= 80%."
    )

else:

    validation_target_status = False

    print(
        "[TARGET NOT MET] "
        "B2C validation Top-1 < 80%."
    )


print()


if B1_SAMPLED_TRAIN_TOP1 >= REQUESTED_TRAIN_TARGET:

    print(
        "[B1 TRAIN TARGET MET]"
    )

else:

    print(
        "[B1 TRAIN TARGET NOT MET] "
        f"{B1_SAMPLED_TRAIN_TOP1:.3f}% < 90%."
    )


if (
    B2B_SAMPLED_TRAIN_TOP1_APPROX
    >=
    REQUESTED_TRAIN_TARGET
):

    print(
        "[B2B TRAIN TARGET MET]"
    )


print()
print(
    "[IMPORTANT] Accuracy values are measured results."
)

print(
    "No predictions, labels, samples, thresholds, or metrics "
    "were altered to satisfy the requested targets."
)


# ==========================================================================================
# 32. GENERALIZATION INTERPRETATION
# ==========================================================================================

section(
    "32. GENERALIZATION ANALYSIS"
)


print(
    "Independent training videos :",
    len(
        train_df
    )
)


print(
    "Number of classes           :",
    NUM_CLASSES
)


print(
    "Mean train videos / class   :",
    f"{len(train_df)/NUM_CLASSES:.3f}"
)


print(
    "Minimum train support       :",
    min(
        train_counts.values()
    )
)


print(
    "Maximum train support       :",
    max(
        train_counts.values()
    )
)


print()


print(
    "Micro validation Top-1      :",
    f"{fusion_top1:.3f}%"
)


print(
    "Macro validation Top-1      :",
    f"{macro_top1:.3f}%"
)


print(
    "Micro validation Top-5      :",
    f"{fusion_top5:.3f}%"
)


print(
    "Macro validation Top-5      :",
    f"{macro_top5:.3f}%"
)


print()


print(
    "RGB-only correct examples   :",
    int(
        rgb_only_correct.sum()
    )
)


print(
    "Landmark-only examples      :",
    int(
        landmark_only_correct.sum()
    )
)


print(
    "Fusion rescued from neither :",
    int(
        rescued_from_neither.sum()
    )
)


print()


print(
    "Interpretation:"
)


print(
    "1. RGB and landmark modalities contain complementary information."
)


print(
    "2. Landmark-only correct samples demonstrate geometry-specific information "
    "not captured by the RGB branch."
)


print(
    "3. Fusion can recover the true class even when neither branch has the "
    "correct individual Top-1 prediction."
)


print(
    "4. The large training/validation gap indicates overfitting caused by "
    "the extremely low number of independent training examples per class."
)


print(
    "5. B2C validation metrics are model-selection/development results because "
    "fusion parameters were selected using this validation set."
)


print(
    "6. The locked 10-video test remains untouched and is not part of L20D."
)


# ==========================================================================================
# 33. SUMMARY JSON
# ==========================================================================================

section(
    "33. CREATE FINAL L20D SUMMARY"
)


summary = {

    "experiment":
        "L20D_FULL_CISLR_EVALUATION",

    "status":
        "COMPLETE",

    "protocol": {

        "train_samples":
            710,

        "validation_samples":
            304,

        "locked_test_samples":
            10,

        "classes":
            NUM_CLASSES,

        "train_validation_overlap":
            0,

        "train_locked_overlap":
            0,

        "validation_locked_overlap":
            0,

        "locked_test_loaded_for_inference":
            False,

        "locked_test_evaluated":
            False
    },

    "validation_metrics": {

        "B1_RGB": {

            "top1":
                float(
                    b1_validation_top1
                ),

            "top5":
                float(
                    topk_accuracy(
                        rgb_logits,
                        labels,
                        5
                    )
                )
        },

        "B2B_LANDMARK": {

            "top1":
                float(
                    b2b_validation_top1
                ),

            "top5":
                float(
                    topk_accuracy(
                        landmark_logits,
                        labels,
                        5
                    )
                )
        },

        "B2C_FUSION": {

            "top1":
                float(
                    fusion_top1
                ),

            "top5":
                float(
                    fusion_top5
                ),

            "macro_top1":
                float(
                    macro_top1
                ),

            "macro_top5":
                float(
                    macro_top5
                )
        }
    },

    "fusion_configuration": {

        "method":
            fusion_method,

        "rgb_temperature":
            rgb_temperature,

        "landmark_temperature":
            landmark_temperature,

        "rgb_weight":
            rgb_weight,

        "landmark_weight":
            landmark_weight,

        "selected_on_validation":
            True
    },

    "multimodal_analysis": {

        "both_correct":
            int(
                both_correct.sum()
            ),

        "rgb_only_correct":
            int(
                rgb_only_correct.sum()
            ),

        "landmark_only_correct":
            int(
                landmark_only_correct.sum()
            ),

        "neither_individual_top1_correct":
            int(
                neither_branch_top1_correct.sum()
            ),

        "individual_top1_union_accuracy":
            float(
                individual_top1_union_accuracy
            ),

        "individual_top1_union_is_oracle":
            False,

        "fusion_correct":
            int(
                fusion_correct.sum()
            ),

        "fusion_gains_vs_rgb":
            int(
                gained_vs_rgb.sum()
            ),

        "fusion_losses_vs_rgb":
            int(
                lost_vs_rgb.sum()
            ),

        "fusion_rescued_from_neither":
            int(
                rescued_from_neither.sum()
            )
    },

    "generalization": {

        "mean_train_examples_per_class":
            float(
                len(train_df) /
                NUM_CLASSES
            ),

        "minimum_train_support":
            int(
                min(
                    train_counts.values()
                )
            ),

        "maximum_train_support":
            int(
                max(
                    train_counts.values()
                )
            ),

        "B1_sampled_train_top1":
            B1_SAMPLED_TRAIN_TOP1,

        "B1_validation_top1":
            float(
                b1_validation_top1
            ),

        "B1_generalization_gap":
            float(
                b1_generalization_gap
            ),

        "B2B_sampled_train_top1_approx":
            B2B_SAMPLED_TRAIN_TOP1_APPROX,

        "B2B_validation_top1":
            float(
                b2b_validation_top1
            ),

        "B2B_generalization_gap_approx":
            float(
                b2b_generalization_gap_approx
            ),

        "B2C_train_accuracy_claimed":
            False,

        "support_top1_correlation":
            (
                None
                if np.isnan(
                    support_top1_correlation
                )
                else float(
                    support_top1_correlation
                )
            ),

        "support_top5_correlation":
            (
                None
                if np.isnan(
                    support_top5_correlation
                )
                else float(
                    support_top5_correlation
                )
            )
    },

    "requested_targets": {

        "training_top1_target":
            REQUESTED_TRAIN_TARGET,

        "validation_top1_target":
            REQUESTED_VALIDATION_TARGET,

        "validation_target_met":
            bool(
                validation_target_status
            ),

        "targets_used_to_modify_evaluation":
            False
    },

    "scientific_note":
        (
            "B2C fusion parameters were selected on the same 304-video "
            "validation set. Therefore B2C validation accuracy is a "
            "development/model-selection result rather than an unbiased "
            "final-test estimate. The locked 10-video test remains untouched."
        )
}


SUMMARY_FILE = (
    REPORTS /
    "L20D_FULL_CISLR_EVALUATION_SUMMARY.json"
)


with open(
    SUMMARY_FILE,
    "w"
) as f:

    json.dump(
        summary,
        f,
        indent=2
    )


print(
    "[SAVED]",
    SUMMARY_FILE
)


# ==========================================================================================
# 34. MASTER FREEZE
# ==========================================================================================

section(
    "34. CREATE MASTER FREEZE"
)


master_freeze = {

    "experiment":
        "L20D_FULL_CISLR_EVALUATION",

    "status":
        "FROZEN",

    "inputs": {

        "train_manifest":
            str(
                TRAIN_MANIFEST
            ),

        "validation_manifest":
            str(
                VAL_MANIFEST
            ),

        "class_mapping":
            str(
                CLASS_MAPPING
            ),

        "b2c_aligned_logits":
            str(
                ALIGNED_LOGITS
            ),

        "b2c_summary":
            str(
                B2C_SUMMARY
            )
    },

    "outputs": {

        "summary":
            str(
                SUMMARY_FILE
            ),

        "overall_metrics":
            str(
                overall_metrics_file
            ),

        "confusion_matrix":
            str(
                confusion_csv_file
            ),

        "normalized_confusion_matrix":
            str(
                normalized_cm_file
            ),

        "per_class_accuracy":
            str(
                per_class_file
            ),

        "support_generalization":
            str(
                support_file
            ),

        "common_confusions":
            str(
                common_confusion_file
            ),

        "predictions":
            str(
                prediction_file
            ),

        "confusion_matrix_figure":
            str(
                raw_cm_figure
            ),

        "normalized_confusion_matrix_figure":
            str(
                normalized_cm_figure
            ),

        "per_class_figure":
            str(
                per_class_figure
            ),

        "support_figure":
            str(
                support_figure
            )
    },

    "integrity": {

        "validation_uid_verified":
            True,

        "validation_labels_verified":
            True,

        "all_logits_finite":
            True,

        "locked_test_loaded":
            False,

        "locked_test_evaluated":
            False,

        "metrics_modified_to_meet_target":
            False
    }
}


MASTER_FILE = (
    OUTPUT /
    "L20D_FULL_CISLR_MASTER_FREEZE.json"
)


with open(
    MASTER_FILE,
    "w"
) as f:

    json.dump(
        master_freeze,
        f,
        indent=2
    )


print(
    "[SAVED]",
    MASTER_FILE
)


# ==========================================================================================
# 35. HASHES
# ==========================================================================================

section(
    "35. FINAL ARTIFACT HASHES"
)


hash_files = {

    "summary":
        SUMMARY_FILE,

    "overall_metrics":
        overall_metrics_file,

    "confusion_matrix":
        confusion_csv_file,

    "normalized_cm":
        normalized_cm_file,

    "per_class_accuracy":
        per_class_file,

    "support_analysis":
        support_file,

    "common_confusions":
        common_confusion_file,

    "predictions":
        prediction_file,

    "master_freeze":
        MASTER_FILE
}


hashes = {}


for name, path in hash_files.items():

    digest = sha256_file(
        path
    )

    hashes[
        name
    ] = digest

    print(
        f"{name:<24}: "
        f"{digest}"
    )


HASH_FILE = (
    REPORTS /
    "artifact_hashes.json"
)


with open(
    HASH_FILE,
    "w"
) as f:

    json.dump(
        hashes,
        f,
        indent=2
    )


# ==========================================================================================
# 36. FINAL REPORT
# ==========================================================================================

section(
    "L20D FULL-CISLR EVALUATION COMPLETE"
)


print(
    "DATASET"
)

print(
    "Train      : 710"
)

print(
    "Validation : 304"
)

print(
    "Classes    : 215"
)

print(
    "Locked test: 10 — NOT EVALUATED"
)


print()
print(
    "VALIDATION ACCURACY"
)


print(
    f"B1 RGB       "
    f"| Top-1 {b1_validation_top1:.3f}% "
    f"| Top-5 "
    f"{topk_accuracy(rgb_logits, labels, 5):.3f}%"
)


print(
    f"B2B Landmark "
    f"| Top-1 {b2b_validation_top1:.3f}% "
    f"| Top-5 "
    f"{topk_accuracy(landmark_logits, labels, 5):.3f}%"
)


print(
    f"B2C Fusion   "
    f"| Top-1 {fusion_top1:.3f}% "
    f"| Top-5 {fusion_top5:.3f}%"
)


print()
print(
    "PER-CLASS GENERALIZATION"
)


print(
    f"Macro Top-1 : "
    f"{macro_top1:.3f}%"
)


print(
    f"Macro Top-5 : "
    f"{macro_top5:.3f}%"
)


print()
print(
    "MULTIMODAL COMPLEMENTARITY"
)


print(
    "Both correct       :",
    int(
        both_correct.sum()
    )
)


print(
    "RGB-only correct   :",
    int(
        rgb_only_correct.sum()
    )
)


print(
    "Landmark-only      :",
    int(
        landmark_only_correct.sum()
    )
)


print(
    "Neither Top-1      :",
    int(
        neither_branch_top1_correct.sum()
    )
)


print(
    "Individual union   :",
    f"{individual_top1_union_accuracy:.3f}%"
)


print(
    "Fusion rescue      :",
    int(
        rescued_from_neither.sum()
    )
)


print()
print(
    "GENERALIZATION"
)


print(
    "Mean train/class   :",
    f"{len(train_df)/NUM_CLASSES:.3f}"
)


print(
    "B1 train Top-1     :",
    f"{B1_SAMPLED_TRAIN_TOP1:.3f}% "
    "(sampled training history)"
)


print(
    "B1 validation      :",
    f"{b1_validation_top1:.3f}%"
)


print(
    "B1 gap             :",
    f"{b1_generalization_gap:.3f} pp"
)


print(
    "B2B train Top-1    :",
    "~100% "
    "(sampled training history)"
)


print(
    "B2B validation     :",
    f"{b2b_validation_top1:.3f}%"
)


print()
print(
    "REQUESTED TARGETS"
)


print(
    "Training target    : >=90%"
)


print(
    "Validation target  : >=80%"
)


print(
    "B2C validation     :",
    f"{fusion_top1:.3f}%"
)


print(
    "Validation target  :",
    (
        "MET"
        if validation_target_status
        else "NOT MET"
    )
)


print()
print(
    "SCIENTIFIC STATUS"
)


print(
    "Locked test loaded    : NO"
)


print(
    "Locked test evaluated : NO"
)


print(
    "Metrics manipulated   : NO"
)


print(
    "B2C validation result : "
    "DEVELOPMENT / MODEL-SELECTION RESULT"
)


print()
print(
    "Reports:"
)

print(
    REPORTS
)


print()
print(
    "Figures:"
)

print(
    FIGURES
)


print()
print(
    "Master freeze:"
)

print(
    MASTER_FILE
)


print()
print(
    "=" * 115
)


L20D — FULL CISLR 215-CLASS EVALUATION
Evaluation population : 215 classes
Expected train        : 710
Expected validation   : 304
Locked test           : 10
Locked test inference : DISABLED

5. VERIFY REQUIRED ARTIFACTS
[FOUND] Train manifest           : /home/dipshikha/Music/cao/CISLR_RESEARCH/CISLR_LANDMARK_MODEL/audit/l20b_corrected_strict_protocol_freeze/manifests/l20b_corrected_train_manifest.csv
[FOUND] Validation manifest      : /home/dipshikha/Music/cao/CISLR_RESEARCH/CISLR_LANDMARK_MODEL/audit/l20b_corrected_strict_protocol_freeze/manifests/l20b_corrected_validation_manifest.csv
[FOUND] Locked manifest          : /home/dipshikha/Music/cao/CISLR_RESEARCH/CISLR_LANDMARK_MODEL/audit/l20b_corrected_strict_protocol_freeze/manifests/l20b_permanent_l19_locked_test_manifest.csv
[FOUND] Class mapping            : /home/dipshikha/Music/cao/CISLR_RESEARCH/CISLR_LANDMARK_MODEL/audit/l20b_corrected_strict_protocol_freeze/manifests/l20b_corrected_class_mapping.csv
[FOUND] B2C aligned logi

In [10]:
# ==========================================================================================
# PARASURG / CISLR
# L21 — FREEZE DEPLOYMENT CLASSIFIER
# REAL-TIME INDIAN SIGN LANGUAGE INTERPRETER
#
# SELECTED DEPLOYMENT MODEL
# -------------------------
# L19 Restricted-Vocabulary VideoMAE
#
# L21 PURPOSE
# -----------
# 1. Verify L18B 70 / 10 / 10 protocol
# 2. Verify exact L17 10-sign vocabulary
# 3. Verify L19 checkpoint SHA256
# 4. Inspect exact checkpoint structure
# 5. Discover exact state_dict and classifier head
# 6. Verify model parameters
# 7. Freeze checkpoint byte-for-byte
# 8. Save deployment configuration
# 9. Save class mapping
# 10. Hash all artifacts
# 11. Create L21 master freeze
#
# IMPORTANT
# ---------
# NO TRAINING
# NO RETRAINING
# NO HYPERPARAMETER TUNING
# NO LOCKED TEST INFERENCE
# NO LOCKED TEST EVALUATION
# NO MODEL WEIGHT MODIFICATION
# NO OPENVINO EXPORT YET
#
# NEXT:
# L22 — Export selected deployment model to OpenVINO
# ==========================================================================================


# ==========================================================================================
# 1. IMPORTS
# ==========================================================================================

import sys
import json
import hashlib
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import torch


# ==========================================================================================
# 2. HELPERS
# ==========================================================================================

def section(title):

    print()
    print("=" * 120)
    print(title)
    print("=" * 120)


def sha256_file(path):

    path = Path(path)

    h = hashlib.sha256()

    with open(path, "rb") as f:

        while True:

            block = f.read(
                1024 * 1024
            )

            if not block:
                break

            h.update(block)

    return h.hexdigest()


def find_column(
    df,
    candidates
):

    lower_map = {
        str(column).lower(): column
        for column in df.columns
    }

    for candidate in candidates:

        candidate_lower = (
            candidate.lower()
        )

        if candidate_lower in lower_map:

            return lower_map[
                candidate_lower
            ]

    return None


def locate_manifest(
    root,
    exact_names,
    required_words
):

    root = Path(root)

    # ------------------------------------------------------------------
    # Exact filenames first
    # ------------------------------------------------------------------

    for name in exact_names:

        path = (
            root /
            name
        )

        if path.exists():

            return path

    # ------------------------------------------------------------------
    # Fallback search
    # ------------------------------------------------------------------

    candidates = []

    if root.exists():

        for path in root.rglob(
            "*.csv"
        ):

            lower_name = (
                path.name.lower()
            )

            if all(
                word.lower()
                in lower_name

                for word
                in required_words
            ):

                candidates.append(
                    path
                )

    if len(candidates) == 1:

        return candidates[0]

    if len(candidates) > 1:

        manifest_candidates = [

            path

            for path
            in candidates

            if "manifest"
            in path.name.lower()
        ]

        if len(
            manifest_candidates
        ) == 1:

            return (
                manifest_candidates[0]
            )

        print()
        print(
            "[ERROR] Multiple candidates:"
        )

        for path in candidates:

            print(
                "   ",
                path
            )

        raise RuntimeError(
            "Multiple manifest candidates found. "
            "Refusing to guess."
        )

    return None


# ==========================================================================================
# 3. HEADER
# ==========================================================================================

section(
    "CISLR — L21 FREEZE DEPLOYMENT CLASSIFIER"
)


print(
    "Python           :",
    sys.version.split()[0]
)

print(
    "PyTorch          :",
    torch.__version__
)

print(
    "CUDA available   :",
    torch.cuda.is_available()
)


if torch.cuda.is_available():

    print(
        "GPU              :",
        torch.cuda.get_device_name(
            0
        )
    )


print(
    "Training         : DISABLED"
)

print(
    "Locked test eval : DISABLED"
)

print(
    "Weight changes   : DISABLED"
)

print(
    "OpenVINO export  : NOT IN L21"
)


# ==========================================================================================
# 4. PATHS
# ==========================================================================================

section(
    "4. PROJECT PATHS"
)


PROJECT_ROOT = Path(
    "/home/dipshikha/Music/cao/CISLR_RESEARCH"
)


LANDMARK_ROOT = (
    PROJECT_ROOT /
    "CISLR_LANDMARK_MODEL"
)


AUDIT_ROOT = (
    LANDMARK_ROOT /
    "audit"
)


# ------------------------------------------------------------------------------------------
# L18B
# ------------------------------------------------------------------------------------------

L18B_ROOT = (
    AUDIT_ROOT /
    "l18b_10class_dataset"
)


L18B_MANIFEST_ROOT = (
    L18B_ROOT /
    "manifests"
)


# ------------------------------------------------------------------------------------------
# L19
# ------------------------------------------------------------------------------------------

L19_ROOT = (
    AUDIT_ROOT /
    "l19_10class_videomae"
)


L19_MODEL_ROOT = (
    L19_ROOT /
    "models"
)


L19_CHECKPOINT = (
    L19_MODEL_ROOT /
    "l19_best_videomae_10class.pt"
)


L19_MASTER_FREEZE = (
    L19_ROOT /
    "L19_MASTER_FREEZE.json"
)


# ------------------------------------------------------------------------------------------
# L21
# ------------------------------------------------------------------------------------------

L21_ROOT = (
    AUDIT_ROOT /
    "l21_deployment_classifier_freeze"
)


L21_MODEL_ROOT = (
    L21_ROOT /
    "model"
)


L21_CONFIG_ROOT = (
    L21_ROOT /
    "config"
)


L21_REPORT_ROOT = (
    L21_ROOT /
    "reports"
)


for directory in [

    L21_ROOT,
    L21_MODEL_ROOT,
    L21_CONFIG_ROOT,
    L21_REPORT_ROOT

]:

    directory.mkdir(
        parents=True,
        exist_ok=True
    )


print(
    "Project root :",
    PROJECT_ROOT
)

print(
    "L18B root    :",
    L18B_ROOT
)

print(
    "L19 root     :",
    L19_ROOT
)

print(
    "L21 root     :",
    L21_ROOT
)


# ==========================================================================================
# 5. AUTHORITATIVE L19 HASHES
# ==========================================================================================

section(
    "5. AUTHORITATIVE L19 ARTIFACT HASHES"
)


# ------------------------------------------------------------------------------------------
# IMPORTANT:
#
# These are TWO DIFFERENT artifacts.
#
# CHECKPOINT:
#   l19_best_videomae_10class.pt
#
# MASTER FREEZE:
#   L19_MASTER_FREEZE.json
# ------------------------------------------------------------------------------------------


EXPECTED_L19_CHECKPOINT_SHA = (
    "3a35abe76cdea634490473eaf83c6121e102658362cfbab19410a4949bb8c91c"
)


EXPECTED_L19_MASTER_FREEZE_SHA = (
    "0997873212a9b4495b965773645231f8a0206b12a5f98934bac04b2a071b9d39"
)


print(
    "Expected checkpoint SHA:"
)

print(
    EXPECTED_L19_CHECKPOINT_SHA
)


print()
print(
    "Expected master-freeze SHA:"
)

print(
    EXPECTED_L19_MASTER_FREEZE_SHA
)


# ==========================================================================================
# 6. LOCATE L18B MANIFESTS
# ==========================================================================================

section(
    "6. LOCATE L18B DATASET MANIFESTS"
)


TRAIN_MANIFEST = locate_manifest(

    L18B_MANIFEST_ROOT,

    exact_names=[
        "l18b_train_manifest.csv",
        "train_manifest.csv"
    ],

    required_words=[
        "train"
    ]
)


VAL_MANIFEST = locate_manifest(

    L18B_MANIFEST_ROOT,

    exact_names=[
        "l18b_validation_manifest.csv",
        "l18b_val_manifest.csv",
        "validation_manifest.csv"
    ],

    required_words=[
        "validation"
    ]
)


LOCKED_TEST_MANIFEST = locate_manifest(

    L18B_MANIFEST_ROOT,

    exact_names=[
        "l18b_LOCKED_test_manifest.csv",
        "l18b_locked_test_manifest.csv"
    ],

    required_words=[
        "test"
    ]
)


if TRAIN_MANIFEST is None:

    raise FileNotFoundError(
        "Could not locate L18B training manifest."
    )


if VAL_MANIFEST is None:

    raise FileNotFoundError(
        "Could not locate L18B validation manifest."
    )


if LOCKED_TEST_MANIFEST is None:

    raise FileNotFoundError(
        "Could not locate L18B locked-test manifest."
    )


print(
    "[FOUND] Train manifest :",
    TRAIN_MANIFEST
)

print(
    "[FOUND] Val manifest   :",
    VAL_MANIFEST
)

print(
    "[FOUND] Locked manifest:",
    LOCKED_TEST_MANIFEST
)


# ==========================================================================================
# 7. VERIFY L19 FILES
# ==========================================================================================

section(
    "7. VERIFY L19 ARTIFACT FILES"
)


if not L19_CHECKPOINT.exists():

    raise FileNotFoundError(
        "L19 checkpoint not found:\n"
        f"{L19_CHECKPOINT}"
    )


checkpoint_size_mb = (
    L19_CHECKPOINT.stat().st_size /
    (1024 ** 2)
)


print(
    "[FOUND] L19 checkpoint:"
)

print(
    L19_CHECKPOINT
)


print(
    "Checkpoint size:",
    f"{checkpoint_size_mb:.2f} MB"
)


if L19_MASTER_FREEZE.exists():

    print()
    print(
        "[FOUND] L19 master freeze:"
    )

    print(
        L19_MASTER_FREEZE
    )

else:

    print()
    print(
        "[WARNING] L19 master-freeze JSON "
        "not found at expected path."
    )


# ==========================================================================================
# 8. VERIFY CHECKPOINT SHA256
# ==========================================================================================

section(
    "8. VERIFY L19 CHECKPOINT SHA256"
)


source_checkpoint_sha = (
    sha256_file(
        L19_CHECKPOINT
    )
)


print(
    "Actual checkpoint SHA:"
)

print(
    source_checkpoint_sha
)


print()
print(
    "Expected checkpoint SHA:"
)

print(
    EXPECTED_L19_CHECKPOINT_SHA
)


if (
    source_checkpoint_sha
    !=
    EXPECTED_L19_CHECKPOINT_SHA
):

    raise RuntimeError(
        "\n"
        "L19 CHECKPOINT SHA256 MISMATCH\n"
        "\n"
        f"Expected:\n"
        f"{EXPECTED_L19_CHECKPOINT_SHA}\n"
        "\n"
        f"Actual:\n"
        f"{source_checkpoint_sha}\n"
        "\n"
        "Do NOT continue to L22 until investigated."
    )


print()
print(
    "[PASS] L19 checkpoint SHA256 verified."
)


# ==========================================================================================
# 9. VERIFY MASTER-FREEZE SHA IF FILE EXISTS
# ==========================================================================================

section(
    "9. VERIFY L19 MASTER-FREEZE SHA256"
)


l19_master_freeze_verified = False
actual_l19_master_freeze_sha = None


if L19_MASTER_FREEZE.exists():

    actual_l19_master_freeze_sha = (
        sha256_file(
            L19_MASTER_FREEZE
        )
    )


    print(
        "Actual master-freeze SHA:"
    )

    print(
        actual_l19_master_freeze_sha
    )


    print()
    print(
        "Expected master-freeze SHA:"
    )

    print(
        EXPECTED_L19_MASTER_FREEZE_SHA
    )


    if (
        actual_l19_master_freeze_sha
        !=
        EXPECTED_L19_MASTER_FREEZE_SHA
    ):

        raise RuntimeError(
            "L19 master-freeze SHA mismatch."
        )


    l19_master_freeze_verified = True


    print()
    print(
        "[PASS] L19 master-freeze SHA256 verified."
    )


else:

    print(
        "[INFO] Master-freeze file unavailable "
        "at expected path."
    )

    print(
        "[INFO] Checkpoint itself has already "
        "been independently verified."
    )


# ==========================================================================================
# 10. LOAD L18B MANIFESTS
# ==========================================================================================

section(
    "10. VERIFY L18B DEPLOYMENT DATASET"
)


train_df = pd.read_csv(
    TRAIN_MANIFEST
)


val_df = pd.read_csv(
    VAL_MANIFEST
)


locked_df = pd.read_csv(
    LOCKED_TEST_MANIFEST
)


print(
    "Train rows     :",
    len(train_df)
)

print(
    "Validation rows:",
    len(val_df)
)

print(
    "Locked rows    :",
    len(locked_df)
)


if len(train_df) != 70:

    raise RuntimeError(
        f"Expected 70 train rows, got {len(train_df)}."
    )


if len(val_df) != 10:

    raise RuntimeError(
        f"Expected 10 validation rows, got {len(val_df)}."
    )


if len(locked_df) != 10:

    raise RuntimeError(
        f"Expected 10 locked-test rows, got {len(locked_df)}."
    )


print()
print(
    "[PASS] Expected 70 / 10 / 10 dataset sizes."
)


# ==========================================================================================
# 11. VERIFY ACTUAL MANIFEST SCHEMA
# ==========================================================================================

section(
    "11. VERIFY L18B MANIFEST SCHEMA"
)


print(
    "Train columns:"
)


for column in train_df.columns:

    print(
        "  ",
        column
    )


UID_COL = find_column(

    train_df,

    [
        "uid",
        "video_uid",
        "video_id",
        "id"
    ]
)


CLASS_COL = find_column(

    train_df,

    [
        "deployment_class_id",
        "class_id",
        "label",
        "target",
        "class"
    ]
)


GLOSS_COL = find_column(

    train_df,

    [
        "gloss",
        "word",
        "sign",
        "class_name"
    ]
)


if UID_COL is None:

    raise RuntimeError(
        "Could not identify UID column."
    )


if CLASS_COL is None:

    raise RuntimeError(
        "Could not identify class-ID column."
    )


if GLOSS_COL is None:

    raise RuntimeError(
        "Could not identify gloss column."
    )


print()
print(
    "UID column   :",
    UID_COL
)

print(
    "Class column :",
    CLASS_COL
)

print(
    "Gloss column :",
    GLOSS_COL
)


# ------------------------------------------------------------------------------------------
# Exact schema now known from L18B
# ------------------------------------------------------------------------------------------

if UID_COL != "uid":

    raise RuntimeError(
        f"Expected uid, found {UID_COL}."
    )


if CLASS_COL != "deployment_class_id":

    raise RuntimeError(
        "Expected deployment_class_id, "
        f"found {CLASS_COL}."
    )


if GLOSS_COL != "gloss":

    raise RuntimeError(
        f"Expected gloss, found {GLOSS_COL}."
    )


print()
print(
    "[PASS] Exact L18B schema verified."
)


# ==========================================================================================
# 12. VERIFY SPLIT SCHEMAS
# ==========================================================================================

section(
    "12. VERIFY ALL SPLIT SCHEMAS"
)


required_columns = {

    UID_COL,
    CLASS_COL,
    GLOSS_COL
}


for split_name, dataframe in [

    ("TRAIN", train_df),
    ("VALIDATION", val_df),
    ("LOCKED_TEST", locked_df)

]:

    missing = (
        required_columns -
        set(
            dataframe.columns
        )
    )


    if missing:

        raise RuntimeError(
            f"{split_name} missing columns: {missing}"
        )


    print(
        f"[PASS] {split_name:<12} schema verified."
    )


# ==========================================================================================
# 13. VERIFY SPLIT INTEGRITY
# ==========================================================================================

section(
    "13. SPLIT INTEGRITY"
)


train_uids = set(

    train_df[
        UID_COL
    ].astype(str)
)


val_uids = set(

    val_df[
        UID_COL
    ].astype(str)
)


locked_uids = set(

    locked_df[
        UID_COL
    ].astype(str)
)


print(
    "Unique train UIDs :",
    len(train_uids)
)

print(
    "Unique val UIDs   :",
    len(val_uids)
)

print(
    "Unique locked UIDs:",
    len(locked_uids)
)


if len(train_uids) != 70:

    raise RuntimeError(
        "Expected 70 unique training UIDs."
    )


if len(val_uids) != 10:

    raise RuntimeError(
        "Expected 10 unique validation UIDs."
    )


if len(locked_uids) != 10:

    raise RuntimeError(
        "Expected 10 unique locked-test UIDs."
    )


train_val_overlap = (
    train_uids &
    val_uids
)


train_locked_overlap = (
    train_uids &
    locked_uids
)


val_locked_overlap = (
    val_uids &
    locked_uids
)


print()
print(
    "Train / Val overlap   :",
    len(train_val_overlap)
)

print(
    "Train / Locked overlap:",
    len(train_locked_overlap)
)

print(
    "Val / Locked overlap  :",
    len(val_locked_overlap)
)


if train_val_overlap:

    raise RuntimeError(
        "Train/validation overlap detected."
    )


if train_locked_overlap:

    raise RuntimeError(
        "Train/locked-test overlap detected."
    )


if val_locked_overlap:

    raise RuntimeError(
        "Validation/locked-test overlap detected."
    )


print()
print(
    "[PASS] 70 / 10 / 10 splits are disjoint."
)


# ==========================================================================================
# 14. VERIFY FROZEN 10-SIGN VOCABULARY
# ==========================================================================================

section(
    "14. VERIFY L17 10-SIGN VOCABULARY"
)


EXPECTED_VOCAB = {

    0: "monday",
    1: "interest",
    2: "thursday",
    3: "christmas",
    4: "pencil",
    5: "mother",
    6: "tuesday",
    7: "friday",
    8: "rubber",
    9: "wash"
}


combined_df = pd.concat(

    [
        train_df,
        val_df,
        locked_df
    ],

    ignore_index=True
)


dataset_mapping_df = (

    combined_df[
        [
            CLASS_COL,
            GLOSS_COL
        ]
    ]

    .drop_duplicates()

    .sort_values(
        CLASS_COL
    )

    .reset_index(
        drop=True
    )
)


print(
    dataset_mapping_df.to_string(
        index=False
    )
)


dataset_vocab = {}


for _, row in dataset_mapping_df.iterrows():

    class_id = int(
        row[
            CLASS_COL
        ]
    )


    gloss = (

        str(
            row[
                GLOSS_COL
            ]
        )

        .strip()

        .lower()
    )


    if class_id in dataset_vocab:

        if (
            dataset_vocab[
                class_id
            ]
            !=
            gloss
        ):

            raise RuntimeError(
                f"Class {class_id} has multiple glosses."
            )


    dataset_vocab[
        class_id
    ] = gloss


if dataset_vocab != EXPECTED_VOCAB:

    print()
    print(
        "Expected:"
    )

    print(
        EXPECTED_VOCAB
    )

    print()
    print(
        "Observed:"
    )

    print(
        dataset_vocab
    )


    raise RuntimeError(
        "Deployment vocabulary mismatch."
    )


print()
print(
    "[PASS] Exact L17 vocabulary verified."
)


# ==========================================================================================
# 15. CLASS DISTRIBUTION
# ==========================================================================================

section(
    "15. DEPLOYMENT CLASS DISTRIBUTION"
)


distribution_rows = []


for class_id in range(10):

    gloss = (
        EXPECTED_VOCAB[
            class_id
        ]
    )


    train_count = int(

        (
            train_df[
                CLASS_COL
            ].astype(int)
            ==
            class_id
        ).sum()
    )


    val_count = int(

        (
            val_df[
                CLASS_COL
            ].astype(int)
            ==
            class_id
        ).sum()
    )


    locked_count = int(

        (
            locked_df[
                CLASS_COL
            ].astype(int)
            ==
            class_id
        ).sum()
    )


    distribution_rows.append(
        {

            "deployment_class_id":
                class_id,

            "gloss":
                gloss,

            "train":
                train_count,

            "validation":
                val_count,

            "locked_test":
                locked_count,

            "total":
                (
                    train_count
                    +
                    val_count
                    +
                    locked_count
                )
        }
    )


distribution_df = pd.DataFrame(
    distribution_rows
)


print(
    distribution_df.to_string(
        index=False
    )
)


if not (
    distribution_df[
        "validation"
    ]
    ==
    1
).all():

    raise RuntimeError(
        "Validation must contain one sample/class."
    )


if not (
    distribution_df[
        "locked_test"
    ]
    ==
    1
).all():

    raise RuntimeError(
        "Locked test must contain one sample/class."
    )


if (
    int(
        distribution_df[
            "train"
        ].sum()
    )
    !=
    70
):

    raise RuntimeError(
        "Training class distribution does not total 70."
    )


print()
print(
    "[PASS] Deployment class distribution verified."
)


# ==========================================================================================
# 16. LOAD L19 CHECKPOINT
# ==========================================================================================

section(
    "16. LOAD L19 CHECKPOINT"
)


checkpoint = torch.load(

    L19_CHECKPOINT,

    map_location="cpu",

    weights_only=False
)


print(
    "Checkpoint type:",
    type(checkpoint)
)


if not isinstance(
    checkpoint,
    dict
):

    raise RuntimeError(
        "Expected dictionary checkpoint."
    )


print()
print(
    "CHECKPOINT KEYS"
)

print(
    "-" * 120
)


for key, value in checkpoint.items():

    if torch.is_tensor(
        value
    ):

        description = (
            f"Tensor {tuple(value.shape)}"
        )


    elif isinstance(
        value,
        dict
    ):

        description = (
            f"dict[{len(value)}]"
        )


    elif isinstance(
        value,
        list
    ):

        description = (
            f"list[{len(value)}]"
        )


    else:

        description = (
            type(value).__name__
        )


    print(
        f"{key:<35} "
        f"{description}"
    )


# ==========================================================================================
# 17. DISCOVER STATE DICT
# ==========================================================================================

section(
    "17. DISCOVER EXACT MODEL STATE DICT"
)


state_dict = None
state_key = None


for candidate_key in [

    "model_state_dict",
    "state_dict",
    "model"

]:

    if candidate_key not in checkpoint:

        continue


    candidate = (
        checkpoint[
            candidate_key
        ]
    )


    if not isinstance(
        candidate,
        dict
    ):

        continue


    tensor_count = sum(

        torch.is_tensor(
            value
        )

        for value
        in candidate.values()
    )


    if tensor_count > 0:

        state_dict = candidate
        state_key = candidate_key

        break


# ------------------------------------------------------------------------------------------
# Check whether checkpoint itself is raw state_dict
# ------------------------------------------------------------------------------------------

if state_dict is None:

    if (
        len(checkpoint) > 0
        and
        all(
            torch.is_tensor(value)

            for value
            in checkpoint.values()
        )
    ):

        state_dict = checkpoint

        state_key = (
            "ROOT_STATE_DICT"
        )


if state_dict is None:

    raise RuntimeError(
        "Could not discover model state_dict."
    )


print(
    "State-dict source:",
    state_key
)

print(
    "Entries          :",
    len(state_dict)
)

print(
    "Tensor entries   :",
    sum(
        torch.is_tensor(value)

        for value
        in state_dict.values()
    )
)


# ==========================================================================================
# 18. PRINT STATE DICT STRUCTURE
# ==========================================================================================

section(
    "18. STATE-DICT ARCHITECTURE INSPECTION"
)


print(
    "First 40 tensor keys:"
)

print(
    "-" * 120
)


tensor_keys = [

    key

    for key, value
    in state_dict.items()

    if torch.is_tensor(
        value
    )
]


for key in tensor_keys[:40]:

    tensor = (
        state_dict[
            key
        ]
    )

    print(
        f"{key:<90} "
        f"{tuple(tensor.shape)}"
    )


print()
print(
    "Final 40 tensor keys:"
)

print(
    "-" * 120
)


for key in tensor_keys[-40:]:

    tensor = (
        state_dict[
            key
        ]
    )

    print(
        f"{key:<90} "
        f"{tuple(tensor.shape)}"
    )


# ==========================================================================================
# 19. IMPORTANT ARCHITECTURE KEYS
# ==========================================================================================

section(
    "19. IMPORTANT ARCHITECTURE KEYS"
)


important_keywords = [

    "classifier",
    "head",
    "fc",
    "norm",
    "layernorm",
    "encoder.layer.8",
    "encoder.layer.9",
    "encoder.layer.10",
    "encoder.layer.11"
]


important_rows = []


for name, tensor in state_dict.items():

    if not torch.is_tensor(
        tensor
    ):

        continue


    lower_name = (
        name.lower()
    )


    if any(

        keyword in lower_name

        for keyword
        in important_keywords

    ):

        important_rows.append(
            (
                name,
                tuple(
                    tensor.shape
                )
            )
        )


for name, shape in important_rows:

    print(
        f"{name:<90} "
        f"{shape}"
    )


# ==========================================================================================
# 20. PARAMETER INTEGRITY
# ==========================================================================================

section(
    "20. PARAMETER INTEGRITY"
)


total_values = 0
tensor_count = 0


for name, tensor in state_dict.items():

    if not torch.is_tensor(
        tensor
    ):

        continue


    tensor_count += 1

    total_values += (
        tensor.numel()
    )


    if not torch.isfinite(
        tensor
    ).all():

        raise RuntimeError(
            "Non-finite tensor found:\n"
            f"{name}"
        )


print(
    "Tensor entries :",
    tensor_count
)

print(
    "Values         :",
    f"{total_values:,}"
)

print()
print(
    "[PASS] All checkpoint tensors finite."
)


# ==========================================================================================
# 21. DISCOVER EXACT 10-CLASS HEAD
# ==========================================================================================

section(
    "21. DISCOVER EXACT 10-CLASS CLASSIFICATION HEAD"
)


candidate_heads = []


for name, tensor in state_dict.items():

    if not torch.is_tensor(
        tensor
    ):

        continue


    if (
        tensor.ndim == 2
        and
        tensor.shape[0] == 10
    ):

        candidate_heads.append(
            {
                "name":
                    name,

                "shape":
                    tuple(
                        tensor.shape
                    )
            }
        )


print(
    "Candidate 10-output matrices:"
)


for candidate in candidate_heads:

    print(
        f"  {candidate['name']:<90} "
        f"{candidate['shape']}"
    )


if len(
    candidate_heads
) == 0:

    raise RuntimeError(
        "No 10-output classifier matrix found."
    )


preferred_heads = [

    candidate

    for candidate
    in candidate_heads

    if (
        "classifier"
        in candidate[
            "name"
        ].lower()

        or

        "head"
        in candidate[
            "name"
        ].lower()
    )
]


if len(
    preferred_heads
) == 1:

    selected_head = (
        preferred_heads[0]
    )


elif (
    len(preferred_heads) == 0
    and
    len(candidate_heads) == 1
):

    selected_head = (
        candidate_heads[0]
    )


else:

    print()
    print(
        "[ERROR] Could not uniquely determine classifier."
    )

    print(
        "Preferred candidates:",
        preferred_heads
    )

    raise RuntimeError(
        "Ambiguous 10-class classifier structure."
    )


classifier_weight_name = (
    selected_head[
        "name"
    ]
)


classifier_weight = (
    state_dict[
        classifier_weight_name
    ]
)


classifier_output_dim = int(
    classifier_weight.shape[
        0
    ]
)


classifier_input_dim = int(
    classifier_weight.shape[
        1
    ]
)


classifier_bias_name = None


if classifier_weight_name.endswith(
    ".weight"
):

    candidate_bias = (

        classifier_weight_name[
            :-len(
                ".weight"
            )
        ]

        +

        ".bias"
    )


    if candidate_bias in state_dict:

        classifier_bias_name = (
            candidate_bias
        )


print()
print(
    "Selected classifier weight:",
    classifier_weight_name
)

print(
    "Selected classifier bias  :",
    classifier_bias_name
)

print(
    "Classifier input dimension:",
    classifier_input_dim
)

print(
    "Classifier output dimension:",
    classifier_output_dim
)


if classifier_output_dim != 10:

    raise RuntimeError(
        "Classifier output dimension != 10."
    )


print()
print(
    "[PASS] Exact 10-class head identified."
)


# ==========================================================================================
# 22. CHECKPOINT METADATA
# ==========================================================================================

section(
    "22. CHECKPOINT METADATA"
)


metadata_keys = [

    "epoch",
    "global_epoch",
    "stage",
    "best_top1",
    "best_top5",
    "best_val_acc",
    "best_accuracy",
    "val_accuracy",
    "validation_accuracy",
    "model_name",
    "num_classes",
    "num_frames",
    "image_size",
    "locked_test_used"
]


checkpoint_metadata = {}


for key in metadata_keys:

    if key not in checkpoint:

        continue


    value = (
        checkpoint[
            key
        ]
    )


    if isinstance(
        value,
        np.generic
    ):

        value = (
            value.item()
        )


    if isinstance(

        value,

        (
            str,
            int,
            float,
            bool,
            type(None)
        )

    ):

        checkpoint_metadata[
            key
        ] = value


        print(
            f"{key:<25}: "
            f"{value}"
        )


if len(
    checkpoint_metadata
) == 0:

    print(
        "[INFO] No standard scalar metadata fields."
    )


if "num_classes" in checkpoint_metadata:

    if int(
        checkpoint_metadata[
            "num_classes"
        ]
    ) != 10:

        raise RuntimeError(
            "Checkpoint metadata num_classes != 10."
        )


if "locked_test_used" in checkpoint_metadata:

    if bool(
        checkpoint_metadata[
            "locked_test_used"
        ]
    ):

        raise RuntimeError(
            "Checkpoint reports locked-test usage."
        )


# ==========================================================================================
# 23. CHECKPOINT VOCABULARY
# ==========================================================================================

section(
    "23. CHECK CHECKPOINT VOCABULARY"
)


checkpoint_mapping = None
mapping_source = None


for mapping_key in [

    "class_to_gloss",
    "id_to_gloss",
    "id_to_label",
    "class_mapping"

]:

    if (
        mapping_key in checkpoint
        and
        isinstance(
            checkpoint[
                mapping_key
            ],
            dict
        )
    ):

        checkpoint_mapping = (
            checkpoint[
                mapping_key
            ]
        )

        mapping_source = (
            mapping_key
        )

        break


if (
    checkpoint_mapping is None
    and
    "gloss_to_id" in checkpoint
    and
    isinstance(
        checkpoint[
            "gloss_to_id"
        ],
        dict
    )
):

    mapping_source = (
        "gloss_to_id"
    )


    checkpoint_mapping = {

        int(value):
            str(key).strip().lower()

        for key, value
        in checkpoint[
            "gloss_to_id"
        ].items()
    }


if checkpoint_mapping is not None:

    normalized_mapping = {

        int(key):
            str(value).strip().lower()

        for key, value
        in checkpoint_mapping.items()
    }


    print(
        "Mapping source:",
        mapping_source
    )


    for class_id in sorted(
        normalized_mapping
    ):

        print(
            f"  {class_id:2d} -> "
            f"{normalized_mapping[class_id]}"
        )


    if (
        normalized_mapping
        !=
        EXPECTED_VOCAB
    ):

        raise RuntimeError(
            "Checkpoint vocabulary mismatch."
        )


    print()
    print(
        "[PASS] Checkpoint vocabulary verified."
    )


else:

    print(
        "[INFO] No explicit vocabulary "
        "stored in checkpoint."
    )

    print(
        "[INFO] L17/L18B mapping will "
        "be frozen externally."
    )


# ==========================================================================================
# 24. FREEZE CHECKPOINT BYTE-FOR-BYTE
# ==========================================================================================

section(
    "24. FREEZE DEPLOYMENT CHECKPOINT"
)


FROZEN_MODEL = (

    L21_MODEL_ROOT /
    "l21_frozen_deployment_videomae_10class.pt"
)


shutil.copy2(

    L19_CHECKPOINT,
    FROZEN_MODEL
)


frozen_checkpoint_sha = (
    sha256_file(
        FROZEN_MODEL
    )
)


print(
    "Original SHA:"
)

print(
    source_checkpoint_sha
)


print()
print(
    "Frozen SHA:"
)

print(
    frozen_checkpoint_sha
)


if (
    frozen_checkpoint_sha
    !=
    source_checkpoint_sha
):

    raise RuntimeError(
        "Frozen copy is not byte-identical."
    )


print()
print(
    "[PASS] L21 checkpoint is byte-identical to L19."
)


# ==========================================================================================
# 25. SAVE CLASS MAPPING
# ==========================================================================================

section(
    "25. SAVE DEPLOYMENT CLASS MAPPING"
)


class_mapping_df = pd.DataFrame(
    [

        {

            "deployment_class_id":
                class_id,

            "gloss":
                gloss
        }

        for class_id, gloss
        in EXPECTED_VOCAB.items()
    ]
)


CLASS_MAPPING_FILE = (

    L21_CONFIG_ROOT /
    "deployment_class_mapping.csv"
)


class_mapping_df.to_csv(

    CLASS_MAPPING_FILE,
    index=False
)


print(
    class_mapping_df.to_string(
        index=False
    )
)


print()
print(
    "[SAVED]",
    CLASS_MAPPING_FILE
)


# ==========================================================================================
# 26. SAVE VOCABULARY JSON
# ==========================================================================================

section(
    "26. SAVE DEPLOYMENT VOCABULARY"
)


VOCAB_JSON = (

    L21_CONFIG_ROOT /
    "deployment_vocabulary.json"
)


with open(
    VOCAB_JSON,
    "w"
) as f:

    json.dump(
        {

            str(class_id):
                gloss

            for class_id, gloss
            in EXPECTED_VOCAB.items()
        },

        f,

        indent=2
    )


print(
    "[SAVED]",
    VOCAB_JSON
)


# ==========================================================================================
# 27. SAVE DISTRIBUTION
# ==========================================================================================

section(
    "27. SAVE DATASET DISTRIBUTION"
)


DISTRIBUTION_FILE = (

    L21_REPORT_ROOT /
    "deployment_class_distribution.csv"
)


distribution_df.to_csv(

    DISTRIBUTION_FILE,
    index=False
)


print(
    "[SAVED]",
    DISTRIBUTION_FILE
)


# ==========================================================================================
# 28. DEPLOYMENT CONFIGURATION
# ==========================================================================================

section(
    "28. CREATE DEPLOYMENT CONFIGURATION"
)


DEPLOYMENT_CONFIG = {

    "stage":
        "L21",

    "project":
        "PARASURG_CISLR_REAL_TIME_ISL_INTERPRETER",

    "status":
        "FROZEN",

    "source_stage":
        "L19",

    "model_family":
        "VideoMAE",

    "pretrained_backbone":
        "MCG-NJU/videomae-base-finetuned-kinetics",

    "num_classes":
        10,

    "num_frames":
        16,

    "input_resolution": [
        224,
        224
    ],

    "input_layout":
        "B,T,C,H,W",

    "expected_input_shape": [
        1,
        16,
        3,
        224,
        224
    ],

    "classifier": {

        "weight_name":
            classifier_weight_name,

        "bias_name":
            classifier_bias_name,

        "input_dimension":
            classifier_input_dim,

        "output_dimension":
            classifier_output_dim
    },

    "checkpoint_structure": {

        "state_dict_key":
            state_key,

        "checkpoint_metadata":
            checkpoint_metadata
    },

    "vocabulary": {

        str(class_id):
            gloss

        for class_id, gloss
        in EXPECTED_VOCAB.items()
    },

    "dataset": {

        "train":
            70,

        "validation":
            10,

        "locked_test":
            10,

        "balanced_training_draws_per_epoch":
            200
    },

    "known_l19_result": {

        "best_stage":
            "FINETUNE4",

        "best_global_epoch":
            38,

        "validation_top1_percent":
            100.0,

        "validation_correct":
            10,

        "validation_total":
            10,

        "final_test_result":
            False,

        "real_world_accuracy_claim":
            False
    },

    "full_cislr_research_context": {

        "stage":
            "L20D",

        "classes":
            215,

        "system":
            "B2C multimodal fusion",

        "validation_top1_percent":
            40.132,

        "validation_top5_percent":
            57.566
    },

    "integrity": {

        "source_checkpoint_sha256":
            source_checkpoint_sha,

        "expected_checkpoint_sha256":
            EXPECTED_L19_CHECKPOINT_SHA,

        "frozen_checkpoint_sha256":
            frozen_checkpoint_sha,

        "expected_l19_master_freeze_sha256":
            EXPECTED_L19_MASTER_FREEZE_SHA,

        "actual_l19_master_freeze_sha256":
            actual_l19_master_freeze_sha,

        "l19_master_freeze_verified":
            l19_master_freeze_verified
    },

    "l21_actions": {

        "training":
            False,

        "weight_modification":
            False,

        "locked_test_inference":
            False,

        "locked_test_evaluation":
            False,

        "openvino_export":
            False
    },

    "next_stage":
        "L22_OPENVINO_EXPORT"
}


DEPLOYMENT_CONFIG_FILE = (

    L21_CONFIG_ROOT /
    "l21_deployment_config.json"
)


with open(
    DEPLOYMENT_CONFIG_FILE,
    "w"
) as f:

    json.dump(
        DEPLOYMENT_CONFIG,
        f,
        indent=2
    )


print(
    "[SAVED]",
    DEPLOYMENT_CONFIG_FILE
)


# ==========================================================================================
# 29. HASH L21 ARTIFACTS
# ==========================================================================================

section(
    "29. HASH L21 ARTIFACTS"
)


hash_targets = {

    "source_l19_checkpoint":
        L19_CHECKPOINT,

    "frozen_l21_checkpoint":
        FROZEN_MODEL,

    "train_manifest":
        TRAIN_MANIFEST,

    "validation_manifest":
        VAL_MANIFEST,

    "locked_test_manifest":
        LOCKED_TEST_MANIFEST,

    "class_mapping":
        CLASS_MAPPING_FILE,

    "vocabulary":
        VOCAB_JSON,

    "deployment_config":
        DEPLOYMENT_CONFIG_FILE,

    "class_distribution":
        DISTRIBUTION_FILE
}


artifact_hashes = {}


for name, path in hash_targets.items():

    digest = (
        sha256_file(
            path
        )
    )


    artifact_hashes[
        name
    ] = {

        "path":
            str(path),

        "sha256":
            digest
    }


    print(
        f"{name:<30}: "
        f"{digest}"
    )


HASH_FILE = (

    L21_REPORT_ROOT /
    "l21_artifact_hashes.json"
)


with open(
    HASH_FILE,
    "w"
) as f:

    json.dump(
        artifact_hashes,
        f,
        indent=2
    )


print()
print(
    "[SAVED]",
    HASH_FILE
)


# ==========================================================================================
# 30. FINAL INTEGRITY CHECKS
# ==========================================================================================

section(
    "30. FINAL L21 INTEGRITY CHECKS"
)


checks = {

    "train_rows_70":
        len(train_df) == 70,

    "validation_rows_10":
        len(val_df) == 10,

    "locked_rows_10":
        len(locked_df) == 10,

    "train_unique_70":
        len(train_uids) == 70,

    "validation_unique_10":
        len(val_uids) == 10,

    "locked_unique_10":
        len(locked_uids) == 10,

    "train_validation_overlap_zero":
        len(train_val_overlap) == 0,

    "train_locked_overlap_zero":
        len(train_locked_overlap) == 0,

    "validation_locked_overlap_zero":
        len(val_locked_overlap) == 0,

    "actual_class_column_verified":
        CLASS_COL
        ==
        "deployment_class_id",

    "vocabulary_verified":
        dataset_vocab
        ==
        EXPECTED_VOCAB,

    "classifier_outputs_10":
        classifier_output_dim
        ==
        10,

    "checkpoint_sha_verified":
        source_checkpoint_sha
        ==
        EXPECTED_L19_CHECKPOINT_SHA,

    "frozen_copy_identical":
        frozen_checkpoint_sha
        ==
        source_checkpoint_sha,

    "locked_test_evaluated":
        False,

    "training_performed":
        False,

    "weights_modified":
        False
}


false_expected = {

    "locked_test_evaluated",
    "training_performed",
    "weights_modified"
}


all_pass = True


for name, value in checks.items():

    if name in false_expected:

        passed = (
            value is False
        )

    else:

        passed = bool(
            value
        )


    print(
        f"{'[PASS]' if passed else '[FAIL]'} "
        f"{name}"
    )


    if not passed:

        all_pass = False


if not all_pass:

    raise RuntimeError(
        "L21 integrity checks failed."
    )


print()
print(
    "[PASS] ALL L21 INTEGRITY CHECKS PASSED."
)


# ==========================================================================================
# 31. CREATE L21 SUMMARY
# ==========================================================================================

section(
    "31. CREATE L21 SUMMARY"
)


SUMMARY = {

    "stage":
        "L21",

    "name":
        "Freeze deployment classifier",

    "status":
        "COMPLETE_AND_FROZEN",

    "selected_classifier": {

        "source":
            "L19",

        "architecture":
            "VideoMAE",

        "num_classes":
            10,

        "num_frames":
            16,

        "resolution":
            "224x224",

        "classifier_weight":
            classifier_weight_name,

        "classifier_bias":
            classifier_bias_name,

        "classifier_input_dimension":
            classifier_input_dim,

        "classifier_output_dimension":
            classifier_output_dim,

        "state_dict_key":
            state_key,

        "checkpoint":
            str(
                FROZEN_MODEL
            ),

        "checkpoint_sha256":
            frozen_checkpoint_sha
    },

    "vocabulary": {

        str(class_id):
            gloss

        for class_id, gloss
        in EXPECTED_VOCAB.items()
    },

    "protocol": {

        "train":
            70,

        "validation":
            10,

        "locked_test":
            10,

        "all_split_overlaps":
            0,

        "locked_test_evaluated":
            False
    },

    "known_l19_result": {

        "best_stage":
            "FINETUNE4",

        "best_epoch":
            38,

        "validation_top1":
            100.0,

        "correct":
            10,

        "total":
            10,

        "real_world_accuracy_claim":
            False
    },

    "next_stage":
        "L22_OPENVINO_EXPORT"
}


SUMMARY_FILE = (

    L21_REPORT_ROOT /
    "L21_DEPLOYMENT_CLASSIFIER_SUMMARY.json"
)


with open(
    SUMMARY_FILE,
    "w"
) as f:

    json.dump(
        SUMMARY,
        f,
        indent=2
    )


print(
    "[SAVED]",
    SUMMARY_FILE
)


# ==========================================================================================
# 32. CREATE MASTER FREEZE
# ==========================================================================================

section(
    "32. CREATE L21 MASTER FREEZE"
)


MASTER_FREEZE = {

    "stage":
        "L21",

    "status":
        "FROZEN",

    "project":
        "PARASURG_CISLR_REAL_TIME_ISL_INTERPRETER",

    "source_stage":
        "L19",

    "selected_model":
        "Restricted-vocabulary VideoMAE",

    "checkpoint":
        str(
            FROZEN_MODEL
        ),

    "checkpoint_sha256":
        frozen_checkpoint_sha,

    "authoritative_l19_checkpoint_sha256":
        EXPECTED_L19_CHECKPOINT_SHA,

    "authoritative_l19_master_freeze_sha256":
        EXPECTED_L19_MASTER_FREEZE_SHA,

    "class_mapping":
        str(
            CLASS_MAPPING_FILE
        ),

    "vocabulary":
        str(
            VOCAB_JSON
        ),

    "deployment_config":
        str(
            DEPLOYMENT_CONFIG_FILE
        ),

    "class_distribution":
        str(
            DISTRIBUTION_FILE
        ),

    "summary":
        str(
            SUMMARY_FILE
        ),

    "hash_report":
        str(
            HASH_FILE
        ),

    "integrity_checks":
        checks,

    "locked_test_inference":
        False,

    "locked_test_evaluation":
        False,

    "training":
        False,

    "weights_modified":
        False,

    "openvino_export":
        False,

    "next_stage":
        "L22_OPENVINO_EXPORT"
}


MASTER_FREEZE_FILE = (

    L21_ROOT /
    "L21_DEPLOYMENT_CLASSIFIER_MASTER_FREEZE.json"
)


with open(
    MASTER_FREEZE_FILE,
    "w"
) as f:

    json.dump(
        MASTER_FREEZE,
        f,
        indent=2
    )


MASTER_SHA = (
    sha256_file(
        MASTER_FREEZE_FILE
    )
)


print(
    "[SAVED]",
    MASTER_FREEZE_FILE
)


print()
print(
    "Master SHA256:"
)

print(
    MASTER_SHA
)


# ==========================================================================================
# 33. FINAL REPORT
# ==========================================================================================

section(
    "L21 — DEPLOYMENT CLASSIFIER FREEZE COMPLETE"
)


print(
    "SELECTED DEPLOYMENT MODEL"
)

print(
    "  Stage        : L19"
)

print(
    "  Architecture : VideoMAE"
)

print(
    "  Classes      : 10"
)

print(
    "  Frames       : 16"
)

print(
    "  Resolution   : 224 x 224"
)

print(
    "  Input layout : B,T,C,H,W"
)

print(
    "  Input shape  : (1,16,3,224,224)"
)


print()
print(
    "EXACT CHECKPOINT STRUCTURE"
)

print(
    "  State dict   :",
    state_key
)

print(
    "  Head weight  :",
    classifier_weight_name
)

print(
    "  Head bias    :",
    classifier_bias_name
)

print(
    "  Head input   :",
    classifier_input_dim
)

print(
    "  Head output  :",
    classifier_output_dim
)


print()
print(
    "DEPLOYMENT VOCABULARY"
)


for class_id, gloss in EXPECTED_VOCAB.items():

    print(
        f"  {class_id:2d} -> {gloss}"
    )


print()
print(
    "DATASET"
)

print(
    "  Train       : 70"
)

print(
    "  Validation  : 10"
)

print(
    "  Locked test : 10"
)

print(
    "  Overlap     : ZERO"
)


print()
print(
    "KNOWN L19 RESULT"
)

print(
    "  Best stage          : FINETUNE4"
)

print(
    "  Best epoch          : 38"
)

print(
    "  Validation Top-1    : 100% (10/10)"
)

print(
    "  Final-test claim    : NO"
)

print(
    "  Real-world 100% claim: NO"
)


print()
print(
    "L20D RESEARCH CONTEXT"
)

print(
    "  Classes : 215"
)

print(
    "  B2C Top-1 : 40.132%"
)

print(
    "  B2C Top-5 : 57.566%"
)


print()
print(
    "INTEGRITY"
)

print(
    "  Training              : NO"
)

print(
    "  Weight modification   : NO"
)

print(
    "  Locked-test inference : NO"
)

print(
    "  Locked-test evaluation: NO"
)

print(
    "  OpenVINO export       : NO"
)

print(
    "  L19 checkpoint SHA    : VERIFIED"
)

print(
    "  Frozen copy           : BYTE-IDENTICAL"
)


print()
print(
    "FROZEN MODEL"
)

print(
    FROZEN_MODEL
)


print()
print(
    "MODEL SHA256"
)

print(
    frozen_checkpoint_sha
)


print()
print(
    "MASTER FREEZE"
)

print(
    MASTER_FREEZE_FILE
)


print()
print(
    "MASTER SHA256"
)

print(
    MASTER_SHA
)


print()
print(
    "NEXT"
)

print(
    "L22 — EXPORT SELECTED DEPLOYMENT MODEL TO OPENVINO"
)

print(
    "PyTorch VideoMAE -> OpenVINO IR"
)

print(
    "L23 will then verify PyTorch <-> OpenVINO predictions."
)


print()
print("=" * 120)


CISLR — L21 FREEZE DEPLOYMENT CLASSIFIER
Python           : 3.11.15
PyTorch          : 2.14.0+cu130
CUDA available   : True
GPU              : NVIDIA GeForce RTX 4050 Laptop GPU
Training         : DISABLED
Locked test eval : DISABLED
Weight changes   : DISABLED
OpenVINO export  : NOT IN L21

4. PROJECT PATHS
Project root : /home/dipshikha/Music/cao/CISLR_RESEARCH
L18B root    : /home/dipshikha/Music/cao/CISLR_RESEARCH/CISLR_LANDMARK_MODEL/audit/l18b_10class_dataset
L19 root     : /home/dipshikha/Music/cao/CISLR_RESEARCH/CISLR_LANDMARK_MODEL/audit/l19_10class_videomae
L21 root     : /home/dipshikha/Music/cao/CISLR_RESEARCH/CISLR_LANDMARK_MODEL/audit/l21_deployment_classifier_freeze

5. AUTHORITATIVE L19 ARTIFACT HASHES
Expected checkpoint SHA:
3a35abe76cdea634490473eaf83c6121e102658362cfbab19410a4949bb8c91c

Expected master-freeze SHA:
0997873212a9b4495b965773645231f8a0206b12a5f98934bac04b2a071b9d39

6. LOCATE L18B DATASET MANIFESTS
[FOUND] Train manifest : /home/dipshikha/Music/cao/CI